In [ ]:
import cv2
import numpy as np
import time
from datetime import datetime
import csv
import os
from collections import defaultdict
import pickle
from sklearn.neighbors import KNeighborsClassifier

class DeskTracker:
    def __init__(self):
        # Load YOLOv4-tiny model 
        cfg_path = 'Yolo_model/yolov4-tiny.cfg'
        names_path = 'Yolo_model/coco.names'
        weights_path = 'Yolo_model/yolov4-tiny.weights'

        # Load names
        with open(names_path, 'r') as f:
            self.classes = [line.strip() for line in f.readlines()]

        # Load model
        self.net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
        
        # Get output layer names
        layer_names = self.net.getLayerNames()
        self.output_layers = [layer_names[i - 1] for i in self.net.getUnconnectedOutLayers()]
        
        # Detection parameters
        self.confidence_threshold = 0.3
        self.nms_threshold = 0.4
        
        # Load face recognition data
        try:
            with open('data/Departments.pkl', 'rb') as f:
                self.DEPARTMENTS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/Entry_numbers.pkl', 'rb') as f:
                self.LABELS = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f)
                self.FACES = self.FACES / 255.0
                print(f"FACES shape: {self.FACES.shape}")
            
            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.LABELS)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []
            self.LABELS = []
            self.face_recognition_enabled = False

        # Load face cascade
        self.face_cascade = cv2.CascadeClassifier('data/haarcascade_frontalface_default.xml')
        
        # Desk ROI - will be detected automatically
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask = None
        
        # Tracking
        self.trackers = {}
        self.MAX_CENTROIDS = 10
        self.DISAPPEARANCE_TIME = 2.0
        
        # Logging
        self.log_file = f'desk_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Name', 'Event', 'Location'])

    def log_event(self, message):
        """Log events to console"""
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def detect_desk_area(self, cap):
        """Detect the desk area using object detection"""
        self.log_event("Detecting desk area...")
        
        # Take multiple frames to improve detection reliability
        desk_candidates = []
        frame_count = 0
        max_frames = 5
        
        frame = None  # Initialize frame variable
        
        # Process several frames to get a better desk detection
        while frame_count < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
                
            height, width = frame.shape[:2]
            
            # Create a blob from the frame and perform a forward pass
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                break
            
            # Variables to store detection info
            desks = []
            
            # Process detections to find tables/desks/chairs
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    
                    # COCO class IDs: 56=chair, 60=dining table, 63=laptop, 64=mouse, 65=keyboard
                    desk_related_classes = [56, 60, 63, 64, 65]
                    
                    if confidence > self.confidence_threshold and class_id in desk_related_classes:
                        center_x = int(obj_detection[0] * width)
                        center_y = int(obj_detection[1] * height)
                        w = int(obj_detection[2] * width)
                        h = int(obj_detection[3] * height)
                        
                        # Calculate top-left corner coordinates
                        x = int(center_x - w / 2)
                        y = int(center_y - h / 2)
                        
                        desks.append([x, y, w, h, confidence, class_id])
            
            # If we found desk-related objects, add them to candidates
            if desks:
                desk_candidates.extend(desks)
            
            frame_count += 1
        
        # If we don't have enough desk candidates, use a default area
        if len(desk_candidates) < 2:
            self.log_event("Not enough desk objects detected. Using default desk area.")
            if frame is not None:
                height, width = frame.shape[:2]
            else:
                height, width = 480, 640  # Default dimensions
            
            # Default to middle-bottom portion of the frame
            min_x = int(width * 0.2)
            min_y = int(height * 0.4)
            max_x = int(width * 0.8)
            max_y = int(height * 0.9)
        else:
            # Group desk objects to determine the desk area
            x_coords = [d[0] for d in desk_candidates]
            y_coords = [d[1] for d in desk_candidates]
            max_x_coords = [d[0] + d[2] for d in desk_candidates]
            max_y_coords = [d[1] + d[3] for d in desk_candidates]
            
            # Create a bounding box that covers all desk-related objects with padding
            min_x = max(0, int(min(x_coords) - 0.1 * width))
            min_y = max(0, int(min(y_coords) - 0.1 * height))
            max_x = min(width, int(max(max_x_coords) + 0.1 * width))
            max_y = min(height, int(max(max_y_coords) + 0.1 * height))
            
            self.log_event(f"Desk area detected: ({min_x}, {min_y}) to ({max_x}, {max_y})")
        
        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        
        return (min_x, min_y, max_x, max_y)

    def get_name_from_centroid(self, cx, cy, frame):
        """Extract name from face recognition around a centroid"""
        if not self.face_recognition_enabled:
            return "Unknown"
        
        h, w = frame.shape[:2]
        
        # Define region around centroid for face detection
        rx1, ry1 = max(0, cx - 100), max(0, cy - 150)
        rx2, ry2 = min(w, cx + 100), min(h, cy + 30)
        
        face_region = frame[ry1:ry2, rx1:rx2]
        
        if face_region.size == 0:
            return "Unknown"

        gray_region = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        faces = self.face_cascade.detectMultiScale(gray_region, 1.3, 5)

        # Default if no face / no match
        Name = "Unknown"

        for (x, y, w_face, h_face) in faces:
            crop_img = face_region[y:y+h_face, x:x+w_face, :]
          
            # Use 75x100 => 7500 features to match FACES shape
            resized_img = cv2.resize(crop_img, (75, 100))
            gray = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
            resized_flat = gray.reshape(1, -1) / 255.0

            # Check if feature dimension matches
            if resized_flat.shape[1] != self.FACES.shape[1]:
                self.log_event(f"Feature mismatch: {resized_flat.shape[1]} vs {self.FACES.shape[1]}")
                continue

            distances, indices = self.knn.kneighbors(resized_flat)
            threshold = 28
            
            if distances[0][0] > threshold:
                output = ["Unknown"]
            else:
                output = self.knn.predict(resized_flat)

            entry_no = output[0]
            if entry_no in self.LABELS:
                index = self.LABELS.index(entry_no)
                Name = self.NAMES[index]
                break  # Use first detected face

        return Name

    def run(self):
        """Main tracking loop"""
        cap = cv2.VideoCapture(0)
        
        if not cap.isOpened():
            self.log_event("Error: Cannot open camera")
            return
        
        # Detect desk area from initial frames
        self.detect_desk_area(cap)
        
        # Reset camera to beginning
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        
        last_cleanup = time.time()

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            h, w = frame.shape[:2]

            # Create desk mask if needed
            if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                self.desk_mask = np.zeros((h, w), np.uint8)
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(self.desk_mask, self.DESK_ROI_PT1, self.DESK_ROI_PT2, 255, -1)

            # YOLO person detection
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            outputs = self.net.forward(self.output_layers)

            boxes, confidences = [], []
            for output in outputs:
                for detection in output:
                    scores = detection[5:]
                    class_id = np.argmax(scores)
                    conf = scores[class_id]
                    if conf > 0.5 and self.classes[class_id] == 'person':
                        cx = int(detection[0] * w)
                        cy = int(detection[1] * h)
                        bw = int(detection[2] * w)
                        bh = int(detection[3] * h)
                        x1 = int(cx - bw / 2)
                        y1 = int(cy - bh / 2)
                        boxes.append([x1, y1, bw, bh])
                        confidences.append(float(conf))

            indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, self.nms_threshold)

            current_in_desk = set()
            detections = indices.flatten() if len(indices) > 0 else []

            for i in detections:
                x1, y1, bw, bh = boxes[i]
                x2, y2 = x1 + bw, y1 + bh
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

                name = self.get_name_from_centroid(cx, cy, frame)

                if name not in self.trackers:
                    self.trackers[name] = {'centroids': [], 'last_seen': 0}
                
                self.trackers[name]['bbox'] = (x1, y1, x2, y2)
                self.trackers[name]['centroids'].append((cx, cy))
                self.trackers[name]['last_seen'] = time.time()
                
                if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                    self.trackers[name]['centroids'].pop(0)

                # Calculate overlap with desk area
                overlap_ratio = 0
                if x2 > x1 and y2 > y1:
                    y_slice = slice(max(0, y1), min(h, y2))
                    x_slice = slice(max(0, x1), min(w, x2))
                    mask_roi = self.desk_mask[y_slice, x_slice]
                    if mask_roi.size > 0:
                        overlap_ratio = np.sum(mask_roi > 0) / mask_roi.size
                
                location = 'Desk (Chair)' if overlap_ratio > 0.4 else 'Outside'
                if 'Desk' in location:
                    current_in_desk.add(name)

                # Draw bounding box and labels
                color = (0, 255, 0) if 'Desk' in location else (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                
                name_w = len(name) * 15 + 20
                cv2.rectangle(frame, (x1, y1 - 35), (x1 + name_w, y1), color, -1)
                cv2.putText(frame, name, (x1 + 10, y1 - 10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                cv2.putText(frame, location, (x1, y2 + 25), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            # Cleanup disappeared trackers
            if time.time() - last_cleanup > 1.0:
                to_remove = [name for name, data in self.trackers.items() 
                            if time.time() - data['last_seen'] > self.DISAPPEARANCE_TIME]
                for name in to_remove:
                    del self.trackers[name]
                last_cleanup = time.time()

            # Log entry/exit events
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            with open(self.log_file, 'a', newline='') as f:
                writer = csv.writer(f)
                for name, data in self.trackers.items():
                    if len(data['centroids']) >= 2:
                        prev_cx, prev_cy = data['centroids'][-2]
                        prev_loc = 'Desk' if (0 <= prev_cy < h and 0 <= prev_cx < w and 
                                             self.desk_mask[int(prev_cy), int(prev_cx)] > 0) else 'Outside'
                        curr_loc = 'Desk' if name in current_in_desk else 'Outside'
                        if prev_loc != curr_loc:
                            event = 'Entered' if curr_loc == 'Desk' else 'Left'
                            writer.writerow([timestamp, name, event, curr_loc])

            # Draw desk ROI
            if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                cv2.rectangle(frame, self.DESK_ROI_PT1, self.DESK_ROI_PT2, (255, 255, 0), 2)
                cv2.putText(frame, f'Desk:{len(current_in_desk)} Total:{len(self.trackers)}', 
                           (self.DESK_ROI_PT1[0], self.DESK_ROI_PT1[1] - 10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

            cv2.imshow('Employee Desk Tracking - Auto Desk Detection', frame)
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved to: {self.log_file}")


if __name__ == "__main__":
    tracker = DeskTracker()
    tracker.run()

In [ ]:
import cv2
import numpy as np
import time
from datetime import datetime
import csv
import os
from collections import defaultdict
import pickle
from sklearn.neighbors import KNeighborsClassifier

class DeskTracker:
    def __init__(self):
        # Load YOLOv4-tiny model 
        cfg_path = 'Yolo_model/yolov4-tiny.cfg'
        names_path = 'Yolo_model/coco.names'
        weights_path = 'Yolo_model/yolov4-tiny.weights'

        # Load names
        with open(names_path, 'r') as f:
            self.classes = [line.strip() for line in f.readlines()]

        # Load model
        self.net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
        
        # Get output layer names
        layer_names = self.net.getLayerNames()
        self.output_layers = [layer_names[i - 1] for i in self.net.getUnconnectedOutLayers()]
        
        # Detection parameters
        self.confidence_threshold = 0.3
        self.nms_threshold = 0.4
        
        # Load face recognition data
        try:
            with open('data/Departments.pkl', 'rb') as f:
                self.DEPARTMENTS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/Entry_numbers.pkl', 'rb') as f:
                self.LABELS = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f)
                self.FACES = self.FACES / 255.0
                print(f"FACES shape: {self.FACES.shape}")
            
            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.LABELS)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []
            self.LABELS = []
            self.face_recognition_enabled = False

        # Load face cascade
        self.face_cascade = cv2.CascadeClassifier('data/haarcascade_frontalface_default.xml')
        
        # Desk ROI - will be detected automatically
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask = None
        
        # Tracking
        self.trackers = {}
        self.MAX_CENTROIDS = 10
        self.DISAPPEARANCE_TIME = 2.0
        
        # Logging
        self.log_file = f'desk_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Name', 'Event', 'Location'])
        
        # Monitor area (will be set from desk detection)
        self.monitor_area = None
        self.absence_start_time = None
        self.source_type = "camera"

    def log_event(self, message):
        """Log events to console"""
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def detect_desk_area(self, cap):
        """Detect the desk area using object detection"""
        self.log_event("Detecting desk area...")
        
        # Take multiple frames to improve detection reliability
        desk_candidates = []
        frame_count = 0
        max_frames = 5
        
        frame = None  # Initialize frame variable
        
        # Process several frames to get a better desk detection
        while frame_count < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
                
            height, width = frame.shape[:2]
            
            # Create a blob from the frame and perform a forward pass
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                break
            
            # Variables to store detection info
            desks = []
            
            # Process detections to find tables/desks/chairs
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    
                    # COCO class IDs: 56=chair, 60=dining table, 63=laptop, 64=mouse, 65=keyboard
                    desk_related_classes = [56, 60, 63, 64, 65]
                    
                    if confidence > self.confidence_threshold and class_id in desk_related_classes:
                        center_x = int(obj_detection[0] * width)
                        center_y = int(obj_detection[1] * height)
                        w = int(obj_detection[2] * width)
                        h = int(obj_detection[3] * height)
                        
                        # Calculate top-left corner coordinates
                        x = int(center_x - w / 2)
                        y = int(center_y - h / 2)
                        
                        desks.append([x, y, w, h, confidence, class_id])
            
            # If we found desk-related objects, add them to candidates
            if desks:
                desk_candidates.extend(desks)
            
            frame_count += 1
        
        # If we don't have enough desk candidates, use a default area
        if len(desk_candidates) < 2:
            self.log_event("Not enough desk objects detected. Using default desk area.")
            if frame is not None:
                height, width = frame.shape[:2]
            else:
                height, width = 480, 640  # Default dimensions
            
            # Default to middle-bottom portion of the frame
            min_x = int(width * 0.2)
            min_y = int(height * 0.4)
            max_x = int(width * 0.8)
            max_y = int(height * 0.9)
        else:
            # Group desk objects to determine the desk area
            x_coords = [d[0] for d in desk_candidates]
            y_coords = [d[1] for d in desk_candidates]
            max_x_coords = [d[0] + d[2] for d in desk_candidates]
            max_y_coords = [d[1] + d[3] for d in desk_candidates]
            
            # Create a bounding box that covers all desk-related objects with padding
            min_x = max(0, int(min(x_coords) - 0.1 * width))
            min_y = max(0, int(min(y_coords) - 0.1 * height))
            max_x = min(width, int(max(max_x_coords) + 0.1 * width))
            max_y = min(height, int(max(max_y_coords) + 0.1 * height))
            
            self.log_event(f"Desk area detected: ({min_x}, {min_y}) to ({max_x}, {max_y})")
        
        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        
        # Set monitor area for _process_frame method
        self.monitor_area = (min_x, min_y, max_x, max_y)
        
        return (min_x, min_y, max_x, max_y)

    def _process_frame(self, frame):
        """Process a frame to detect people and update status"""
        height, width = frame.shape[:2]
        employee_detected = False
        
        # Create a blob from the frame and perform object detection
        blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
        self.net.setInput(blob)
        
        try:
            detections = self.net.forward(self.output_layers)
        except Exception as e:
            self.log_event(f"Error during detection: {e}")
            return frame, False
        
        # Variables to store detection info
        boxes = []
        confidences = []
        class_ids = []
        
        # Process detections to find people
        for detection in detections:
            for obj_detection in detection:
                scores = obj_detection[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]
                
                # Check if the detected object is a person (class ID 0 for COCO dataset)
                if confidence > self.confidence_threshold and class_id == 0:  # 0 is index for person in COCO
                    center_x = int(obj_detection[0] * width)
                    center_y = int(obj_detection[1] * height)
                    w = int(obj_detection[2] * width)
                    h = int(obj_detection[3] * height)
                    
                    # Calculate top-left corner coordinates of the bounding box
                    x = int(center_x - w / 2)
                    y = int(center_y - h / 2)
                    
                    boxes.append([x, y, w, h])
                    confidences.append(float(confidence))
                    class_ids.append(class_id)
        
        # Apply non-maximum suppression to remove redundant overlapping boxes
        if len(boxes) > 0:
            indices = cv2.dnn.NMSBoxes(boxes, confidences, self.confidence_threshold, 0.4)
            
            # Check if any person was detected in the monitored area
            if len(indices) > 0:
                for i in indices.flatten():
                    x, y, w, h = boxes[i]
                    
                    # Calculate person box
                    person_box = (x, y, x+w, y+h)
                    
                    # Calculate intersection with monitor area
                    x_intersection = max(self.monitor_area[0], person_box[0])
                    y_intersection = max(self.monitor_area[1], person_box[1])
                    w_intersection = min(self.monitor_area[2], person_box[2]) - x_intersection
                    h_intersection = min(self.monitor_area[3], person_box[3]) - y_intersection
                    
                    is_in_desk_area = False
                    if w_intersection > 0 and h_intersection > 0:
                        intersection_area = w_intersection * h_intersection
                        person_area = w * h
                        overlap_ratio = intersection_area / person_area
                        
                        if overlap_ratio > 0.3:  # If more than 30% of person is in desk area
                            employee_detected = True
                            is_in_desk_area = True
                    
                    # Draw person boxes with different colors
                    if is_in_desk_area:
                        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 2)  # Red for employee in desk area
                        label = f"Employee: {confidences[i]:.2f}"
                        cv2.putText(frame, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                    else:
                        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)  # Blue for other people
                        label = f"Person: {confidences[i]:.2f}"
                        cv2.putText(frame, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
        
        # Draw monitoring area
        cv2.rectangle(frame, 
                     (self.monitor_area[0], self.monitor_area[1]), 
                     (self.monitor_area[2], self.monitor_area[3]), 
                     (0, 255, 0), 2)  # Green rectangle for monitored area
        
        # Display status on frame
        status_text = "Status: PRESENT" if employee_detected else "Status: ABSENT"
        cv2.putText(frame, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 
                   0.7, (0, 255, 0) if employee_detected else (0, 0, 255), 2)
        
        if not employee_detected and self.absence_start_time is not None:
            absence_duration = time.time() - self.absence_start_time
            duration_text = f"Absence: {absence_duration:.1f}s"
            cv2.putText(frame, duration_text, (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 
                       0.7, (0, 0, 255), 2)
        
        # Add timestamp and source type
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        cv2.putText(frame, f"Time: {timestamp}", (10, 90), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(frame, f"Source: {self.source_type}", (10, 110), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        return frame, employee_detected

    def get_name_from_centroid(self, cx, cy, frame):
        """Extract name from face recognition around a centroid"""
        if not self.face_recognition_enabled:
            return "Unknown"
        
        h, w = frame.shape[:2]
        
        # Define region around centroid for face detection
        rx1, ry1 = max(0, cx - 100), max(0, cy - 150)
        rx2, ry2 = min(w, cx + 100), min(h, cy + 30)
        
        face_region = frame[ry1:ry2, rx1:rx2]
        
        if face_region.size == 0:
            return "Unknown"

        gray_region = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        faces = self.face_cascade.detectMultiScale(gray_region, 1.3, 5)

        # Default if no face / no match
        Name = "Unknown"

        for (x, y, w_face, h_face) in faces:
            crop_img = face_region[y:y+h_face, x:x+w_face, :]
          
            # Use 75x100 => 7500 features to match FACES shape
            resized_img = cv2.resize(crop_img, (75, 100))
            gray = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
            resized_flat = gray.reshape(1, -1) / 255.0

            # Check if feature dimension matches
            if resized_flat.shape[1] != self.FACES.shape[1]:
                self.log_event(f"Feature mismatch: {resized_flat.shape[1]} vs {self.FACES.shape[1]}")
                continue

            distances, indices = self.knn.kneighbors(resized_flat)
            threshold = 28
            
            if distances[0][0] > threshold:
                output = ["Unknown"]
            else:
                output = self.knn.predict(resized_flat)

            entry_no = output[0]
            if entry_no in self.LABELS:
                index = self.LABELS.index(entry_no)
                Name = self.NAMES[index]
                break  # Use first detected face

        return Name

    def run(self, use_simple_mode=False):
        """Main tracking loop
        
        Args:
            use_simple_mode: If True, uses _process_frame method for simpler detection.
                           If False, uses full face recognition and detailed tracking.
        """
        cap = cv2.VideoCapture(0)
        
        if not cap.isOpened():
            self.log_event("Error: Cannot open camera")
            return
        
        # Detect desk area from initial frames
        self.detect_desk_area(cap)
        
        # Reset camera to beginning
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        
        last_cleanup = time.time()
        
        if use_simple_mode:
            self.log_event("Running in SIMPLE mode (presence detection only)")
            self._run_simple_mode(cap)
        else:
            self.log_event("Running in FULL mode (face recognition + tracking)")
            self._run_full_mode(cap, last_cleanup)
    
    def _run_simple_mode(self, cap):
        """Run with simple presence detection"""
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # Process frame for presence detection
            frame, employee_detected = self._process_frame(frame)
            
            # Track absence duration
            if not employee_detected:
                if self.absence_start_time is None:
                    self.absence_start_time = time.time()
            else:
                self.absence_start_time = None
            
            cv2.imshow('Employee Desk Tracking - Simple Mode', frame)
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved to: {self.log_file}")
    
        def _run_full_mode(self, cap, last_cleanup):
         def _run_full_mode(self, cap, last_cleanup):
            """Run with full face recognition and detailed tracking"""
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                
                h, w = frame.shape[:2]
    
                # Create desk mask if needed
                if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                    self.desk_mask = np.zeros((h, w), np.uint8)
                    if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                        cv2.rectangle(self.desk_mask, self.DESK_ROI_PT1, self.DESK_ROI_PT2, 255, -1)
    
                # YOLO person detection
                blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
                self.net.setInput(blob)
                outputs = self.net.forward(self.output_layers)
    
                boxes, confidences = [], []
                for output in outputs:
                    for detection in output:
                        scores = detection[5:]
                        class_id = np.argmax(scores)
                        conf = scores[class_id]
                        if conf > 0.5 and self.classes[class_id] == 'person':
                            cx = int(detection[0] * w)
                            cy = int(detection[1] * h)
                            bw = int(detection[2] * w)
                            bh = int(detection[3] * h)
                            x1 = int(cx - bw / 2)
                            y1 = int(cy - bh / 2)
                            boxes.append([x1, y1, bw, bh])
                            confidences.append(float(conf))
    
                indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, self.nms_threshold)
    
                current_in_desk = set()
                detections = indices.flatten() if len(indices) > 0 else []
    
                for i in detections:
                    x1, y1, bw, bh = boxes[i]
                    x2, y2 = x1 + bw, y1 + bh
                    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
    
                    name = self.get_name_from_centroid(cx, cy, frame)
    
                    if name not in self.trackers:
                        self.trackers[name] = {'centroids': [], 'last_seen': 0}
                    
                    self.trackers[name]['bbox'] = (x1, y1, x2, y2)
                    self.trackers[name]['centroids'].append((cx, cy))
                    self.trackers[name]['last_seen'] = time.time()
                    
                    if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                        self.trackers[name]['centroids'].pop(0)
    
                    # Calculate overlap with desk area
                    overlap_ratio = 0
                    if x2 > x1 and y2 > y1:
                        y_slice = slice(max(0, y1), min(h, y2))
                        x_slice = slice(max(0, x1), min(w, x2))
                        mask_roi = self.desk_mask[y_slice, x_slice]
                        if mask_roi.size > 0:
                            overlap_ratio = np.sum(mask_roi > 0) / mask_roi.size
                    
                    location = 'Desk (Chair)' if overlap_ratio > 0.4 else 'Outside'
                    if 'Desk' in location:
                        current_in_desk.add(name)
    
                    # Draw bounding box and labels
                    color = (0, 255, 0) if 'Desk' in location else (0, 0, 255)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    
                    name_w = len(name) * 15 + 20
                    cv2.rectangle(frame, (x1, y1 - 35), (x1 + name_w, y1), color, -1)
                    cv2.putText(frame, name, (x1 + 10, y1 - 10), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                    cv2.putText(frame, location, (x1, y2 + 25), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
                # Cleanup disappeared trackers
                if time.time() - last_cleanup > 1.0:
                    to_remove = [name for name, data in self.trackers.items() 
                                if time.time() - data['last_seen'] > self.DISAPPEARANCE_TIME]
                    for name in to_remove:
                        del self.trackers[name]
                    last_cleanup = time.time()
    
                # Log entry/exit events
                timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                with open(self.log_file, 'a', newline='') as f:
                    writer = csv.writer(f)
                    for name, data in self.trackers.items():
                        if len(data['centroids']) >= 2:
                            prev_cx, prev_cy = data['centroids'][-2]
                            prev_loc = 'Desk' if (0 <= prev_cy < h and 0 <= prev_cx < w and 
                                                 self.desk_mask[int(prev_cy), int(prev_cx)] > 0) else 'Outside'
                            curr_loc = 'Desk' if name in current_in_desk else 'Outside'
                            if prev_loc != curr_loc:
                                event = 'Entered' if curr_loc == 'Desk' else 'Left'
                                writer.writerow([timestamp, name, event, curr_loc])
    
                # Draw desk ROI
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(frame, self.DESK_ROI_PT1, self.DESK_ROI_PT2, (255, 255, 0), 2)
                    cv2.putText(frame, f'Desk:{len(current_in_desk)} Total:{len(self.trackers)}', 
                               (self.DESK_ROI_PT1[0], self.DESK_ROI_PT1[1] - 10), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
                cv2.imshow('Employee Desk Tracking - Full Mode (Face Recognition)', frame)
                if cv2.waitKey(2) & 0xFF == ord('q'):
                    break
    
            cap.release()
            cv2.destroyAllWindows()
            self.log_event(f"Logs saved to: {self.log_file}")


if __name__ == "__main__":
    import sys
    
    tracker = DeskTracker()
    
    # Check if user wants simple mode or full mode
    mode = "full"  # default
    if len(sys.argv) > 1:
        if sys.argv[1].lower() in ['simple', 's']:
            mode = "simple"
        elif sys.argv[1].lower() in ['full', 'f']:
            mode = "full"
    
    use_simple = (mode == "simple")
    
    print("\n" + "="*60)
    print(f"Starting Desk Tracker in {mode.upper()} mode")
    print("="*60)
    print("\nModes:")
    print("  SIMPLE: Quick presence detection (no face recognition)")
    print("  FULL:   Face recognition + detailed tracking")
    print("\nUsage:")
    print("  python desk_tracking_fixed.py simple  # Simple mode")
    print("  python desk_tracking_fixed.py full    # Full mode")
    print("  python desk_tracking_fixed.py         # Full mode (default)")
    print("\nPress 'q' to quit\n")
    print("="*60 + "\n")
    
    tracker.run(use_simple_mode=use_simple)

In [ ]:
import cv2
import numpy as np
import time
from datetime import datetime
import csv
import os
from collections import defaultdict
import pickle
from sklearn.neighbors import KNeighborsClassifier

class DeskTracker:
    def __init__(self):
        # Load YOLOv4-tiny model 
        cfg_path = 'Yolo_model/yolov4-tiny.cfg'
        names_path = 'Yolo_model/coco.names'
        weights_path = 'Yolo_model/yolov4-tiny.weights'

        # Load names
        with open(names_path, 'r') as f:
            self.classes = [line.strip() for line in f.readlines()]

        # Load model
        self.net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
        
        # Get output layer names
        layer_names = self.net.getLayerNames()
        self.output_layers = [layer_names[i - 1] for i in self.net.getUnconnectedOutLayers()]
        
        # Detection parameters
        self.confidence_threshold = 0.3
        self.nms_threshold = 0.4
        
        # Load face recognition data
        try:
            with open('data/Departments.pkl', 'rb') as f:
                self.DEPARTMENTS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/Entry_numbers.pkl', 'rb') as f:
                self.LABELS = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f)
                self.FACES = self.FACES / 255.0
                print(f"FACES shape: {self.FACES.shape}")
            
            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.LABELS)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []
            self.LABELS = []
            self.face_recognition_enabled = False

        # Load face cascade
        self.face_cascade = cv2.CascadeClassifier('data/haarcascade_frontalface_default.xml')
        
        # Desk ROI - will be detected automatically
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask = None
        
        # Tracking
        self.trackers = {}
        self.MAX_CENTROIDS = 10
        self.DISAPPEARANCE_TIME = 2.0
        
        # Logging
        self.log_file = f'desk_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Name', 'Event', 'Location'])
        
        # Monitor area (will be set from desk detection)
        self.monitor_area = None
        self.absence_start_time = None
        self.source_type = "camera"

    def log_event(self, message):
        """Log events to console"""
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def detect_desk_area(self, cap):
        """Detect the desk area using object detection"""
        self.log_event("Detecting desk area...")
        
        # Take multiple frames to improve detection reliability
        desk_candidates = []
        frame_count = 0
        max_frames = 5
        
        frame = None  # Initialize frame variable
        
        # Process several frames to get a better desk detection
        while frame_count < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
                
            height, width = frame.shape[:2]
            
            # Create a blob from the frame and perform a forward pass
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                break
            
            # Variables to store detection info
            desks = []
            
            # Process detections to find tables/desks/chairs
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    
                    # COCO class IDs: 56=chair, 60=dining table, 63=laptop, 64=mouse, 65=keyboard
                    desk_related_classes = [56, 60, 63, 64, 65]
                    
                    if confidence > self.confidence_threshold and class_id in desk_related_classes:
                        center_x = int(obj_detection[0] * width)
                        center_y = int(obj_detection[1] * height)
                        w = int(obj_detection[2] * width)
                        h = int(obj_detection[3] * height)
                        
                        # Calculate top-left corner coordinates
                        x = int(center_x - w / 2)
                        y = int(center_y - h / 2)
                        
                        desks.append([x, y, w, h, confidence, class_id])
            
            # If we found desk-related objects, add them to candidates
            if desks:
                desk_candidates.extend(desks)
            
            frame_count += 1
        
        # If we don't have enough desk candidates, use a default area
        if len(desk_candidates) < 2:
            self.log_event("Not enough desk objects detected. Using default desk area.")
            if frame is not None:
                height, width = frame.shape[:2]
            else:
                height, width = 480, 640  # Default dimensions
            
            # Default to middle-bottom portion of the frame
            min_x = int(width * 0.2)
            min_y = int(height * 0.4)
            max_x = int(width * 0.8)
            max_y = int(height * 0.9)
        else:
            # Group desk objects to determine the desk area
            x_coords = [d[0] for d in desk_candidates]
            y_coords = [d[1] for d in desk_candidates]
            max_x_coords = [d[0] + d[2] for d in desk_candidates]
            max_y_coords = [d[1] + d[3] for d in desk_candidates]
            
            # Create a bounding box that covers all desk-related objects with padding
            min_x = max(0, int(min(x_coords) - 0.1 * width))
            min_y = max(0, int(min(y_coords) - 0.1 * height))
            max_x = min(width, int(max(max_x_coords) + 0.1 * width))
            max_y = min(height, int(max(max_y_coords) + 0.1 * height))
            
            self.log_event(f"Desk area detected: ({min_x}, {min_y}) to ({max_x}, {max_y})")
        
        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        
        # Set monitor area for _process_frame method
        self.monitor_area = (min_x, min_y, max_x, max_y)
        
        return (min_x, min_y, max_x, max_y)

    def _process_frame(self, frame):
        """Process a frame to detect people and update status"""
        height, width = frame.shape[:2]
        employee_detected = False
        
        # Create a blob from the frame and perform object detection
        blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
        self.net.setInput(blob)
        
        try:
            detections = self.net.forward(self.output_layers)
        except Exception as e:
            self.log_event(f"Error during detection: {e}")
            return frame, False
        
        # Variables to store detection info
        boxes = []
        confidences = []
        class_ids = []
        
        # Process detections to find people
        for detection in detections:
            for obj_detection in detection:
                scores = obj_detection[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]
                
                # Check if the detected object is a person (class ID 0 for COCO dataset)
                if confidence > self.confidence_threshold and class_id == 0:  # 0 is index for person in COCO
                    center_x = int(obj_detection[0] * width)
                    center_y = int(obj_detection[1] * height)
                    w = int(obj_detection[2] * width)
                    h = int(obj_detection[3] * height)
                    
                    # Calculate top-left corner coordinates of the bounding box
                    x = int(center_x - w / 2)
                    y = int(center_y - h / 2)
                    
                    boxes.append([x, y, w, h])
                    confidences.append(float(confidence))
                    class_ids.append(class_id)
        
        # Apply non-maximum suppression to remove redundant overlapping boxes
        if len(boxes) > 0:
            indices = cv2.dnn.NMSBoxes(boxes, confidences, self.confidence_threshold, 0.4)
            
            # Check if any person was detected in the monitored area
            if len(indices) > 0:
                for i in indices.flatten():
                    x, y, w, h = boxes[i]
                    
                    # Calculate person box
                    person_box = (x, y, x+w, y+h)
                    
                    # Calculate intersection with monitor area
                    x_intersection = max(self.monitor_area[0], person_box[0])
                    y_intersection = max(self.monitor_area[1], person_box[1])
                    w_intersection = min(self.monitor_area[2], person_box[2]) - x_intersection
                    h_intersection = min(self.monitor_area[3], person_box[3]) - y_intersection
                    
                    is_in_desk_area = False
                    if w_intersection > 0 and h_intersection > 0:
                        intersection_area = w_intersection * h_intersection
                        person_area = w * h
                        overlap_ratio = intersection_area / person_area
                        
                        if overlap_ratio > 0.3:  # If more than 30% of person is in desk area
                            employee_detected = True
                            is_in_desk_area = True
                    
                    # Draw person boxes with different colors
                    if is_in_desk_area:
                        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 2)  # Red for employee in desk area
                        label = f"Employee: {confidences[i]:.2f}"
                        cv2.putText(frame, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                    else:
                        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)  # Blue for other people
                        label = f"Person: {confidences[i]:.2f}"
                        cv2.putText(frame, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
        
        # Draw monitoring area
        cv2.rectangle(frame, 
                     (self.monitor_area[0], self.monitor_area[1]), 
                     (self.monitor_area[2], self.monitor_area[3]), 
                     (0, 255, 0), 2)  # Green rectangle for monitored area
        
        # Display status on frame
        status_text = "Status: PRESENT" if employee_detected else "Status: ABSENT"
        cv2.putText(frame, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 
                   0.7, (0, 255, 0) if employee_detected else (0, 0, 255), 2)
        
        if not employee_detected and self.absence_start_time is not None:
            absence_duration = time.time() - self.absence_start_time
            duration_text = f"Absence: {absence_duration:.1f}s"
            cv2.putText(frame, duration_text, (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 
                       0.7, (0, 0, 255), 2)
        
        # Add timestamp and source type
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        cv2.putText(frame, f"Time: {timestamp}", (10, 90), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(frame, f"Source: {self.source_type}", (10, 110), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        return frame, employee_detected

    def get_name_from_centroid(self, cx, cy, frame):
        """Extract name from face recognition around a centroid"""
        if not self.face_recognition_enabled:
            return "Unknown"
        
        h, w = frame.shape[:2]
        
        # Define region around centroid for face detection
        rx1, ry1 = max(0, cx - 100), max(0, cy - 150)
        rx2, ry2 = min(w, cx + 100), min(h, cy + 30)
        
        face_region = frame[ry1:ry2, rx1:rx2]
        
        if face_region.size == 0:
            return "Unknown"

        gray_region = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        faces = self.face_cascade.detectMultiScale(gray_region, 1.3, 5)

        # Default if no face / no match
        Name = "Unknown"

        for (x, y, w_face, h_face) in faces:
            crop_img = face_region[y:y+h_face, x:x+w_face, :]
          
            # Use 75x100 => 7500 features to match FACES shape
            resized_img = cv2.resize(crop_img, (75, 100))
            gray = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
            resized_flat = gray.reshape(1, -1) / 255.0

            # Check if feature dimension matches
            if resized_flat.shape[1] != self.FACES.shape[1]:
                self.log_event(f"Feature mismatch: {resized_flat.shape[1]} vs {self.FACES.shape[1]}")
                continue

            distances, indices = self.knn.kneighbors(resized_flat)
            threshold = 28
            
            if distances[0][0] > threshold:
                output = ["Unknown"]
            else:
                output = self.knn.predict(resized_flat)

            entry_no = output[0]
            if entry_no in self.LABELS:
                index = self.LABELS.index(entry_no)
                Name = self.NAMES[index]
                break  # Use first detected face

        return Name

    def run(self, use_simple_mode=False):
        """Main tracking loop
        
        Args:
            use_simple_mode: If True, uses _process_frame method for simpler detection.
                           If False, uses full face recognition and detailed tracking.
        """
        cap = cv2.VideoCapture(0)
        
        if not cap.isOpened():
            self.log_event("Error: Cannot open camera")
            return
        
        # Detect desk area from initial frames
        self.detect_desk_area(cap)
        
        # Reset camera to beginning
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        
        last_cleanup = time.time()
        
        if use_simple_mode:
            self.log_event("Running in SIMPLE mode (presence detection only)")
            self._run_simple_mode(cap)
        else:
            self.log_event("Running in FULL mode (face recognition + tracking)")
            self._run_full_mode(cap, last_cleanup)
    
    def _run_simple_mode(self, cap):
        """Run with simple presence detection"""
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # Process frame for presence detection
            frame, employee_detected = self._process_frame(frame)
            
            # Track absence duration
            if not employee_detected:
                if self.absence_start_time is None:
                    self.absence_start_time = time.time()
            else:
                self.absence_start_time = None
            
            cv2.imshow('Employee Desk Tracking - Simple Mode', frame)
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved to: {self.log_file}")
    
    def _run_full_mode(self, cap, last_cleanup):
        """Run with full face recognition and detailed tracking"""
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            h, w = frame.shape[:2]

            # Create desk mask if needed
            if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                self.desk_mask = np.zeros((h, w), np.uint8)
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(self.desk_mask, self.DESK_ROI_PT1, self.DESK_ROI_PT2, 255, -1)

            # YOLO person detection
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            outputs = self.net.forward(self.output_layers)

            boxes, confidences = [], []
            for output in outputs:
                for detection in output:
                    scores = detection[5:]
                    class_id = np.argmax(scores)
                    conf = scores[class_id]
                    if conf > 0.5 and self.classes[class_id] == 'person':
                        cx = int(detection[0] * w)
                        cy = int(detection[1] * h)
                        bw = int(detection[2] * w)
                        bh = int(detection[3] * h)
                        x1 = int(cx - bw / 2)
                        y1 = int(cy - bh / 2)
                        boxes.append([x1, y1, bw, bh])
                        confidences.append(float(conf))

            indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, self.nms_threshold)

            current_in_desk = set()
            detections = indices.flatten() if len(indices) > 0 else []

            for i in detections:
                x1, y1, bw, bh = boxes[i]
                x2, y2 = x1 + bw, y1 + bh
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

                name = self.get_name_from_centroid(cx, cy, frame)

                if name not in self.trackers:
                    self.trackers[name] = {'centroids': [], 'last_seen': 0}
                
                self.trackers[name]['bbox'] = (x1, y1, x2, y2)
                self.trackers[name]['centroids'].append((cx, cy))
                self.trackers[name]['last_seen'] = time.time()
                
                if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                    self.trackers[name]['centroids'].pop(0)

                # Calculate overlap with desk area
                overlap_ratio = 0
                if x2 > x1 and y2 > y1:
                    y_slice = slice(max(0, y1), min(h, y2))
                    x_slice = slice(max(0, x1), min(w, x2))
                    mask_roi = self.desk_mask[y_slice, x_slice]
                    if mask_roi.size > 0:
                        overlap_ratio = np.sum(mask_roi > 0) / mask_roi.size
                
                location = 'Desk (Chair)' if overlap_ratio > 0.4 else 'Outside'
                if 'Desk' in location:
                    current_in_desk.add(name)

                # Draw bounding box and labels
                color = (0, 255, 0) if 'Desk' in location else (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                
                name_w = len(name) * 15 + 20
                cv2.rectangle(frame, (x1, y1 - 35), (x1 + name_w, y1), color, -1)
                cv2.putText(frame, name, (x1 + 10, y1 - 10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                cv2.putText(frame, location, (x1, y2 + 25), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            # Cleanup disappeared trackers
            if time.time() - last_cleanup > 1.0:
                to_remove = [name for name, data in self.trackers.items() 
                            if time.time() - data['last_seen'] > self.DISAPPEARANCE_TIME]
                for name in to_remove:
                    del self.trackers[name]
                last_cleanup = time.time()

            # Log entry/exit events
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            with open(self.log_file, 'a', newline='') as f:
                writer = csv.writer(f)
                for name, data in self.trackers.items():
                    if len(data['centroids']) >= 2:
                        prev_cx, prev_cy = data['centroids'][-2]
                        prev_loc = 'Desk' if (0 <= prev_cy < h and 0 <= prev_cx < w and 
                                             self.desk_mask[int(prev_cy), int(prev_cx)] > 0) else 'Outside'
                        curr_loc = 'Desk' if name in current_in_desk else 'Outside'
                        if prev_loc != curr_loc:
                            event = 'Entered' if curr_loc == 'Desk' else 'Left'
                            writer.writerow([timestamp, name, event, curr_loc])

            # Draw desk ROI
            if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                cv2.rectangle(frame, self.DESK_ROI_PT1, self.DESK_ROI_PT2, (255, 255, 0), 2)
                cv2.putText(frame, f'Desk:{len(current_in_desk)} Total:{len(self.trackers)}', 
                           (self.DESK_ROI_PT1[0], self.DESK_ROI_PT1[1] - 10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

            cv2.imshow('Employee Desk Tracking - Full Mode (Face Recognition)', frame)
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved to: {self.log_file}")


if __name__ == "__main__":
    import sys
    
    tracker = DeskTracker()
    
    # Check if user wants simple mode or full mode
    mode = "full"  # default
    if len(sys.argv) > 1:
        if sys.argv[1].lower() in ['simple', 's']:
            mode = "simple"
        elif sys.argv[1].lower() in ['full', 'f']:
            mode = "full"
    
    use_simple = (mode == "simple")
    
    print("\n" + "="*60)
    print(f"Starting Desk Tracker in {mode.upper()} mode")
    print("="*60)
    print("\nModes:")
    print("  SIMPLE: Quick presence detection (no face recognition)")
    print("  FULL:   Face recognition + detailed tracking")
    print("\nUsage:")
    print("  python desk_tracking_fixed.py simple  # Simple mode")
    print("  python desk_tracking_fixed.py full    # Full mode")
    print("  python desk_tracking_fixed.py         # Full mode (default)")
    print("\nPress 'q' to quit\n")
    print("="*60 + "\n")
    
    tracker.run(use_simple_mode=use_simple)

In [ ]:
import cv2
import numpy as np
import time
from datetime import datetime
import csv
import os
from collections import defaultdict
import pickle
from sklearn.neighbors import KNeighborsClassifier

class DeskTracker:
    def __init__(self):
        # Load YOLOv4-tiny model 
        cfg_path = 'Yolo_model/yolov4-tiny.cfg'
        names_path = 'Yolo_model/coco.names'
        weights_path = 'Yolo_model/yolov4-tiny.weights'

        # Load names
        with open(names_path, 'r') as f:
            self.classes = [line.strip() for line in f.readlines()]

        # Load model
        self.net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
        
        # Get output layer names
        layer_names = self.net.getLayerNames()
        self.output_layers = [layer_names[i - 1] for i in self.net.getUnconnectedOutLayers()]
        
        # Detection parameters
        self.confidence_threshold = 0.3
        self.nms_threshold = 0.4
        
        # Load face recognition data
        try:
            with open('data/Departments.pkl', 'rb') as f:
                self.DEPARTMENTS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/Entry_numbers.pkl', 'rb') as f:
                self.LABELS = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f)
                self.FACES = self.FACES / 255.0
                print(f"FACES shape: {self.FACES.shape}")
            
            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.LABELS)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []
            self.LABELS = []
            self.DEPARTMENTS = []
            self.EMAILS = []
            self.face_recognition_enabled = False

        # Load face cascade
        self.face_cascade = cv2.CascadeClassifier('data/haarcascade_frontalface_default.xml')
        
        # Desk ROI - will be detected automatically
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask = None
        
        # Tracking
        self.trackers = {}
        self.person_status = {}  # Track last known status of each person
        self.MAX_CENTROIDS = 10
        self.DISAPPEARANCE_TIME = 2.0
        
        # Logging
        timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_file = f'desk_log_{timestamp_str}.csv'
        self.detailed_log_file = f'detailed_log_{timestamp_str}.csv'
        
        # Create main log file
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Name', 'Event', 'Location', 'Department', 'Email'])
        
        # Create detailed log file
        if not os.path.exists(self.detailed_log_file):
            with open(self.detailed_log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Name', 'Location', 'Confidence', 'X', 'Y', 'Department'])
        
        # Monitor area (will be set from desk detection)
        self.monitor_area = None
        self.absence_start_time = None
        self.source_type = "camera"

    def log_event(self, message):
        """Log events to console"""
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def log_person_event(self, name, event, location, department="", email=""):
        """Log person entry/exit events"""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, name, event, location, department, email])
        self.log_event(f"{name} {event} {location}")

    def log_person_position(self, name, location, confidence, x, y, department=""):
        """Log detailed person position"""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.detailed_log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, name, location, f"{confidence:.2f}", x, y, department])

    def detect_desk_area(self, cap):
        """Detect the desk area using object detection"""
        self.log_event("Detecting desk area...")
        
        # Take multiple frames to improve detection reliability
        desk_candidates = []
        frame_count = 0
        max_frames = 5
        
        frame = None  # Initialize frame variable
        
        # Process several frames to get a better desk detection
        while frame_count < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
                
            height, width = frame.shape[:2]
            
            # Create a blob from the frame and perform a forward pass
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                break
            
            # Variables to store detection info
            desks = []
            
            # Process detections to find tables/desks/chairs
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    
                    # COCO class IDs: 56=chair, 60=dining table, 63=laptop, 64=mouse, 65=keyboard
                    desk_related_classes = [56, 60, 63, 64, 65]
                    
                    if confidence > self.confidence_threshold and class_id in desk_related_classes:
                        center_x = int(obj_detection[0] * width)
                        center_y = int(obj_detection[1] * height)
                        w = int(obj_detection[2] * width)
                        h = int(obj_detection[3] * height)
                        
                        # Calculate top-left corner coordinates
                        x = int(center_x - w / 2)
                        y = int(center_y - h / 2)
                        
                        desks.append([x, y, w, h, confidence, class_id])
            
            # If we found desk-related objects, add them to candidates
            if desks:
                desk_candidates.extend(desks)
            
            frame_count += 1
        
        # If we don't have enough desk candidates, use a default area
        if len(desk_candidates) < 2:
            self.log_event("Not enough desk objects detected. Using default desk area.")
            if frame is not None:
                height, width = frame.shape[:2]
            else:
                height, width = 480, 640  # Default dimensions
            
            # Default to middle-bottom portion of the frame
            min_x = int(width * 0.2)
            min_y = int(height * 0.4)
            max_x = int(width * 0.8)
            max_y = int(height * 0.9)
        else:
            # Group desk objects to determine the desk area
            x_coords = [d[0] for d in desk_candidates]
            y_coords = [d[1] for d in desk_candidates]
            max_x_coords = [d[0] + d[2] for d in desk_candidates]
            max_y_coords = [d[1] + d[3] for d in desk_candidates]
            
            # Create a bounding box that covers all desk-related objects with padding
            min_x = max(0, int(min(x_coords) - 0.1 * width))
            min_y = max(0, int(min(y_coords) - 0.1 * height))
            max_x = min(width, int(max(max_x_coords) + 0.1 * width))
            max_y = min(height, int(max(max_y_coords) + 0.1 * height))
            
            self.log_event(f"Desk area detected: ({min_x}, {min_y}) to ({max_x}, {max_y})")
        
        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        
        # Set monitor area for _process_frame method
        self.monitor_area = (min_x, min_y, max_x, max_y)
        
        return (min_x, min_y, max_x, max_y)

    def get_name_from_face(self, face_region):
        """Extract name from face region using face recognition"""
        if not self.face_recognition_enabled or face_region.size == 0:
            return "Unknown", "", ""
        
        gray_region = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        faces = self.face_cascade.detectMultiScale(gray_region, 1.3, 5)

        # Default if no face / no match
        Name = "Unknown"
        Department = ""
        Email = ""

        for (x, y, w_face, h_face) in faces:
            crop_img = face_region[y:y+h_face, x:x+w_face, :]
          
            # Use 75x100 => 7500 features to match FACES shape
            resized_img = cv2.resize(crop_img, (75, 100))
            gray = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
            resized_flat = gray.reshape(1, -1) / 255.0

            # Check if feature dimension matches
            if resized_flat.shape[1] != self.FACES.shape[1]:
                continue

            distances, indices = self.knn.kneighbors(resized_flat)
            threshold = 28
            
            if distances[0][0] > threshold:
                continue
            else:
                output = self.knn.predict(resized_flat)
                entry_no = output[0]
                
                if entry_no in self.LABELS:
                    index = self.LABELS.index(entry_no)
                    Name = self.NAMES[index]
                    Department = self.DEPARTMENTS[index] if index < len(self.DEPARTMENTS) else ""
                    Email = self.EMAILS[index] if index < len(self.EMAILS) else ""
                    break  # Use first detected face

        return Name, Department, Email

    def get_name_from_centroid(self, cx, cy, frame):
        """Extract name from face recognition around a centroid"""
        h, w = frame.shape[:2]
        
        # Define region around centroid for face detection
        rx1, ry1 = max(0, cx - 100), max(0, cy - 150)
        rx2, ry2 = min(w, cx + 100), min(h, cy + 30)
        
        face_region = frame[ry1:ry2, rx1:rx2]
        name, dept, email = self.get_name_from_face(face_region)
        
        return name, dept, email

    def run(self, use_simple_mode=False):
        """Main tracking loop
        
        Args:
            use_simple_mode: If True, uses simpler detection with face recognition.
                           If False, uses full detailed tracking.
        """
        cap = cv2.VideoCapture(0)
        
        if not cap.isOpened():
            self.log_event("Error: Cannot open camera")
            return
        
        # Detect desk area from initial frames
        self.detect_desk_area(cap)
        
        # Reset camera to beginning
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        
        if use_simple_mode:
            self.log_event("Running in SIMPLE mode (face recognition + presence)")
            self._run_simple_mode(cap)
        else:
            self.log_event("Running in FULL mode (face recognition + detailed tracking)")
            self._run_full_mode(cap)
    
    def _run_simple_mode(self, cap):
        """Run with simple presence detection and face recognition"""
        frame_count = 0
        log_interval = 30  # Log every 30 frames
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            h, w = frame.shape[:2]
            
            # YOLO person detection
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                continue
            
            # Variables to store detection info
            boxes = []
            confidences = []
            
            # Process detections to find people
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    
                    # Check if the detected object is a person
                    if confidence > self.confidence_threshold and class_id == 0:
                        center_x = int(obj_detection[0] * w)
                        center_y = int(obj_detection[1] * h)
                        width_box = int(obj_detection[2] * w)
                        height_box = int(obj_detection[3] * h)
                        
                        # Calculate top-left corner coordinates
                        x = int(center_x - width_box / 2)
                        y = int(center_y - height_box / 2)
                        
                        boxes.append([x, y, width_box, height_box])
                        confidences.append(float(confidence))
            
            # Apply non-maximum suppression
            people_in_desk = []
            people_outside_desk = []
            
            if len(boxes) > 0:
                indices = cv2.dnn.NMSBoxes(boxes, confidences, self.confidence_threshold, 0.4)
                
                if len(indices) > 0:
                    for i in indices.flatten():
                        x, y, width_box, height_box = boxes[i]
                        cx, cy = x + width_box // 2, y + height_box // 2
                        
                        # Get name from face recognition
                        name, dept, email = self.get_name_from_centroid(cx, cy, frame)
                        
                        # Calculate person box
                        person_box = (x, y, x + width_box, y + height_box)
                        
                        # Calculate intersection with monitor area
                        x_intersection = max(self.monitor_area[0], person_box[0])
                        y_intersection = max(self.monitor_area[1], person_box[1])
                        w_intersection = min(self.monitor_area[2], person_box[2]) - x_intersection
                        h_intersection = min(self.monitor_area[3], person_box[3]) - y_intersection
                        
                        is_in_desk_area = False
                        if w_intersection > 0 and h_intersection > 0:
                            intersection_area = w_intersection * h_intersection
                            person_area = width_box * height_box
                            overlap_ratio = intersection_area / person_area
                            
                            if overlap_ratio > 0.3:  # If more than 30% of person is in desk area
                                is_in_desk_area = True
                        
                        location = "Inside Desk" if is_in_desk_area else "Outside Desk"
                        
                        # Track status changes and log
                        person_key = name
                        if person_key not in self.person_status:
                            self.person_status[person_key] = location
                            self.log_person_event(name, "Detected", location, dept, email)
                        elif self.person_status[person_key] != location:
                            event = "Entered Desk" if location == "Inside Desk" else "Left Desk"
                            self.log_person_event(name, event, location, dept, email)
                            self.person_status[person_key] = location
                        
                        # Log position periodically
                        if frame_count % log_interval == 0:
                            self.log_person_position(name, location, confidences[i], cx, cy, dept)
                        
                        # Draw person boxes with different colors
                        if is_in_desk_area:
                            people_in_desk.append(name)
                            color = (0, 255, 0)  # Green for employee in desk area
                            label = f"{name} (Inside)"
                        else:
                            people_outside_desk.append(name)
                            color = (0, 0, 255)  # Red for people outside
                            label = f"{name} (Outside)"
                        
                        cv2.rectangle(frame, (x, y), (x + width_box, y + height_box), color, 2)
                        
                        # Draw label background
                        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
                        cv2.rectangle(frame, (x, y - 30), (x + label_size[0] + 10, y), color, -1)
                        cv2.putText(frame, label, (x + 5, y - 10), 
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                        
                        # Show confidence and department
                        if dept:
                            dept_label = f"{dept} ({confidences[i]:.2f})"
                            cv2.putText(frame, dept_label, (x, y + height_box + 20), 
                                      cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            
            # Draw monitoring area
            cv2.rectangle(frame, 
                         (self.monitor_area[0], self.monitor_area[1]), 
                         (self.monitor_area[2], self.monitor_area[3]), 
                         (255, 255, 0), 3)  # Yellow rectangle for monitored area
            
            # Display status on frame
            status_text = f"In Desk: {len(people_in_desk)} | Outside: {len(people_outside_desk)}"
            cv2.rectangle(frame, (5, 5), (550, 120), (0, 0, 0), -1)
            cv2.putText(frame, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 
                       0.7, (0, 255, 0), 2)
            
            # Show names
            if people_in_desk:
                names_in = ", ".join(people_in_desk[:3])  # Show max 3 names
                if len(people_in_desk) > 3:
                    names_in += f" +{len(people_in_desk)-3}"
                cv2.putText(frame, f"Inside: {names_in}", (10, 60), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
            
            # Add timestamp
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {timestamp}", (10, 90), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            cv2.putText(frame, "Mode: SIMPLE | Press 'q' to quit", (10, 110), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            
            cv2.imshow('Employee Desk Tracking - Simple Mode', frame)
            
            frame_count += 1
            
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved to: {self.log_file} and {self.detailed_log_file}")
    
    def _run_full_mode(self, cap):
        """Run with full face recognition and detailed tracking"""
        last_cleanup = time.time()
        frame_count = 0
        log_interval = 30  # Log every 30 frames
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            h, w = frame.shape[:2]

            # Create desk mask if needed
            if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                self.desk_mask = np.zeros((h, w), np.uint8)
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(self.desk_mask, self.DESK_ROI_PT1, self.DESK_ROI_PT2, 255, -1)

            # YOLO person detection
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            outputs = self.net.forward(self.output_layers)

            boxes, confidences = [], []
            for output in outputs:
                for detection in output:
                    scores = detection[5:]
                    class_id = np.argmax(scores)
                    conf = scores[class_id]
                    if conf > 0.5 and self.classes[class_id] == 'person':
                        cx = int(detection[0] * w)
                        cy = int(detection[1] * h)
                        bw = int(detection[2] * w)
                        bh = int(detection[3] * h)
                        x1 = int(cx - bw / 2)
                        y1 = int(cy - bh / 2)
                        boxes.append([x1, y1, bw, bh])
                        confidences.append(float(conf))

            indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, self.nms_threshold)

            current_in_desk = {}  # name -> (dept, email)
            current_outside = {}  # name -> (dept, email)
            detections = indices.flatten() if len(indices) > 0 else []

            for i in detections:
                x1, y1, bw, bh = boxes[i]
                x2, y2 = x1 + bw, y1 + bh
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

                name, dept, email = self.get_name_from_centroid(cx, cy, frame)

                if name not in self.trackers:
                    self.trackers[name] = {
                        'centroids': [], 
                        'last_seen': 0,
                        'department': dept,
                        'email': email
                    }
                
                self.trackers[name]['bbox'] = (x1, y1, x2, y2)
                self.trackers[name]['centroids'].append((cx, cy))
                self.trackers[name]['last_seen'] = time.time()
                self.trackers[name]['department'] = dept
                self.trackers[name]['email'] = email
                
                if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                    self.trackers[name]['centroids'].pop(0)

                # Calculate overlap with desk area
                overlap_ratio = 0
                if x2 > x1 and y2 > y1:
                    y_slice = slice(max(0, y1), min(h, y2))
                    x_slice = slice(max(0, x1), min(w, x2))
                    mask_roi = self.desk_mask[y_slice, x_slice]
                    if mask_roi.size > 0:
                        overlap_ratio = np.sum(mask_roi > 0) / mask_roi.size
                
                location = 'Inside Desk' if overlap_ratio > 0.4 else 'Outside Desk'
                
                if 'Inside' in location:
                    current_in_desk[name] = (dept, email)
                else:
                    current_outside[name] = (dept, email)
                
                # Track status changes
                if name not in self.person_status:
                    self.person_status[name] = location
                    self.log_person_event(name, "Detected", location, dept, email)
                elif self.person_status[name] != location:
                    event = "Entered Desk" if location == "Inside Desk" else "Left Desk"
                    self.log_person_event(name, event, location, dept, email)
                    self.person_status[name] = location
                
                # Log position periodically
                if frame_count % log_interval == 0:
                    self.log_person_position(name, location, confidences[i], cx, cy, dept)

                # Draw bounding box and labels
                color = (0, 255, 0) if 'Inside' in location else (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                
                # Label with name and location
                label = f"{name} ({location.split()[0]})"
                label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)[0]
                cv2.rectangle(frame, (x1, y1 - 35), (x1 + label_size[0] + 10, y1), color, -1)
                cv2.putText(frame, label, (x1 + 5, y1 - 10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                
                # Show department and confidence
                info_label = f"{dept if dept else 'N/A'} | {confidences[i]:.2f}"
                cv2.putText(frame, info_label, (x1, y2 + 20), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            # Cleanup disappeared trackers
            if time.time() - last_cleanup > 1.0:
                to_remove = [name for name, data in self.trackers.items() 
                            if time.time() - data['last_seen'] > self.DISAPPEARANCE_TIME]
                for name in to_remove:
                    if name in self.person_status:
                        dept = self.trackers[name].get('department', '')
                        email = self.trackers[name].get('email', '')
                        self.log_person_event(name, "Disappeared", "Unknown", dept, email)
                        del self.person_status[name]
                    del self.trackers[name]
                last_cleanup = time.time()

            # Draw desk ROI
            if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                cv2.rectangle(frame, self.DESK_ROI_PT1, self.DESK_ROI_PT2, (255, 255, 0), 3)
                
                # Status overlay
                cv2.rectangle(frame, (5, 5), (600, 140), (0, 0, 0), -1)
                cv2.putText(frame, f'Inside Desk: {len(current_in_desk)} | Outside: {len(current_outside)} | Total: {len(self.trackers)}', 
                           (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                
                # Show people inside
                if current_in_desk:
                    names_in = ", ".join(list(current_in_desk.keys())[:3])
                    if len(current_in_desk) > 3:
                        names_in += f" +{len(current_in_desk)-3}"
                    cv2.putText(frame, f"Inside: {names_in}", (10, 60), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
                
                # Show people outside
                if current_outside:
                    names_out = ", ".join(list(current_outside.keys())[:3])
                    if len(current_outside) > 3:
                        names_out += f" +{len(current_outside)-3}"
                    cv2.putText(frame, f"Outside: {names_out}", (10, 85), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                
                # Timestamp
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                cv2.putText(frame, f"Time: {timestamp}", (10, 110), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                cv2.putText(frame, "Mode: FULL | Press 'q' to quit", (10, 130), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

            cv2.imshow('Employee Desk Tracking - Full Mode (Face Recognition)', frame)
            
            frame_count += 1
            
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved to: {self.log_file} and {self.detailed_log_file}")


if __name__ == "__main__":
    import sys
    
    tracker = DeskTracker()
    
    # Check if user wants simple mode or full mode
    mode = "full"  # default
    if len(sys.argv) > 1:
        if sys.argv[1].lower() in ['simple', 's']:
            mode = "simple"
        elif sys.argv[1].lower() in ['full', 'f']:
            mode = "full"
    
    use_simple = (mode == "simple")
    
    print("\n" + "="*60)
    print(f"Starting Desk Tracker in {mode.upper()} mode")
    print("="*60)
    print("\nModes:")
    print("  SIMPLE: Face recognition + presence detection")
    print("  FULL:   Face recognition + detailed tracking + history")
    print("\nFeatures:")
    print("  ✓ Automatic desk area detection")
    print("  ✓ Face recognition with names")
    print("  ✓ Inside/Outside desk tracking")
    print("  ✓ Event logging (Entry/Exit)")
    print("  ✓ Detailed position logging")
    print("  ✓ Department and email tracking")
    print("\nUsage:")
    print("  python desk_tracking_complete.py simple  # Simple mode")
    print("  python desk_tracking_complete.py full    # Full mode")
    print("  python desk_tracking_complete.py         # Full mode (default)")
    print("\nPress 'q' to quit")
    print("="*60 + "\n")
    
    tracker.run(use_simple_mode=use_simple)

In [ ]:
import cv2
import numpy as np
import time
from datetime import datetime
import csv
import os
from collections import defaultdict
import pickle
from sklearn.neighbors import KNeighborsClassifier

class DeskTracker:
    def __init__(self):
        # Load YOLOv4-tiny model 
        cfg_path = 'Yolo_model/yolov4-tiny.cfg'
        names_path = 'Yolo_model/coco.names'
        weights_path = 'Yolo_model/yolov4-tiny.weights'

        # Load names
        with open(names_path, 'r') as f:
            self.classes = [line.strip() for line in f.readlines()]

        # Load model
        self.net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
        
        # Get output layer names
        layer_names = self.net.getLayerNames()
        self.output_layers = [layer_names[i - 1] for i in self.net.getUnconnectedOutLayers()]
        
        # Detection parameters
        self.confidence_threshold = 0.3
        self.nms_threshold = 0.4
        
        # Load face recognition data
        try:
            with open('data/Departments.pkl', 'rb') as f:
                self.DEPARTMENTS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/Entry_numbers.pkl', 'rb') as f:
                self.LABELS = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f)
                self.FACES = self.FACES / 255.0
                print(f"FACES shape: {self.FACES.shape}")
            
            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.LABELS)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []
            self.LABELS = []
            self.DEPARTMENTS = []
            self.EMAILS = []
            self.face_recognition_enabled = False

        # Load face cascade
        self.face_cascade = cv2.CascadeClassifier('data/haarcascade_frontalface_default.xml')
        
        # Desk ROI - will be detected automatically
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask = None
        
        # Tracking
        self.trackers = {}
        self.person_status = {}  # Track last known status of each person
        self.MAX_CENTROIDS = 10
        self.DISAPPEARANCE_TIME = 2.0
        
        # Logging
        timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_file = f'desk_log_{timestamp_str}.csv'
        self.detailed_log_file = f'detailed_log_{timestamp_str}.csv'
        
        # Create main log file
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Name', 'Event', 'Location', 'Department', 'Email'])
        
        # Create detailed log file
        if not os.path.exists(self.detailed_log_file):
            with open(self.detailed_log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Name', 'Location', 'Confidence', 'X', 'Y', 'Department'])
        
        # Monitor area (will be set from desk detection)
        self.monitor_area = None
        self.absence_start_time = None
        self.source_type = "camera"
        
        # Frame size settings for better visualization
        self.display_width = 1280  # Increased from default
        self.display_height = 720  # Increased from default

    def log_event(self, message):
        """Log events to console"""
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def log_person_event(self, name, event, location, department="", email=""):
        """Log person entry/exit events"""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, name, event, location, department, email])
        self.log_event(f"{name} {event} {location}")

    def log_person_position(self, name, location, confidence, x, y, department=""):
        """Log detailed person position"""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.detailed_log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, name, location, f"{confidence:.2f}", x, y, department])

    def detect_desk_area(self, cap):
        """Detect the desk area using object detection"""
        self.log_event("Detecting desk area...")
        
        # Take multiple frames to improve detection reliability
        desk_candidates = []
        frame_count = 0
        max_frames = 5
        
        frame = None  # Initialize frame variable
        
        # Process several frames to get a better desk detection
        while frame_count < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
                
            height, width = frame.shape[:2]
            
            # Create a blob from the frame and perform a forward pass
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                break
            
            # Variables to store detection info
            desks = []
            
            # Process detections to find tables/desks/chairs
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    
                    # COCO class IDs: 56=chair, 60=dining table, 63=laptop, 64=mouse, 65=keyboard
                    desk_related_classes = [56, 60, 63, 64, 65]
                    
                    if confidence > self.confidence_threshold and class_id in desk_related_classes:
                        center_x = int(obj_detection[0] * width)
                        center_y = int(obj_detection[1] * height)
                        w = int(obj_detection[2] * width)
                        h = int(obj_detection[3] * height)
                        
                        # Calculate top-left corner coordinates
                        x = int(center_x - w / 2)
                        y = int(center_y - h / 2)
                        
                        desks.append([x, y, w, h, confidence, class_id])
            
            # If we found desk-related objects, add them to candidates
            if desks:
                desk_candidates.extend(desks)
            
            frame_count += 1
        
        # If we don't have enough desk candidates, use a default area
        if len(desk_candidates) < 2:
            self.log_event("Not enough desk objects detected. Using default desk area.")
            if frame is not None:
                height, width = frame.shape[:2]
            else:
                height, width = 480, 640  # Default dimensions
            
            # Default to middle-bottom portion of the frame
            min_x = int(width * 0.2)
            min_y = int(height * 0.4)
            max_x = int(width * 0.8)
            max_y = int(height * 0.9)
        else:
            # Group desk objects to determine the desk area
            x_coords = [d[0] for d in desk_candidates]
            y_coords = [d[1] for d in desk_candidates]
            max_x_coords = [d[0] + d[2] for d in desk_candidates]
            max_y_coords = [d[1] + d[3] for d in desk_candidates]
            
            # Create a bounding box that covers all desk-related objects with padding
            min_x = max(0, int(min(x_coords) - 0.1 * width))
            min_y = max(0, int(min(y_coords) - 0.1 * height))
            max_x = min(width, int(max(max_x_coords) + 0.1 * width))
            max_y = min(height, int(max(max_y_coords) + 0.1 * height))
            
            self.log_event(f"Desk area detected: ({min_x}, {min_y}) to ({max_x}, {max_y})")
        
        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        
        # Set monitor area for _process_frame method
        self.monitor_area = (min_x, min_y, max_x, max_y)
        
        return (min_x, min_y, max_x, max_y)

    def get_name_from_face(self, face_region):
        """Extract name from face region using face recognition"""
        if not self.face_recognition_enabled or face_region.size == 0:
            return "Unknown", "", ""
        
        gray_region = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        faces = self.face_cascade.detectMultiScale(gray_region, 1.3, 5)

        # Default if no face / no match
        Name = "Unknown"
        Department = ""
        Email = ""

        for (x, y, w_face, h_face) in faces:
            crop_img = face_region[y:y+h_face, x:x+w_face, :]
          
            # Use 75x100 => 7500 features to match FACES shape
            resized_img = cv2.resize(crop_img, (75, 100))
            gray = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
            resized_flat = gray.reshape(1, -1) / 255.0

            # Check if feature dimension matches
            if resized_flat.shape[1] != self.FACES.shape[1]:
                continue

            distances, indices = self.knn.kneighbors(resized_flat)
            threshold = 28
            
            if distances[0][0] > threshold:
                continue
            else:
                output = self.knn.predict(resized_flat)
                entry_no = output[0]
                
                if entry_no in self.LABELS:
                    index = self.LABELS.index(entry_no)
                    Name = self.NAMES[index]
                    Department = self.DEPARTMENTS[index] if index < len(self.DEPARTMENTS) else ""
                    Email = self.EMAILS[index] if index < len(self.EMAILS) else ""
                    break  # Use first detected face

        return Name, Department, Email

    def get_name_from_centroid(self, cx, cy, frame):
        """Extract name from face recognition around a centroid"""
        h, w = frame.shape[:2]
        
        # Define region around centroid for face detection
        rx1, ry1 = max(0, cx - 100), max(0, cy - 150)
        rx2, ry2 = min(w, cx + 100), min(h, cy + 30)
        
        face_region = frame[ry1:ry2, rx1:rx2]
        name, dept, email = self.get_name_from_face(face_region)
        
        return name, dept, email

    def run(self, use_simple_mode=False):
        """Main tracking loop
        
        Args:
            use_simple_mode: If True, uses simpler detection with face recognition.
                           If False, uses full detailed tracking.
        """
        cap = cv2.VideoCapture(0)
        
        if not cap.isOpened():
            self.log_event("Error: Cannot open camera")
            return
        
        # Set camera resolution for better quality
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
        cap.set(cv2.CAP_PROP_FPS, 30)
        
        # Get actual resolution
        actual_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        actual_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.log_event(f"Camera resolution: {actual_width}x{actual_height}")
        
        # Detect desk area from initial frames
        self.detect_desk_area(cap)
        
        # Reset camera to beginning
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        
        if use_simple_mode:
            self.log_event("Running in SIMPLE mode (face recognition + presence)")
            self._run_simple_mode(cap)
        else:
            self.log_event("Running in FULL mode (face recognition + detailed tracking)")
            self._run_full_mode(cap)
    
    def _run_simple_mode(self, cap):
        """Run with simple presence detection and face recognition"""
        frame_count = 0
        log_interval = 30  # Log every 30 frames
        
        # Create named window with resizable option
        cv2.namedWindow('Employee Desk Tracking - Simple Mode', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking - Simple Mode', self.display_width, self.display_height)
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            h, w = frame.shape[:2]
            
            # YOLO person detection
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                continue
            
            # Variables to store detection info
            boxes = []
            confidences = []
            
            # Process detections to find people
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    
                    # Check if the detected object is a person
                    if confidence > self.confidence_threshold and class_id == 0:
                        center_x = int(obj_detection[0] * w)
                        center_y = int(obj_detection[1] * h)
                        width_box = int(obj_detection[2] * w)
                        height_box = int(obj_detection[3] * h)
                        
                        # Calculate top-left corner coordinates
                        x = int(center_x - width_box / 2)
                        y = int(center_y - height_box / 2)
                        
                        boxes.append([x, y, width_box, height_box])
                        confidences.append(float(confidence))
            
            # Apply non-maximum suppression
            people_in_desk = []
            people_outside_desk = []
            
            if len(boxes) > 0:
                indices = cv2.dnn.NMSBoxes(boxes, confidences, self.confidence_threshold, 0.4)
                
                if len(indices) > 0:
                    for i in indices.flatten():
                        x, y, width_box, height_box = boxes[i]
                        cx, cy = x + width_box // 2, y + height_box // 2
                        
                        # Get name from face recognition
                        name, dept, email = self.get_name_from_centroid(cx, cy, frame)
                        
                        # Calculate person box
                        person_box = (x, y, x + width_box, y + height_box)
                        
                        # Calculate intersection with monitor area
                        x_intersection = max(self.monitor_area[0], person_box[0])
                        y_intersection = max(self.monitor_area[1], person_box[1])
                        w_intersection = min(self.monitor_area[2], person_box[2]) - x_intersection
                        h_intersection = min(self.monitor_area[3], person_box[3]) - y_intersection
                        
                        is_in_desk_area = False
                        if w_intersection > 0 and h_intersection > 0:
                            intersection_area = w_intersection * h_intersection
                            person_area = width_box * height_box
                            overlap_ratio = intersection_area / person_area
                            
                            if overlap_ratio > 0.3:  # If more than 30% of person is in desk area
                                is_in_desk_area = True
                        
                        location = "Inside Desk" if is_in_desk_area else "Outside Desk"
                        
                        # Track status changes and log
                        person_key = name
                        if person_key not in self.person_status:
                            self.person_status[person_key] = location
                            self.log_person_event(name, "Detected", location, dept, email)
                        elif self.person_status[person_key] != location:
                            event = "Entered Desk" if location == "Inside Desk" else "Left Desk"
                            self.log_person_event(name, event, location, dept, email)
                            self.person_status[person_key] = location
                        
                        # Log position periodically
                        if frame_count % log_interval == 0:
                            self.log_person_position(name, location, confidences[i], cx, cy, dept)
                        
                        # Draw person boxes with different colors and thicker lines
                        if is_in_desk_area:
                            people_in_desk.append(name)
                            color = (0, 255, 0)  # Green for employee in desk area
                            label = f"{name} (Inside)"
                        else:
                            people_outside_desk.append(name)
                            color = (0, 0, 255)  # Red for people outside
                            label = f"{name} (Outside)"
                        
                        cv2.rectangle(frame, (x, y), (x + width_box, y + height_box), color, 3)
                        
                        # Draw label background with larger size
                        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)[0]
                        cv2.rectangle(frame, (x, y - 40), (x + label_size[0] + 15, y), color, -1)
                        cv2.putText(frame, label, (x + 8, y - 12), 
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
                        
                        # Show confidence and department with larger text
                        if dept:
                            dept_label = f"{dept} ({confidences[i]:.2f})"
                            cv2.putText(frame, dept_label, (x, y + height_box + 25), 
                                      cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            
            # Draw monitoring area with thicker line
            cv2.rectangle(frame, 
                         (self.monitor_area[0], self.monitor_area[1]), 
                         (self.monitor_area[2], self.monitor_area[3]), 
                         (255, 255, 0), 4)  # Thicker yellow rectangle for monitored area
            
            # Add corner markers for desk area
            marker_size = 30
            # Top-left
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[1]), 
                    (self.monitor_area[0] + marker_size, self.monitor_area[1]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[1]), 
                    (self.monitor_area[0], self.monitor_area[1] + marker_size), (0, 255, 255), 6)
            # Top-right
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[1]), 
                    (self.monitor_area[2] - marker_size, self.monitor_area[1]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[1]), 
                    (self.monitor_area[2], self.monitor_area[1] + marker_size), (0, 255, 255), 6)
            # Bottom-left
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[3]), 
                    (self.monitor_area[0] + marker_size, self.monitor_area[3]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[3]), 
                    (self.monitor_area[0], self.monitor_area[3] - marker_size), (0, 255, 255), 6)
            # Bottom-right
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[3]), 
                    (self.monitor_area[2] - marker_size, self.monitor_area[3]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[3]), 
                    (self.monitor_area[2], self.monitor_area[3] - marker_size), (0, 255, 255), 6)
            
            # Display status on frame with larger text
            cv2.rectangle(frame, (5, 5), (700, 150), (0, 0, 0), -1)
            status_text = f"In Desk: {len(people_in_desk)} | Outside: {len(people_outside_desk)}"
            cv2.putText(frame, status_text, (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 
                       0.9, (0, 255, 0), 3)
            
            # Show names with larger text
            if people_in_desk:
                names_in = ", ".join(people_in_desk[:3])  # Show max 3 names
                if len(people_in_desk) > 3:
                    names_in += f" +{len(people_in_desk)-3}"
                cv2.putText(frame, f"Inside: {names_in}", (15, 75), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            
            # Add timestamp with larger text
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {timestamp}", (15, 110), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.putText(frame, "Mode: SIMPLE | Press 'q' to quit", (15, 140), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 200, 0), 2)
            
            cv2.imshow('Employee Desk Tracking - Simple Mode', frame)
            
            frame_count += 1
            
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved to: {self.log_file} and {self.detailed_log_file}")
    
    def _run_full_mode(self, cap):
        """Run with full face recognition and detailed tracking"""
        last_cleanup = time.time()
        frame_count = 0
        log_interval = 30  # Log every 30 frames
        
        # Create named window with resizable option
        cv2.namedWindow('Employee Desk Tracking - Full Mode', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking - Full Mode', self.display_width, self.display_height)
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            h, w = frame.shape[:2]

            # Create desk mask if needed
            if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                self.desk_mask = np.zeros((h, w), np.uint8)
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(self.desk_mask, self.DESK_ROI_PT1, self.DESK_ROI_PT2, 255, -1)

            # YOLO person detection
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            outputs = self.net.forward(self.output_layers)

            boxes, confidences = [], []
            for output in outputs:
                for detection in output:
                    scores = detection[5:]
                    class_id = np.argmax(scores)
                    conf = scores[class_id]
                    if conf > 0.5 and self.classes[class_id] == 'person':
                        cx = int(detection[0] * w)
                        cy = int(detection[1] * h)
                        bw = int(detection[2] * w)
                        bh = int(detection[3] * h)
                        x1 = int(cx - bw / 2)
                        y1 = int(cy - bh / 2)
                        boxes.append([x1, y1, bw, bh])
                        confidences.append(float(conf))

            indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, self.nms_threshold)

            current_in_desk = {}  # name -> (dept, email)
            current_outside = {}  # name -> (dept, email)
            detections = indices.flatten() if len(indices) > 0 else []

            for i in detections:
                x1, y1, bw, bh = boxes[i]
                x2, y2 = x1 + bw, y1 + bh
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

                name, dept, email = self.get_name_from_centroid(cx, cy, frame)

                if name not in self.trackers:
                    self.trackers[name] = {
                        'centroids': [], 
                        'last_seen': 0,
                        'department': dept,
                        'email': email
                    }
                
                self.trackers[name]['bbox'] = (x1, y1, x2, y2)
                self.trackers[name]['centroids'].append((cx, cy))
                self.trackers[name]['last_seen'] = time.time()
                self.trackers[name]['department'] = dept
                self.trackers[name]['email'] = email
                
                if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                    self.trackers[name]['centroids'].pop(0)

                # Calculate overlap with desk area
                overlap_ratio = 0
                if x2 > x1 and y2 > y1:
                    y_slice = slice(max(0, y1), min(h, y2))
                    x_slice = slice(max(0, x1), min(w, x2))
                    mask_roi = self.desk_mask[y_slice, x_slice]
                    if mask_roi.size > 0:
                        overlap_ratio = np.sum(mask_roi > 0) / mask_roi.size
                
                location = 'Inside Desk' if overlap_ratio > 0.4 else 'Outside Desk'
                
                if 'Inside' in location:
                    current_in_desk[name] = (dept, email)
                else:
                    current_outside[name] = (dept, email)
                
                # Track status changes
                if name not in self.person_status:
                    self.person_status[name] = location
                    self.log_person_event(name, "Detected", location, dept, email)
                elif self.person_status[name] != location:
                    event = "Entered Desk" if location == "Inside Desk" else "Left Desk"
                    self.log_person_event(name, event, location, dept, email)
                    self.person_status[name] = location
                
                # Log position periodically
                if frame_count % log_interval == 0:
                    self.log_person_position(name, location, confidences[i], cx, cy, dept)

                # Draw bounding box and labels with thicker lines
                color = (0, 255, 0) if 'Inside' in location else (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                
                # Label with name and location with larger text
                label = f"{name} ({location.split()[0]})"
                label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)[0]
                cv2.rectangle(frame, (x1, y1 - 45), (x1 + label_size[0] + 15, y1), color, -1)
                cv2.putText(frame, label, (x1 + 8, y1 - 15), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
                
                # Show department and confidence with larger text
                info_label = f"{dept if dept else 'N/A'} | {confidences[i]:.2f}"
                cv2.putText(frame, info_label, (x1, y2 + 25), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # Cleanup disappeared trackers
            if time.time() - last_cleanup > 1.0:
                to_remove = [name for name, data in self.trackers.items() 
                            if time.time() - data['last_seen'] > self.DISAPPEARANCE_TIME]
                for name in to_remove:
                    if name in self.person_status:
                        dept = self.trackers[name].get('department', '')
                        email = self.trackers[name].get('email', '')
                        self.log_person_event(name, "Disappeared", "Unknown", dept, email)
                        del self.person_status[name]
                    del self.trackers[name]
                last_cleanup = time.time()

            # Draw desk ROI with enhanced visualization
            if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                # Draw main rectangle
                cv2.rectangle(frame, self.DESK_ROI_PT1, self.DESK_ROI_PT2, (255, 255, 0), 4)
                
                # Add corner markers
                marker_size = 40
                # Top-left
                cv2.line(frame, self.DESK_ROI_PT1, 
                        (self.DESK_ROI_PT1[0] + marker_size, self.DESK_ROI_PT1[1]), (0, 255, 255), 6)
                cv2.line(frame, self.DESK_ROI_PT1, 
                        (self.DESK_ROI_PT1[0], self.DESK_ROI_PT1[1] + marker_size), (0, 255, 255), 6)
                # Top-right
                cv2.line(frame, (self.DESK_ROI_PT2[0], self.DESK_ROI_PT1[1]), 
                        (self.DESK_ROI_PT2[0] - marker_size, self.DESK_ROI_PT1[1]), (0, 255, 255), 6)
                cv2.line(frame, (self.DESK_ROI_PT2[0], self.DESK_ROI_PT1[1]), 
                        (self.DESK_ROI_PT2[0], self.DESK_ROI_PT1[1] + marker_size), (0, 255, 255), 6)
                # Bottom-left
                cv2.line(frame, (self.DESK_ROI_PT1[0], self.DESK_ROI_PT2[1]), 
                        (self.DESK_ROI_PT1[0] + marker_size, self.DESK_ROI_PT2[1]), (0, 255, 255), 6)
                cv2.line(frame, (self.DESK_ROI_PT1[0], self.DESK_ROI_PT2[1]), 
                        (self.DESK_ROI_PT1[0], self.DESK_ROI_PT2[1] - marker_size), (0, 255, 255), 6)
                # Bottom-right
                cv2.line(frame, self.DESK_ROI_PT2, 
                        (self.DESK_ROI_PT2[0] - marker_size, self.DESK_ROI_PT2[1]), (0, 255, 255), 6)
                cv2.line(frame, self.DESK_ROI_PT2, 
                        (self.DESK_ROI_PT2[0], self.DESK_ROI_PT2[1] - marker_size), (0, 255, 255), 6)
                
                # Status overlay with larger text
                cv2.rectangle(frame, (5, 5), (750, 175), (0, 0, 0), -1)
                cv2.putText(frame, f'Inside: {len(current_in_desk)} | Outside: {len(current_outside)} | Total: {len(self.trackers)}', 
                           (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 3)
                
                # Show people inside with larger text
                if current_in_desk:
                    names_in = ", ".join(list(current_in_desk.keys())[:3])
                    if len(current_in_desk) > 3:
                        names_in += f" +{len(current_in_desk)-3}"
                    cv2.putText(frame, f"Inside: {names_in}", (15, 75), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                
                # Show people outside with larger text
                if current_outside:
                    names_out = ", ".join(list(current_outside.keys())[:3])
                    if len(current_outside) > 3:
                        names_out += f" +{len(current_outside)-3}"
                    cv2.putText(frame, f"Outside: {names_out}", (15, 110), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                
                # Timestamp with larger text
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                cv2.putText(frame, f"Time: {timestamp}", (15, 140), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(frame, "Mode: FULL | Press 'q' to quit", (15, 165), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 200, 0), 2)

            cv2.imshow('Employee Desk Tracking - Full Mode (Face Recognition)', frame)
            
            frame_count += 1
            
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved to: {self.log_file} and {self.detailed_log_file}")


if __name__ == "__main__":
    import sys
    
    tracker = DeskTracker()
    
    # Check if user wants simple mode or full mode
    mode = "full"  # default
    if len(sys.argv) > 1:
        if sys.argv[1].lower() in ['simple', 's']:
            mode = "simple"
        elif sys.argv[1].lower() in ['full', 'f']:
            mode = "full"
    
    use_simple = (mode == "simple")
    
    print("\n" + "="*60)
    print(f"Starting Desk Tracker in {mode.upper()} mode")
    print("="*60)
    print("\nModes:")
    print("  SIMPLE: Face recognition + presence detection")
    print("  FULL:   Face recognition + detailed tracking + history")
    print("\nFeatures:")
    print("  ✓ Automatic desk area detection")
    print("  ✓ Face recognition with names")
    print("  ✓ Inside/Outside desk tracking")
    print("  ✓ Event logging (Entry/Exit)")
    print("  ✓ Detailed position logging")
    print("  ✓ Department and email tracking")
    print("\nUsage:")
    print("  python desk_tracking_complete.py simple  # Simple mode")
    print("  python desk_tracking_complete.py full    # Full mode")
    print("  python desk_tracking_complete.py         # Full mode (default)")
    print("\nPress 'q' to quit")
    print("="*60 + "\n")
    
    tracker.run(use_simple_mode=use_simple)

In [ ]:
import cv2
import numpy as np
import time
from datetime import datetime
import csv
import os
from collections import defaultdict
import pickle
from sklearn.neighbors import KNeighborsClassifier

class DeskTracker:
    def __init__(self):
        # Load YOLOv4-tiny model 
        cfg_path = 'Yolo_model/yolov4-tiny.cfg'
        names_path = 'Yolo_model/coco.names'
        weights_path = 'Yolo_model/yolov4-tiny.weights'

        # Load names
        with open(names_path, 'r') as f:
            self.classes = [line.strip() for line in f.readlines()]

        # Load model
        self.net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
        
        # Get output layer names
        layer_names = self.net.getLayerNames()
        self.output_layers = [layer_names[i - 1] for i in self.net.getUnconnectedOutLayers()]
        
        # Detection parameters
        self.confidence_threshold = 0.3
        self.nms_threshold = 0.4
        
        # Face recognition parameters
        self.face_recognition_threshold = 35  # Lowered for better recognition
        self.face_min_size = (30, 30)  # Minimum face size to detect
        
        # Load face recognition data
        try:
            with open('data/Departments.pkl', 'rb') as f:
                self.DEPARTMENTS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/EmployeesID.pkl', 'rb') as f:
                self.LABELS = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f)
                self.FACES = self.FACES / 255.0
                print(f"FACES shape: {self.FACES.shape}")
            
            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.LABELS)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []
            self.LABELS = []
            self.DEPARTMENTS = []
            self.EMAILS = []
            self.face_recognition_enabled = False

        # Load face cascade
        self.face_cascade = cv2.CascadeClassifier('data/haarcascade_frontalface_default.xml')
        
        # Desk ROI - will be detected automatically
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask = None
        
        # Tracking
        self.trackers = {}
        self.person_status = {}  # Track last known status of each person
        self.MAX_CENTROIDS = 10
        self.DISAPPEARANCE_TIME = 2.0
        
        # Logging
        timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_file = f'desk_log_{timestamp_str}.csv'
        self.detailed_log_file = f'detailed_log_{timestamp_str}.csv'
        
        # Create main log file
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Name', 'Event', 'Location', 'Department', 'Email'])
        
        # Create detailed log file
        if not os.path.exists(self.detailed_log_file):
            with open(self.detailed_log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Name', 'Location', 'Confidence', 'X', 'Y', 'Department'])
        
        # Monitor area (will be set from desk detection)
        self.monitor_area = None
        self.absence_start_time = None
        self.source_type = "camera"
        
        # Frame size settings for better visualization
        self.display_width = 1280  # Increased from default
        self.display_height = 720  # Increased from default
        
        # Debug mode - set to True to visualize face detection regions
        self.debug_mode = False

    def log_event(self, message):
        """Log events to console"""
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def log_person_event(self, name, event, location, department="", email=""):
        """Log person entry/exit events"""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, name, event, location, department, email])
        self.log_event(f"{name} {event} {location}")

    def log_person_position(self, name, location, confidence, x, y, department=""):
        """Log detailed person position"""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.detailed_log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, name, location, f"{confidence:.2f}", x, y, department])

    def detect_desk_area(self, cap):
        """Detect the desk area using object detection"""
        self.log_event("Detecting desk area...")
        
        # Take multiple frames to improve detection reliability
        desk_candidates = []
        frame_count = 0
        max_frames = 5
        
        frame = None  # Initialize frame variable
        
        # Process several frames to get a better desk detection
        while frame_count < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
                
            height, width = frame.shape[:2]
            
            # Create a blob from the frame and perform a forward pass
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                break
            
            # Variables to store detection info
            desks = []
            
            # Process detections to find tables/desks/chairs
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    
                    # COCO class IDs: 56=chair, 60=dining table, 63=laptop, 64=mouse, 65=keyboard
                    desk_related_classes = [56, 60, 63, 64, 65]
                    
                    if confidence > self.confidence_threshold and class_id in desk_related_classes:
                        center_x = int(obj_detection[0] * width)
                        center_y = int(obj_detection[1] * height)
                        w = int(obj_detection[2] * width)
                        h = int(obj_detection[3] * height)
                        
                        # Calculate top-left corner coordinates
                        x = int(center_x - w / 2)
                        y = int(center_y - h / 2)
                        
                        desks.append([x, y, w, h, confidence, class_id])
            
            # If we found desk-related objects, add them to candidates
            if desks:
                desk_candidates.extend(desks)
            
            frame_count += 1
        
        # If we don't have enough desk candidates, use a default area
        if len(desk_candidates) < 2:
            self.log_event("Not enough desk objects detected. Using default desk area.")
            if frame is not None:
                height, width = frame.shape[:2]
            else:
                height, width = 480, 640  # Default dimensions
            
            # Default to middle-bottom portion of the frame
            min_x = int(width * 0.2)
            min_y = int(height * 0.4)
            max_x = int(width * 0.8)
            max_y = int(height * 0.9)
        else:
            # Group desk objects to determine the desk area
            x_coords = [d[0] for d in desk_candidates]
            y_coords = [d[1] for d in desk_candidates]
            max_x_coords = [d[0] + d[2] for d in desk_candidates]
            max_y_coords = [d[1] + d[3] for d in desk_candidates]
            
            # Create a bounding box that covers all desk-related objects with padding
            min_x = max(0, int(min(x_coords) - 0.1 * width))
            min_y = max(0, int(min(y_coords) - 0.1 * height))
            max_x = min(width, int(max(max_x_coords) + 0.1 * width))
            max_y = min(height, int(max(max_y_coords) + 0.1 * height))
            
            self.log_event(f"Desk area detected: ({min_x}, {min_y}) to ({max_x}, {max_y})")
        
        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        
        # Set monitor area for _process_frame method
        self.monitor_area = (min_x, min_y, max_x, max_y)
        
        return (min_x, min_y, max_x, max_y)

    def get_name_from_face(self, face_region):
        """Extract name from face region using face recognition"""
        if not self.face_recognition_enabled or face_region.size == 0:
            return "People", "General", ""
        
        gray_region = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        
        # Try multiple scale factors for better detection
        faces = []
        for scale in [1.1, 1.2, 1.3]:
            detected = self.face_cascade.detectMultiScale(
                gray_region, scale, 5, minSize=self.face_min_size
            )
            if len(detected) > 0:
                faces = detected
                break

        # Default if no face / no match
        Name = "People"
        Department = "General"
        Email = ""

        for (x, y, w_face, h_face) in faces:
            crop_img = face_region[y:y+h_face, x:x+w_face, :]
            
            if crop_img.size == 0:
                continue
          
            # Use 75x100 => 7500 features to match FACES shape
            try:
                resized_img = cv2.resize(crop_img, (75, 100))
                gray = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
                resized_flat = gray.reshape(1, -1) / 255.0

                # Check if feature dimension matches
                if resized_flat.shape[1] != self.FACES.shape[1]:
                    continue

                distances, indices = self.knn.kneighbors(resized_flat)
                
                # Use configurable threshold
                if distances[0][0] <= self.face_recognition_threshold:
                    output = self.knn.predict(resized_flat)
                    entry_no = output[0]
                    
                    if entry_no in self.LABELS:
                        index = self.LABELS.index(entry_no)
                        Name = self.NAMES[index]
                        Department = self.DEPARTMENTS[index] if index < len(self.DEPARTMENTS) else "Employee"
                        Email = self.EMAILS[index] if index < len(self.EMAILS) else ""
                        break  # Use first detected face
            except Exception as e:
                # If any error occurs, continue to next face
                continue

        return Name, Department, Email

    def get_name_from_centroid(self, cx, cy, frame):
        """Extract name from face recognition around a centroid"""
        h, w = frame.shape[:2]
        
        # Define LARGER region around centroid for better face detection
        # Expand the search area significantly
        rx1, ry1 = max(0, cx - 150), max(0, cy - 200)
        rx2, ry2 = min(w, cx + 150), min(h, cy + 100)
        
        # Draw face search region in debug mode
        if self.debug_mode:
            cv2.rectangle(frame, (rx1, ry1), (rx2, ry2), (255, 0, 255), 2)
            cv2.putText(frame, "Face Search", (rx1, ry1 - 5), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 255), 1)
        
        face_region = frame[ry1:ry2, rx1:rx2]
        name, dept, email = self.get_name_from_face(face_region)
        
        return name, dept, email

    def run(self, use_simple_mode=False):
        """Main tracking loop
        
        Args:
            use_simple_mode: If True, uses simpler detection with face recognition.
                           If False, uses full detailed tracking.
        """
        cap = cv2.VideoCapture(0)
        
        if not cap.isOpened():
            self.log_event("Error: Cannot open camera")
            return
        
        # Set camera resolution for better quality
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
        cap.set(cv2.CAP_PROP_FPS, 30)
        
        # Get actual resolution
        actual_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        actual_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.log_event(f"Camera resolution: {actual_width}x{actual_height}")
        
        # Detect desk area from initial frames
        self.detect_desk_area(cap)
        
        # Reset camera to beginning
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        
        if use_simple_mode:
            self.log_event("Running in SIMPLE mode (face recognition + presence)")
            self._run_simple_mode(cap)
        else:
            self.log_event("Running in FULL mode (face recognition + detailed tracking)")
            self._run_full_mode(cap)
    
    def _run_simple_mode(self, cap):
        """Run with simple presence detection and face recognition"""
        frame_count = 0
        log_interval = 30  # Log every 30 frames
        
        # Create named window with resizable option
        cv2.namedWindow('Employee Desk Tracking - Simple Mode', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking - Simple Mode', self.display_width, self.display_height)
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            h, w = frame.shape[:2]
            
            # YOLO person detection
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                continue
            
            # Variables to store detection info
            boxes = []
            confidences = []
            
            # Process detections to find people
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    
                    # Check if the detected object is a person
                    if confidence > self.confidence_threshold and class_id == 0:
                        center_x = int(obj_detection[0] * w)
                        center_y = int(obj_detection[1] * h)
                        width_box = int(obj_detection[2] * w)
                        height_box = int(obj_detection[3] * h)
                        
                        # Calculate top-left corner coordinates
                        x = int(center_x - width_box / 2)
                        y = int(center_y - height_box / 2)
                        
                        boxes.append([x, y, width_box, height_box])
                        confidences.append(float(confidence))
            
            # Apply non-maximum suppression
            people_in_desk = []
            people_outside_desk = []
            
            if len(boxes) > 0:
                indices = cv2.dnn.NMSBoxes(boxes, confidences, self.confidence_threshold, 0.4)
                
                if len(indices) > 0:
                    for i in indices.flatten():
                        x, y, width_box, height_box = boxes[i]
                        cx, cy = x + width_box // 2, y + height_box // 2
                        
                        # Get name from face recognition
                        name, dept, email = self.get_name_from_centroid(cx, cy, frame)
                        
                        # Calculate person box
                        person_box = (x, y, x + width_box, y + height_box)
                        
                        # Calculate intersection with monitor area
                        x_intersection = max(self.monitor_area[0], person_box[0])
                        y_intersection = max(self.monitor_area[1], person_box[1])
                        w_intersection = min(self.monitor_area[2], person_box[2]) - x_intersection
                        h_intersection = min(self.monitor_area[3], person_box[3]) - y_intersection
                        
                        is_in_desk_area = False
                        if w_intersection > 0 and h_intersection > 0:
                            intersection_area = w_intersection * h_intersection
                            person_area = width_box * height_box
                            overlap_ratio = intersection_area / person_area
                            
                            if overlap_ratio > 0.3:  # If more than 30% of person is in desk area
                                is_in_desk_area = True
                        
                        location = "Inside Desk" if is_in_desk_area else "Outside Desk"
                        
                        # Track status changes and log
                        person_key = name
                        if person_key not in self.person_status:
                            self.person_status[person_key] = location
                            self.log_person_event(name, "Detected", location, dept, email)
                        elif self.person_status[person_key] != location:
                            event = "Entered Desk" if location == "Inside Desk" else "Left Desk"
                            self.log_person_event(name, event, location, dept, email)
                            self.person_status[person_key] = location
                        
                        # Log position periodically
                        if frame_count % log_interval == 0:
                            self.log_person_position(name, location, confidences[i], cx, cy, dept)
                        
                        # Draw person boxes with different colors and thicker lines
                        if is_in_desk_area:
                            people_in_desk.append(name)
                            color = (0, 255, 0)  # Green for employee in desk area
                            label = f"{name} - Employee (Inside)"
                        else:
                            people_outside_desk.append(name)
                            color = (0, 0, 255)  # Red for people outside
                            label = f"{name} - Visitor (Outside)"
                        
                        cv2.rectangle(frame, (x, y), (x + width_box, y + height_box), color, 3)
                        
                        # Draw label background with larger size
                        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)[0]
                        cv2.rectangle(frame, (x, y - 40), (x + label_size[0] + 15, y), color, -1)
                        cv2.putText(frame, label, (x + 8, y - 12), 
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
                        
                        # Show confidence and department with larger text
                        dept_display = dept if dept else "Employee" if is_in_desk_area else "Visitor"
                        dept_label = f"{dept_display} | Conf: {confidences[i]:.2f}"
                        cv2.putText(frame, dept_label, (x, y + height_box + 25), 
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            
            # Draw monitoring area with thicker line
            cv2.rectangle(frame, 
                         (self.monitor_area[0], self.monitor_area[1]), 
                         (self.monitor_area[2], self.monitor_area[3]), 
                         (255, 255, 0), 4)  # Thicker yellow rectangle for monitored area
            
            # Add corner markers for desk area
            marker_size = 30
            # Top-left
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[1]), 
                    (self.monitor_area[0] + marker_size, self.monitor_area[1]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[1]), 
                    (self.monitor_area[0], self.monitor_area[1] + marker_size), (0, 255, 255), 6)
            # Top-right
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[1]), 
                    (self.monitor_area[2] - marker_size, self.monitor_area[1]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[1]), 
                    (self.monitor_area[2], self.monitor_area[1] + marker_size), (0, 255, 255), 6)
            # Bottom-left
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[3]), 
                    (self.monitor_area[0] + marker_size, self.monitor_area[3]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[3]), 
                    (self.monitor_area[0], self.monitor_area[3] - marker_size), (0, 255, 255), 6)
            # Bottom-right
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[3]), 
                    (self.monitor_area[2] - marker_size, self.monitor_area[3]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[3]), 
                    (self.monitor_area[2], self.monitor_area[3] - marker_size), (0, 255, 255), 6)
            
            # Display status on frame with larger text
            cv2.rectangle(frame, (5, 5), (700, 150), (0, 0, 0), -1)
            status_text = f"In Desk: {len(people_in_desk)} | Outside: {len(people_outside_desk)}"
            cv2.putText(frame, status_text, (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 
                       0.9, (0, 255, 0), 3)
            
            # Show names with larger text
            if people_in_desk:
                names_in = ", ".join(people_in_desk[:3])  # Show max 3 names
                if len(people_in_desk) > 3:
                    names_in += f" +{len(people_in_desk)-3}"
                cv2.putText(frame, f"Inside: {names_in}", (15, 75), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            
            # Add timestamp with larger text
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {timestamp}", (15, 110), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.putText(frame, "Mode: SIMPLE | Press 'q' to quit", (15, 140), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 200, 0), 2)
            
            cv2.imshow('Employee Desk Tracking - Simple Mode', frame)
            
            frame_count += 1
            
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved to: {self.log_file} and {self.detailed_log_file}")
    
    def _run_full_mode(self, cap):
        """Run with full face recognition and detailed tracking"""
        last_cleanup = time.time()
        frame_count = 0
        log_interval = 30  # Log every 30 frames
        
        # Create named window with resizable option
        cv2.namedWindow('Employee Desk Tracking - Full Mode', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking - Full Mode', self.display_width, self.display_height)
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            h, w = frame.shape[:2]

            # Create desk mask if needed
            if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                self.desk_mask = np.zeros((h, w), np.uint8)
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(self.desk_mask, self.DESK_ROI_PT1, self.DESK_ROI_PT2, 255, -1)

            # YOLO person detection
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            outputs = self.net.forward(self.output_layers)

            boxes, confidences = [], []
            for output in outputs:
                for detection in output:
                    scores = detection[5:]
                    class_id = np.argmax(scores)
                    conf = scores[class_id]
                    if conf > 0.5 and self.classes[class_id] == 'person':
                        cx = int(detection[0] * w)
                        cy = int(detection[1] * h)
                        bw = int(detection[2] * w)
                        bh = int(detection[3] * h)
                        x1 = int(cx - bw / 2)
                        y1 = int(cy - bh / 2)
                        boxes.append([x1, y1, bw, bh])
                        confidences.append(float(conf))

            indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, self.nms_threshold)

            current_in_desk = {}  # name -> (dept, email)
            current_outside = {}  # name -> (dept, email)
            detections = indices.flatten() if len(indices) > 0 else []

            for i in detections:
                x1, y1, bw, bh = boxes[i]
                x2, y2 = x1 + bw, y1 + bh
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

                name, dept, email = self.get_name_from_centroid(cx, cy, frame)

                if name not in self.trackers:
                    self.trackers[name] = {
                        'centroids': [], 
                        'last_seen': 0,
                        'department': dept,
                        'email': email
                    }
                
                self.trackers[name]['bbox'] = (x1, y1, x2, y2)
                self.trackers[name]['centroids'].append((cx, cy))
                self.trackers[name]['last_seen'] = time.time()
                self.trackers[name]['department'] = dept
                self.trackers[name]['email'] = email
                
                if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                    self.trackers[name]['centroids'].pop(0)

                # Calculate overlap with desk area
                overlap_ratio = 0
                if x2 > x1 and y2 > y1:
                    y_slice = slice(max(0, y1), min(h, y2))
                    x_slice = slice(max(0, x1), min(w, x2))
                    mask_roi = self.desk_mask[y_slice, x_slice]
                    if mask_roi.size > 0:
                        overlap_ratio = np.sum(mask_roi > 0) / mask_roi.size
                
                location = 'Inside Desk' if overlap_ratio > 0.4 else 'Outside Desk'
                
                if 'Inside' in location:
                    current_in_desk[name] = (dept, email)
                else:
                    current_outside[name] = (dept, email)
                
                # Track status changes
                if name not in self.person_status:
                    self.person_status[name] = location
                    self.log_person_event(name, "Detected", location, dept, email)
                elif self.person_status[name] != location:
                    event = "Entered Desk" if location == "Inside Desk" else "Left Desk"
                    self.log_person_event(name, event, location, dept, email)
                    self.person_status[name] = location
                
                # Log position periodically
                if frame_count % log_interval == 0:
                    self.log_person_position(name, location, confidences[i], cx, cy, dept)

                # Draw bounding box and labels with thicker lines
                color = (0, 255, 0) if 'Inside' in location else (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                
                # Label with name and location with larger text
                role = "Employee" if 'Inside' in location else "Visitor"
                label = f"{name} - {role} ({location.split()[0]})"
                label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)[0]
                cv2.rectangle(frame, (x1, y1 - 45), (x1 + label_size[0] + 15, y1), color, -1)
                cv2.putText(frame, label, (x1 + 8, y1 - 15), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
                
                # Show department and confidence with larger text
                dept_display = dept if dept else ("Employee" if 'Inside' in location else "Visitor")
                info_label = f"{dept_display} | Conf: {confidences[i]:.2f}"
                cv2.putText(frame, info_label, (x1, y2 + 25), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # Cleanup disappeared trackers
            if time.time() - last_cleanup > 1.0:
                to_remove = [name for name, data in self.trackers.items() 
                            if time.time() - data['last_seen'] > self.DISAPPEARANCE_TIME]
                for name in to_remove:
                    if name in self.person_status:
                        dept = self.trackers[name].get('department', '')
                        email = self.trackers[name].get('email', '')
                        self.log_person_event(name, "Disappeared", "Unknown", dept, email)
                        del self.person_status[name]
                    del self.trackers[name]
                last_cleanup = time.time()

            # Draw desk ROI with enhanced visualization
            if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                # Draw main rectangle
                cv2.rectangle(frame, self.DESK_ROI_PT1, self.DESK_ROI_PT2, (255, 255, 0), 4)
                
                # Add corner markers
                marker_size = 40
                # Top-left
                cv2.line(frame, self.DESK_ROI_PT1, 
                        (self.DESK_ROI_PT1[0] + marker_size, self.DESK_ROI_PT1[1]), (0, 255, 255), 6)
                cv2.line(frame, self.DESK_ROI_PT1, 
                        (self.DESK_ROI_PT1[0], self.DESK_ROI_PT1[1] + marker_size), (0, 255, 255), 6)
                # Top-right
                cv2.line(frame, (self.DESK_ROI_PT2[0], self.DESK_ROI_PT1[1]), 
                        (self.DESK_ROI_PT2[0] - marker_size, self.DESK_ROI_PT1[1]), (0, 255, 255), 6)
                cv2.line(frame, (self.DESK_ROI_PT2[0], self.DESK_ROI_PT1[1]), 
                        (self.DESK_ROI_PT2[0], self.DESK_ROI_PT1[1] + marker_size), (0, 255, 255), 6)
                # Bottom-left
                cv2.line(frame, (self.DESK_ROI_PT1[0], self.DESK_ROI_PT2[1]), 
                        (self.DESK_ROI_PT1[0] + marker_size, self.DESK_ROI_PT2[1]), (0, 255, 255), 6)
                cv2.line(frame, (self.DESK_ROI_PT1[0], self.DESK_ROI_PT2[1]), 
                        (self.DESK_ROI_PT1[0], self.DESK_ROI_PT2[1] - marker_size), (0, 255, 255), 6)
                # Bottom-right
                cv2.line(frame, self.DESK_ROI_PT2, 
                        (self.DESK_ROI_PT2[0] - marker_size, self.DESK_ROI_PT2[1]), (0, 255, 255), 6)
                cv2.line(frame, self.DESK_ROI_PT2, 
                        (self.DESK_ROI_PT2[0], self.DESK_ROI_PT2[1] - marker_size), (0, 255, 255), 6)
                
                # Status overlay with larger text
                cv2.rectangle(frame, (5, 5), (750, 175), (0, 0, 0), -1)
                cv2.putText(frame, f'Inside: {len(current_in_desk)} | Outside: {len(current_outside)} | Total: {len(self.trackers)}', 
                           (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 3)
                
                # Show people inside with larger text
                if current_in_desk:
                    names_in = ", ".join(list(current_in_desk.keys())[:3])
                    if len(current_in_desk) > 3:
                        names_in += f" +{len(current_in_desk)-3}"
                    cv2.putText(frame, f"Inside: {names_in}", (15, 75), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                
                # Show people outside with larger text
                if current_outside:
                    names_out = ", ".join(list(current_outside.keys())[:3])
                    if len(current_outside) > 3:
                        names_out += f" +{len(current_outside)-3}"
                    cv2.putText(frame, f"Outside: {names_out}", (15, 110), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                
                # Timestamp with larger text
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                cv2.putText(frame, f"Time: {timestamp}", (15, 140), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(frame, "Mode: FULL | Press 'q' to quit", (15, 165), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 200, 0), 2)

            cv2.imshow('Employee Desk Tracking - Full Mode (Face Recognition)', frame)
            
            frame_count += 1
            
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved to: {self.log_file} and {self.detailed_log_file}")


if __name__ == "__main__":
    import sys
    
    tracker = DeskTracker()
    
    # Check if user wants simple mode or full mode
    mode = "full"  # default
    if len(sys.argv) > 1:
        if sys.argv[1].lower() in ['simple', 's']:
            mode = "simple"
        elif sys.argv[1].lower() in ['full', 'f']:
            mode = "full"
    
    use_simple = (mode == "simple")
    
    print("\n" + "="*60)
    print(f"Starting Desk Tracker in {mode.upper()} mode")
    print("="*60)
    print("\nModes:")
    print("  SIMPLE: Face recognition + presence detection")
    print("  FULL:   Face recognition + detailed tracking + history")
    print("\nFeatures:")
    print("  ✓ Automatic desk area detection")
    print("  ✓ Face recognition with names")
    print("  ✓ Inside/Outside desk tracking")
    print("  ✓ Event logging (Entry/Exit)")
    print("  ✓ Detailed position logging")
    print("  ✓ Department and email tracking")
    print("\nUsage:")
    print("  python desk_tracking_complete.py simple  # Simple mode")
    print("  python desk_tracking_complete.py full    # Full mode")
    print("  python desk_tracking_complete.py         # Full mode (default)")
    print("\nPress 'q' to quit")
    print("="*60 + "\n")
    
    tracker.run(use_simple_mode=use_simple)

In [ ]:
import cv2
import numpy as np
import time
from datetime import datetime
import csv
import os
from collections import defaultdict
import pickle
from sklearn.neighbors import KNeighborsClassifier

class DeskTracker:
    def __init__(self):
        # Load YOLOv4-tiny model 
        cfg_path = 'Yolo_model/yolov4-tiny.cfg'
        names_path = 'Yolo_model/coco.names'
        weights_path = 'Yolo_model/yolov4-tiny.weights'

        # Load names
        with open(names_path, 'r') as f:
            self.classes = [line.strip() for line in f.readlines()]

        # Load model
        self.net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
        
        # Get output layer names
        layer_names = self.net.getLayerNames()
        self.output_layers = [layer_names[i - 1] for i in self.net.getUnconnectedOutLayers()]
        
        # Detection parameters
        self.confidence_threshold = 0.3
        self.nms_threshold = 0.4
        
        # Face recognition parameters
        self.face_recognition_threshold = 35  # Lowered for better recognition
        self.face_min_size = (30, 30)  # Minimum face size to detect
        
        # Load face recognition data
        try:
            with open('data/Departments.pkl', 'rb') as f:
                self.DEPARTMENTS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/EmployeeIDs.pkl', 'rb') as f:
                self.EmployeeIDs = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f)
                self.FACES = self.FACES / 255.0
                print(f"FACES shape: {self.FACES.shape}")
            
            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.EmployeeIDs)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []
            self.EmployeeIDs = []
            self.DEPARTMENTS = []
            self.EMAILS = []
            self.face_recognition_enabled = False

        # Load face cascade
        self.face_cascade = cv2.CascadeClassifier('data/haarcascade_frontalface_default.xml')
        
        # Desk ROI - will be detected automatically
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask = None
        
        # Tracking
        self.trackers = {}
        self.person_status = {}  # Track last known status of each person
        self.MAX_CENTROIDS = 10
        self.DISAPPEARANCE_TIME = 2.0
        
        # Status tracking for employees and visitors
        self.employee_absence_start = None
        self.visitor_absence_start = None
        self.last_employee_present = None
        self.last_visitor_present = None
        
        # Logging
        timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_file = f'desk_log_{timestamp_str}.csv'
        self.detailed_log_file = f'detailed_log_{timestamp_str}.csv'
        
        # Create main log file
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Name', 'Event', 'Location', 'EmployeeID', 'Email'])
        
        # Create detailed log file
        if not os.path.exists(self.detailed_log_file):
            with open(self.detailed_log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Name', 'Location', 'Confidence', 'X', 'Y', 'EmployeeID'])
        
        # Monitor area (will be set from desk detection)
        self.monitor_area = None
        self.absence_start_time = None
        self.source_type = "camera"
        
        # Frame size settings for better visualization
        self.display_width = 1280  # Increased from default
        self.display_height = 720  # Increased from default
        
        # Debug mode - set to True to visualize face detection regions
        # This will show the search areas where the system looks for faces
        self.debug_mode = False  # Set to True to see face search regions

    def log_event(self, message):
        """Log events to console"""
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def log_person_event(self, name, event, location, EmployeeID="", email=""):
        """Log person entry/exit events"""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, name, event, location, EmployeeID, email])
        self.log_event(f"{name} {event} {location}")

    def log_person_position(self, name, location, confidence, x, y, EmployeeID=""):
        """Log detailed person position"""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.detailed_log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, name, location, f"{confidence:.2f}", x, y, EmployeeID])

    def detect_desk_area(self, cap):
        """Detect the desk area using object detection"""
        self.log_event("Detecting desk area...")
        
        # Take multiple frames to improve detection reliability
        desk_candidates = []
        frame_count = 0
        max_frames = 5
        
        frame = None  # Initialize frame variable
        
        # Process several frames to get a better desk detection
        while frame_count < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
                
            height, width = frame.shape[:2]
            
            # Create a blob from the frame and perform a forward pass
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                break
            
            # Variables to store detection info
            desks = []
            
            # Process detections to find tables/desks/chairs
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    
                    # COCO class IDs: 56=chair, 60=dining table, 63=laptop, 64=mouse, 65=keyboard
                    desk_related_classes = [56, 60, 63, 64, 65]
                    
                    if confidence > self.confidence_threshold and class_id in desk_related_classes:
                        center_x = int(obj_detection[0] * width)
                        center_y = int(obj_detection[1] * height)
                        w = int(obj_detection[2] * width)
                        h = int(obj_detection[3] * height)
                        
                        # Calculate top-left corner coordinates
                        x = int(center_x - w / 2)
                        y = int(center_y - h / 2)
                        
                        desks.append([x, y, w, h, confidence, class_id])
            
            # If we found desk-related objects, add them to candidates
            if desks:
                desk_candidates.extend(desks)
            
            frame_count += 1
        
        # If we don't have enough desk candidates, use a default area
        if len(desk_candidates) < 2:
            self.log_event("Not enough desk objects detected. Using default desk area.")
            if frame is not None:
                height, width = frame.shape[:2]
            else:
                height, width = 480, 640  # Default dimensions
            
            # Default to middle-bottom portion of the frame
            min_x = int(width * 0.2)
            min_y = int(height * 0.4)
            max_x = int(width * 0.8)
            max_y = int(height * 0.9)
        else:
            # Group desk objects to determine the desk area
            x_coords = [d[0] for d in desk_candidates]
            y_coords = [d[1] for d in desk_candidates]
            max_x_coords = [d[0] + d[2] for d in desk_candidates]
            max_y_coords = [d[1] + d[3] for d in desk_candidates]
            
            # Create a bounding box that covers all desk-related objects with padding
            min_x = max(0, int(min(x_coords) - 0.1 * width))
            min_y = max(0, int(min(y_coords) - 0.1 * height))
            max_x = min(width, int(max(max_x_coords) + 0.1 * width))
            max_y = min(height, int(max(max_y_coords) + 0.1 * height))
            
            self.log_event(f"Desk area detected: ({min_x}, {min_y}) to ({max_x}, {max_y})")
        
        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        
        # Set monitor area for _process_frame method
        self.monitor_area = (min_x, min_y, max_x, max_y)
        
        return (min_x, min_y, max_x, max_y)

    def get_name_from_face(self, face_region):
        """Extract name from face region using face recognition"""
        if not self.face_recognition_enabled or face_region.size == 0:
            return "People", "N/A", ""
        
        # Check if region is too small
        if face_region.shape[0] < 30 or face_region.shape[1] < 30:
            return "People", "N/A", ""
        
        gray_region = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        
        # Try multiple scale factors and parameters for better detection
        faces = []
        detection_params = [
            (1.1, 5, (20, 20)),  # More sensitive
            (1.05, 4, (25, 25)), # Very sensitive
            (1.2, 5, (30, 30)),  # Original
            (1.3, 4, (35, 35)),  # Less sensitive but more accurate
            (1.15, 3, (25, 25)), # Medium sensitivity
        ]
        
        for scale, neighbors, min_size in detection_params:
            detected = self.face_cascade.detectMultiScale(
                gray_region, scale, neighbors, minSize=min_size
            )
            if len(detected) > 0:
                faces = detected
                break

        # Default if no face / no match
        Name = "People"
        EmployeeID = "N/A"
        Email = ""

        for (x, y, w_face, h_face) in faces:
            crop_img = face_region[y:y+h_face, x:x+w_face, :]
            
            if crop_img.size == 0:
                continue
          
            # Use 75x100 => 7500 features to match FACES shape
            try:
                resized_img = cv2.resize(crop_img, (75, 100))
                gray = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
                resized_flat = gray.reshape(1, -1) / 255.0

                # Check if feature dimension matches
                if resized_flat.shape[1] != self.FACES.shape[1]:
                    continue

                distances, indices = self.knn.kneighbors(resized_flat)
                
                # Use configurable threshold
                if distances[0][0] <= self.face_recognition_threshold:
                    output = self.knn.predict(resized_flat)
                    emp_id = output[0]
                    
                    if emp_id in self.EmployeeIDs:
                        index = self.EmployeeIDs.index(emp_id)
                        Name = self.NAMES[index]
                        EmployeeID = self.EmployeeIDs[index] if index < len(self.EmployeeIDs) else "N/A"
                        Email = self.EMAILS[index] if index < len(self.EMAILS) else ""
                        break  # Use first detected face
            except Exception as e:
                # If any error occurs, continue to next face
                continue

        return Name, EmployeeID, Email

    def get_name_from_centroid(self, cx, cy, frame):
        """Extract name from face recognition around a centroid"""
        h, w = frame.shape[:2]
        
        # Try multiple search regions with different sizes and positions
        # This helps catch faces in different positions relative to the body centroid
        search_regions = [
            # Main region - centered above centroid (where head usually is)
            (max(0, cx - 180), max(0, cy - 280), min(w, cx + 180), min(h, cy + 50)),
            # Wider region
            (max(0, cx - 220), max(0, cy - 320), min(w, cx + 220), min(h, cy + 80)),
            # Higher up (for people far from camera)
            (max(0, cx - 150), max(0, cy - 350), min(w, cx + 150), min(h, cy)),
            # Lower region (for people close to camera)
            (max(0, cx - 200), max(0, cy - 200), min(w, cx + 200), min(h, cy + 150)),
        ]
        
        # Try each search region until we find a recognized face
        for idx, (rx1, ry1, rx2, ry2) in enumerate(search_regions):
            # Draw face search region in debug mode
            if self.debug_mode:
                color = [(255, 0, 255), (0, 255, 255), (255, 255, 0), (255, 128, 0)][idx]
                cv2.rectangle(frame, (rx1, ry1), (rx2, ry2), color, 2)
                cv2.putText(frame, f"Search {idx+1}", (rx1, ry1 - 5), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            
            face_region = frame[ry1:ry2, rx1:rx2]
            name, EmployeeID, email = self.get_name_from_face(face_region)
            
            # If we found a recognized person (not "People"), return immediately
            if name != "People":
                return name, EmployeeID, email
        
        # If no recognized face found in any region, return "People"
        return "People", "N/A", ""

    def run(self, use_simple_mode=False):
        """Main tracking loop
        
        Args:
            use_simple_mode: If True, uses simpler detection with face recognition.
                           If False, uses full detailed tracking.
        """
        cap = cv2.VideoCapture(0)
        
        if not cap.isOpened():
            self.log_event("Error: Cannot open camera")
            return
        
        # Set camera resolution for better quality
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
        cap.set(cv2.CAP_PROP_FPS, 30)
        
        # Get actual resolution
        actual_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        actual_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.log_event(f"Camera resolution: {actual_width}x{actual_height}")
        
        # Detect desk area from initial frames
        self.detect_desk_area(cap)
        
        # Reset camera to beginning
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        
        if use_simple_mode:
            self.log_event("Running in SIMPLE mode (face recognition + presence)")
            self._run_simple_mode(cap)
        else:
            self.log_event("Running in FULL mode (face recognition + detailed tracking)")
            self._run_full_mode(cap)
    
    def _run_simple_mode(self, cap):
        """Run with simple presence detection and face recognition"""
        frame_count = 0
        log_interval = 30  # Log every 30 frames
        
        # Create named window with resizable option
        cv2.namedWindow('Employee Desk Tracking - Simple Mode', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking - Simple Mode', self.display_width, self.display_height)
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            h, w = frame.shape[:2]
            
            # YOLO person detection
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                continue
            
            # Variables to store detection info
            boxes = []
            confidences = []
            
            # Process detections to find people
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    
                    # Check if the detected object is a person
                    if confidence > self.confidence_threshold and class_id == 0:
                        center_x = int(obj_detection[0] * w)
                        center_y = int(obj_detection[1] * h)
                        width_box = int(obj_detection[2] * w)
                        height_box = int(obj_detection[3] * h)
                        
                        # Calculate top-left corner coordinates
                        x = int(center_x - width_box / 2)
                        y = int(center_y - height_box / 2)
                        
                        boxes.append([x, y, width_box, height_box])
                        confidences.append(float(confidence))
            
            # Apply non-maximum suppression
            people_in_desk = []
            people_outside_desk = []
            employee_detected = False
            visitor_detected = False
            
            if len(boxes) > 0:
                indices = cv2.dnn.NMSBoxes(boxes, confidences, self.confidence_threshold, 0.4)
                
                if len(indices) > 0:
                    for i in indices.flatten():
                        x, y, width_box, height_box = boxes[i]
                        cx, cy = x + width_box // 2, y + height_box // 2
                        
                        # Get name from face recognition
                        name, EmployeeID, email = self.get_name_from_centroid(cx, cy, frame)
                        
                        # Calculate person box
                        person_box = (x, y, x + width_box, y + height_box)
                        
                        # Calculate intersection with monitor area
                        x_intersection = max(self.monitor_area[0], person_box[0])
                        y_intersection = max(self.monitor_area[1], person_box[1])
                        w_intersection = min(self.monitor_area[2], person_box[2]) - x_intersection
                        h_intersection = min(self.monitor_area[3], person_box[3]) - y_intersection
                        
                        is_in_desk_area = False
                        if w_intersection > 0 and h_intersection > 0:
                            intersection_area = w_intersection * h_intersection
                            person_area = width_box * height_box
                            overlap_ratio = intersection_area / person_area
                            
                            if overlap_ratio > 0.3:  # If more than 30% of person is in desk area
                                is_in_desk_area = True
                        
                        location = "Inside Desk" if is_in_desk_area else "Outside Desk"
                        
                        # Track status changes and log
                        person_key = name
                        if person_key not in self.person_status:
                            self.person_status[person_key] = location
                            self.log_person_event(name, "Detected", location, EmployeeID, email)
                        elif self.person_status[person_key] != location:
                            event = "Entered Desk" if location == "Inside Desk" else "Left Desk"
                            self.log_person_event(name, event, location, EmployeeID, email)
                            self.person_status[person_key] = location
                        
                        # Log position periodically
                        if frame_count % log_interval == 0:
                            self.log_person_position(name, location, confidences[i], cx, cy, EmployeeID)
                        
                        # Update employee/visitor tracking
                        if is_in_desk_area:
                            people_in_desk.append(name)
                            if name != "People":  # Known employee
                                employee_detected = True
                                self.last_employee_present = time.time()
                                if self.employee_absence_start is not None:
                                    self.employee_absence_start = None
                            color = (0, 255, 0)  # Green for people in desk area
                            label = f"{name} - Employee (Inside)"
                        else:
                            people_outside_desk.append(name)
                            # Anyone outside desk area is a visitor
                            visitor_detected = True
                            self.last_visitor_present = time.time()
                            if self.visitor_absence_start is not None:
                                self.visitor_absence_start = None
                            color = (0, 0, 255)  # Red for people outside
                            label = f"{name} - Visitor (Outside)"
                        
                        # Draw person boxes with different colors and thicker lines
                        cv2.rectangle(frame, (x, y), (x + width_box, y + height_box), color, 3)
                        
                        # Draw label background with larger size
                        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)[0]
                        cv2.rectangle(frame, (x, y - 40), (x + label_size[0] + 15, y), color, -1)
                        cv2.putText(frame, label, (x + 8, y - 12), 
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
                        
                        # Show confidence and EmployeeID with larger text
                        EmployeeID_display = EmployeeID if EmployeeID and EmployeeID != "N/A" else ("Employee" if is_in_desk_area else "Visitor")
                        EmployeeID_label = f"{EmployeeID_display} | Conf: {confidences[i]:.2f}"
                        cv2.putText(frame, EmployeeID_label, (x, y + height_box + 25), 
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            
            # Update absence tracking
            if not employee_detected and self.employee_absence_start is None:
                self.employee_absence_start = time.time()
            
            if not visitor_detected and self.visitor_absence_start is None:
                self.visitor_absence_start = time.time()
            
            # Draw monitoring area with thicker line
            cv2.rectangle(frame, 
                         (self.monitor_area[0], self.monitor_area[1]), 
                         (self.monitor_area[2], self.monitor_area[3]), 
                         (255, 255, 0), 4)  # Thicker yellow rectangle for monitored area
            
            # Add corner markers for desk area
            marker_size = 30
            # Top-left
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[1]), 
                    (self.monitor_area[0] + marker_size, self.monitor_area[1]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[1]), 
                    (self.monitor_area[0], self.monitor_area[1] + marker_size), (0, 255, 255), 6)
            # Top-right
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[1]), 
                    (self.monitor_area[2] - marker_size, self.monitor_area[1]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[1]), 
                    (self.monitor_area[2], self.monitor_area[1] + marker_size), (0, 255, 255), 6)
            # Bottom-left
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[3]), 
                    (self.monitor_area[0] + marker_size, self.monitor_area[3]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[3]), 
                    (self.monitor_area[0], self.monitor_area[3] - marker_size), (0, 255, 255), 6)
            # Bottom-right
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[3]), 
                    (self.monitor_area[2] - marker_size, self.monitor_area[3]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[3]), 
                    (self.monitor_area[2], self.monitor_area[3] - marker_size), (0, 255, 255), 6)
            
            # Display status on frame with larger text and enhanced status display
            cv2.rectangle(frame, (5, 5), (700, 260), (0, 0, 0), -1)
            
            # Employee Status
            employee_status_text = "Employee Status: PRESENT" if employee_detected else "Employee Status: ABSENT"
            employee_color = (0, 255, 0) if employee_detected else (0, 0, 255)
            cv2.putText(frame, employee_status_text, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 
                       0.7, employee_color, 2)
            
            # Employee absence duration
            if not employee_detected and self.employee_absence_start is not None:
                absence_duration = time.time() - self.employee_absence_start
                duration_text = f"Employee Absence: {absence_duration:.1f}s"
                cv2.putText(frame, duration_text, (15, 65), cv2.FONT_HERSHEY_SIMPLEX, 
                           0.6, (0, 0, 255), 2)
            
            # Visitor Status with count
            visitor_count = len(people_outside_desk)
            visitor_status_text = f"Visitor Status: {visitor_count} PRESENT" if visitor_detected else "Visitor Status: NONE"
            visitor_color = (0, 165, 255) if visitor_detected else (128, 128, 128)
            cv2.putText(frame, visitor_status_text, (15, 100), cv2.FONT_HERSHEY_SIMPLEX, 
                       0.7, visitor_color, 2)
            
            # Count display
            count_text = f"In Desk: {len(people_in_desk)} | Outside: {len(people_outside_desk)}"
            cv2.putText(frame, count_text, (15, 135), cv2.FONT_HERSHEY_SIMPLEX, 
                       0.7, (255, 255, 0), 2)
            
            # Show names inside with larger text
            if people_in_desk:
                names_in = ", ".join(people_in_desk[:3])  # Show max 3 names
                if len(people_in_desk) > 3:
                    names_in += f" +{len(people_in_desk)-3}"
                cv2.putText(frame, f"Inside: {names_in}", (15, 165), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            
            # Show names outside with larger text
            if people_outside_desk:
                names_out = ", ".join(people_outside_desk[:3])  # Show max 3 names
                if len(people_outside_desk) > 3:
                    names_out += f" +{len(people_outside_desk)-3}"
                cv2.putText(frame, f"Outside: {names_out}", (15, 195), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)
            
            # Add timestamp with larger text
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {timestamp}", (15, 225), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.putText(frame, "Mode: SIMPLE | Press 'q' to quit", (15, 250), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 200, 0), 2)
            
            cv2.imshow('Employee Desk Tracking - Simple Mode', frame)
            
            frame_count += 1
            
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved to: {self.log_file} and {self.detailed_log_file}")
    
    def _run_full_mode(self, cap):
        """Run with full face recognition and detailed tracking"""
        last_cleanup = time.time()
        frame_count = 0
        log_interval = 30  # Log every 30 frames
        
        # Create named window with resizable option
        cv2.namedWindow('Employee Desk Tracking - Full Mode', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking - Full Mode', self.display_width, self.display_height)
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            h, w = frame.shape[:2]

            # Create desk mask if needed
            if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                self.desk_mask = np.zeros((h, w), np.uint8)
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(self.desk_mask, self.DESK_ROI_PT1, self.DESK_ROI_PT2, 255, -1)

            # YOLO person detection
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            outputs = self.net.forward(self.output_layers)

            boxes, confidences = [], []
            for output in outputs:
                for detection in output:
                    scores = detection[5:]
                    class_id = np.argmax(scores)
                    conf = scores[class_id]
                    if conf > 0.5 and self.classes[class_id] == 'person':
                        cx = int(detection[0] * w)
                        cy = int(detection[1] * h)
                        bw = int(detection[2] * w)
                        bh = int(detection[3] * h)
                        x1 = int(cx - bw / 2)
                        y1 = int(cy - bh / 2)
                        boxes.append([x1, y1, bw, bh])
                        confidences.append(float(conf))

            indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, self.nms_threshold)

            current_in_desk = {}  # name -> (EmployeeID, email)
            current_outside = {}  # name -> (EmployeeID, email)
            employee_detected = False
            visitor_detected = False
            detections = indices.flatten() if len(indices) > 0 else []

            for i in detections:
                x1, y1, bw, bh = boxes[i]
                x2, y2 = x1 + bw, y1 + bh
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

                name, EmployeeID, email = self.get_name_from_centroid(cx, cy, frame)

                if name not in self.trackers:
                    self.trackers[name] = {
                        'centroids': [], 
                        'last_seen': 0,
                        'EmployeeID': EmployeeID,
                        'email': email
                    }
                
                self.trackers[name]['bbox'] = (x1, y1, x2, y2)
                self.trackers[name]['centroids'].append((cx, cy))
                self.trackers[name]['last_seen'] = time.time()
                self.trackers[name]['EmployeeID'] = EmployeeID
                self.trackers[name]['email'] = email
                
                if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                    self.trackers[name]['centroids'].pop(0)

                # Calculate overlap with desk area
                overlap_ratio = 0
                if x2 > x1 and y2 > y1:
                    y_slice = slice(max(0, y1), min(h, y2))
                    x_slice = slice(max(0, x1), min(w, x2))
                    mask_roi = self.desk_mask[y_slice, x_slice]
                    if mask_roi.size > 0:
                        overlap_ratio = np.sum(mask_roi > 0) / mask_roi.size
                
                location = 'Inside Desk' if overlap_ratio > 0.4 else 'Outside Desk'
                
                if 'Inside' in location:
                    current_in_desk[name] = (EmployeeID, email)
                    if name != "People":
                        employee_detected = True
                        self.last_employee_present = time.time()
                        if self.employee_absence_start is not None:
                            self.employee_absence_start = None
                else:
                    current_outside[name] = (EmployeeID, email)
                    # Anyone outside desk area is a visitor
                    visitor_detected = True
                    self.last_visitor_present = time.time()
                    if self.visitor_absence_start is not None:
                        self.visitor_absence_start = None
                
                # Track status changes
                if name not in self.person_status:
                    self.person_status[name] = location
                    self.log_person_event(name, "Detected", location, EmployeeID, email)
                elif self.person_status[name] != location:
                    event = "Entered Desk" if location == "Inside Desk" else "Left Desk"
                    self.log_person_event(name, event, location, EmployeeID, email)
                    self.person_status[name] = location
                
                # Log position periodically
                if frame_count % log_interval == 0:
                    self.log_person_position(name, location, confidences[i], cx, cy, EmployeeID)

                # Draw bounding box and labels with thicker lines
                color = (0, 255, 0) if 'Inside' in location else (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                
                # Label with name and location with larger text
                role = "Employee" if 'Inside' in location else "Visitor"
                label = f"{name} - {role} ({location.split()[0]})"
                label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)[0]
                cv2.rectangle(frame, (x1, y1 - 45), (x1 + label_size[0] + 15, y1), color, -1)
                cv2.putText(frame, label, (x1 + 8, y1 - 15), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
                
                # Show EmployeeID and confidence with larger text
                EmployeeID_display = EmployeeID if EmployeeID and EmployeeID != "N/A" else ("Employee" if 'Inside' in location else "Visitor")
                info_label = f"{EmployeeID_display} | Conf: {confidences[i]:.2f}"
                cv2.putText(frame, info_label, (x1, y2 + 25), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # Update absence tracking
            if not employee_detected and self.employee_absence_start is None:
                self.employee_absence_start = time.time()
            
            if not visitor_detected and self.visitor_absence_start is None:
                self.visitor_absence_start = time.time()

            # Cleanup disappeared trackers
            if time.time() - last_cleanup > 1.0:
                to_remove = [name for name, data in self.trackers.items() 
                            if time.time() - data['last_seen'] > self.DISAPPEARANCE_TIME]
                for name in to_remove:
                    if name in self.person_status:
                        EmployeeID = self.trackers[name].get('EmployeeID', '')
                        email = self.trackers[name].get('email', '')
                        self.log_person_event(name, "Disappeared", "Unknown", EmployeeID, email)
                        del self.person_status[name]
                    del self.trackers[name]
                last_cleanup = time.time()

            # Draw desk ROI with enhanced visualization
            if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                # Draw main rectangle
                cv2.rectangle(frame, self.DESK_ROI_PT1, self.DESK_ROI_PT2, (255, 255, 0), 4)
                
                # Add corner markers
                marker_size = 40
                # Top-left
                cv2.line(frame, self.DESK_ROI_PT1, 
                        (self.DESK_ROI_PT1[0] + marker_size, self.DESK_ROI_PT1[1]), (0, 255, 255), 6)
                cv2.line(frame, self.DESK_ROI_PT1, 
                        (self.DESK_ROI_PT1[0], self.DESK_ROI_PT1[1] + marker_size), (0, 255, 255), 6)
                # Top-right
                cv2.line(frame, (self.DESK_ROI_PT2[0], self.DESK_ROI_PT1[1]), 
                        (self.DESK_ROI_PT2[0] - marker_size, self.DESK_ROI_PT1[1]), (0, 255, 255), 6)
                cv2.line(frame, (self.DESK_ROI_PT2[0], self.DESK_ROI_PT1[1]), 
                        (self.DESK_ROI_PT2[0], self.DESK_ROI_PT1[1] + marker_size), (0, 255, 255), 6)
                # Bottom-left
                cv2.line(frame, (self.DESK_ROI_PT1[0], self.DESK_ROI_PT2[1]), 
                        (self.DESK_ROI_PT1[0] + marker_size, self.DESK_ROI_PT2[1]), (0, 255, 255), 6)
                cv2.line(frame, (self.DESK_ROI_PT1[0], self.DESK_ROI_PT2[1]), 
                        (self.DESK_ROI_PT1[0], self.DESK_ROI_PT2[1] - marker_size), (0, 255, 255), 6)
                # Bottom-right
                cv2.line(frame, self.DESK_ROI_PT2, 
                        (self.DESK_ROI_PT2[0] - marker_size, self.DESK_ROI_PT2[1]), (0, 255, 255), 6)
                cv2.line(frame, self.DESK_ROI_PT2, 
                        (self.DESK_ROI_PT2[0], self.DESK_ROI_PT2[1] - marker_size), (0, 255, 255), 6)
                
                # Status overlay with larger text and enhanced display
                cv2.rectangle(frame, (5, 5), (750, 280), (0, 0, 0), -1)
                
                # Employee Status
                employee_status_text = "Employee Status: PRESENT" if employee_detected else "Employee Status: ABSENT"
                employee_color = (0, 255, 0) if employee_detected else (0, 0, 255)
                cv2.putText(frame, employee_status_text, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 
                           0.7, employee_color, 2)
                
                # Employee absence duration
                if not employee_detected and self.employee_absence_start is not None:
                    absence_duration = time.time() - self.employee_absence_start
                    duration_text = f"Employee Absence: {absence_duration:.1f}s"
                    cv2.putText(frame, duration_text, (15, 65), cv2.FONT_HERSHEY_SIMPLEX, 
                               0.6, (0, 0, 255), 2)
                
                # Visitor Status with count
                visitor_count = len(current_outside)
                visitor_status_text = f"Visitor Status: {visitor_count} PRESENT" if visitor_detected else "Visitor Status: NONE"
                visitor_color = (0, 165, 255) if visitor_detected else (128, 128, 128)
                cv2.putText(frame, visitor_status_text, (15, 100), cv2.FONT_HERSHEY_SIMPLEX, 
                           0.7, visitor_color, 2)
                
                # Count display
                cv2.putText(frame, f'Inside: {len(current_in_desk)} | Outside: {len(current_outside)} | Total: {len(self.trackers)}', 
                           (15, 135), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
                
                # Show people inside with larger text
                if current_in_desk:
                    names_in = ", ".join(list(current_in_desk.keys())[:3])
                    if len(current_in_desk) > 3:
                        names_in += f" +{len(current_in_desk)-3}"
                    cv2.putText(frame, f"Inside Desk: {names_in}", (15, 165), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                
                # Show people outside with larger text
                if current_outside:
                    names_out = ", ".join(list(current_outside.keys())[:3])
                    if len(current_outside) > 3:
                        names_out += f" +{len(current_outside)-3}"
                    cv2.putText(frame, f"Outside Desk: {names_out}", (15, 195), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)
                
                # Timestamp with larger text
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                cv2.putText(frame, f"Time: {timestamp}", (15, 225), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(frame, "Mode: FULL | Press 'q' to quit", (15, 250), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 200, 0), 2)

            cv2.imshow('Employee Desk Tracking - Full Mode (Face Recognition)', frame)
            
            frame_count += 1
            
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved to: {self.log_file} and {self.detailed_log_file}")


if __name__ == "__main__":
    import sys
    
    tracker = DeskTracker()
    
    # Check if user wants simple mode or full mode
    mode = "full"  # default
    debug = False
    
    for arg in sys.argv[1:]:
        if arg.lower() in ['simple', 's']:
            mode = "simple"
        elif arg.lower() in ['full', 'f']:
            mode = "full"
        elif arg.lower() in ['debug', 'd', '--debug']:
            debug = True
    
    if debug:
        tracker.debug_mode = True
        print("DEBUG MODE ENABLED - Face search regions will be visualized")
    
    use_simple = (mode == "simple")
    
    print("\n" + "="*60)
    print(f"Starting Desk Tracker in {mode.upper()} mode")
    if debug:
        print("DEBUG MODE: Face search regions visible")
    print("="*60)
    print("\nModes:")
    print("  SIMPLE: Face recognition + presence detection")
    print("  FULL:   Face recognition + detailed tracking + history")
    print("\nFeatures:")
    print("  ✓ Automatic desk area detection")
    print("  ✓ Enhanced face recognition with multiple search regions")
    print("  ✓ Inside/Outside desk tracking")
    print("  ✓ Employee/Visitor status tracking")
    print("  ✓ Absence duration monitoring")
    print("  ✓ Event logging (Entry/Exit)")
    print("  ✓ Detailed position logging")
    print("  ✓ EmployeeID and email tracking")
    print("\nUsage:")
    print("  python desk_tracking_fixed.py simple         # Simple mode")
    print("  python desk_tracking_fixed.py full           # Full mode")
    print("  python desk_tracking_fixed.py debug          # Enable debug visualization")
    print("  python desk_tracking_fixed.py simple debug   # Simple mode + debug")
    print("  python desk_tracking_fixed.py                # Full mode (default)")
    print("\nDebug Mode:")
    print("  Shows colored rectangles where the system searches for faces")
    print("  Useful for troubleshooting face recognition issues")
    print("\nPress 'q' to quit")
    print("="*60 + "\n")
    
    tracker.run(use_simple_mode=use_simple)

In [ ]:
import cv2
import numpy as np
import time
from datetime import datetime
import csv
import os
from collections import defaultdict
import pickle
from sklearn.neighbors import KNeighborsClassifier

class DeskTracker:
    def __init__(self):
        # Load YOLOv4-tiny model 
        cfg_path = 'Yolo_model/yolov4-tiny.cfg'
        names_path = 'Yolo_model/coco.names'
        weights_path = 'Yolo_model/yolov4-tiny.weights'

        # Load names
        with open(names_path, 'r') as f:
            self.classes = [line.strip() for line in f.readlines()]

        # Load model
        self.net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
        
        # Get output layer names
        layer_names = self.net.getLayerNames()
        self.output_layers = [layer_names[i - 1] for i in self.net.getUnconnectedOutLayers()]
        
        # Detection parameters
        self.confidence_threshold = 0.3
        self.nms_threshold = 0.4
        
        # Face recognition parameters
        self.face_recognition_threshold = 35  # Lowered for better recognition
        self.face_min_size = (30, 30)  # Minimum face size to detect
        
        # Load face recognition data
        try:
            with open('data/Departments.pkl', 'rb') as f:
                self.DEPARTMENTS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/EmployeeIDs.pkl', 'rb') as f:
                self.EmployeeIDs = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f)
                self.FACES = self.FACES / 255.0
                print(f"FACES shape: {self.FACES.shape}")
            
            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.EmployeeIDs)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []
            self.EmployeeIDs = []
            self.DEPARTMENTS = []
            self.EMAILS = []
            self.face_recognition_enabled = False

        # Load face cascade
        self.face_cascade = cv2.CascadeClassifier('data/haarcascade_frontalface_default.xml')
        
        # Desk ROI - will be detected automatically
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask = None
        
        # Tracking
        self.trackers = {}
        self.person_status = {}  # Track last known status of each person
        self.MAX_CENTROIDS = 10
        self.DISAPPEARANCE_TIME = 2.0
        
        # Status tracking for employees and visitors
        self.employee_absence_start = None
        self.visitor_absence_start = None
        self.last_employee_present = None
        self.last_visitor_present = None
        
        # Logging
        timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_file = f'desk_log_{timestamp_str}.csv'
        self.detailed_log_file = f'detailed_log_{timestamp_str}.csv'
        
        # Create main log file
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Name', 'Event', 'Location', 'EmployeeID', 'Email'])
        
        # Create detailed log file
        if not os.path.exists(self.detailed_log_file):
            with open(self.detailed_log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Name', 'Location', 'Confidence', 'X', 'Y', 'EmployeeID'])
        
        # Monitor area (will be set from desk detection)
        self.monitor_area = None
        self.absence_start_time = None
        self.source_type = "camera"
        
        # Frame size settings for better visualization
        self.display_width = 1280  # Increased from default
        self.display_height = 720  # Increased from default
        
        # Debug mode - set to True to visualize face detection regions
        # This will show the search areas where the system looks for faces
        self.debug_mode = False  # Set to True to see face search regions

    def log_event(self, message):
        """Log events to console"""
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def log_person_event(self, name, event, location, EmployeeID="", email=""):
        """Log person entry/exit events"""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, name, event, location, EmployeeID, email])
        self.log_event(f"{name} {event} {location}")

    def log_person_position(self, name, location, confidence, x, y, EmployeeID=""):
        """Log detailed person position"""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.detailed_log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, name, location, f"{confidence:.2f}", x, y, EmployeeID])

    def detect_desk_area(self, cap):
        """Detect the desk area using object detection"""
        self.log_event("Detecting desk area...")
        
        # Take multiple frames to improve detection reliability
        desk_candidates = []
        frame_count = 0
        max_frames = 5
        
        frame = None  # Initialize frame variable
        
        # Process several frames to get a better desk detection
        while frame_count < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
                
            height, width = frame.shape[:2]
            
            # Create a blob from the frame and perform a forward pass
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                break
            
            # Variables to store detection info
            desks = []
            
            # Process detections to find tables/desks/chairs
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    
                    # COCO class IDs: 56=chair, 60=dining table, 63=laptop, 64=mouse, 65=keyboard
                    desk_related_classes = [56, 60, 63, 64, 65]
                    
                    if confidence > self.confidence_threshold and class_id in desk_related_classes:
                        center_x = int(obj_detection[0] * width)
                        center_y = int(obj_detection[1] * height)
                        w = int(obj_detection[2] * width)
                        h = int(obj_detection[3] * height)
                        
                        # Calculate top-left corner coordinates
                        x = int(center_x - w / 2)
                        y = int(center_y - h / 2)
                        
                        desks.append([x, y, w, h, confidence, class_id])
            
            # If we found desk-related objects, add them to candidates
            if desks:
                desk_candidates.extend(desks)
            
            frame_count += 1
        
        # If we don't have enough desk candidates, use a default area
        if len(desk_candidates) < 2:
            self.log_event("Not enough desk objects detected. Using default desk area.")
            if frame is not None:
                height, width = frame.shape[:2]
            else:
                height, width = 480, 640  # Default dimensions
            
            # Default to middle-bottom portion of the frame
            min_x = int(width * 0.2)
            min_y = int(height * 0.4)
            max_x = int(width * 0.8)
            max_y = int(height * 0.9)
        else:
            # Group desk objects to determine the desk area
            x_coords = [d[0] for d in desk_candidates]
            y_coords = [d[1] for d in desk_candidates]
            max_x_coords = [d[0] + d[2] for d in desk_candidates]
            max_y_coords = [d[1] + d[3] for d in desk_candidates]
            
            # Create a bounding box that covers all desk-related objects with padding
            min_x = max(0, int(min(x_coords) - 0.1 * width))
            min_y = max(0, int(min(y_coords) - 0.1 * height))
            max_x = min(width, int(max(max_x_coords) + 0.1 * width))
            max_y = min(height, int(max(max_y_coords) + 0.1 * height))
            
            self.log_event(f"Desk area detected: ({min_x}, {min_y}) to ({max_x}, {max_y})")
        
        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        
        # Set monitor area for _process_frame method
        self.monitor_area = (min_x, min_y, max_x, max_y)
        
        return (min_x, min_y, max_x, max_y)

    def get_name_from_face(self, face_region):
        """Extract name from face region using face recognition"""
        if not self.face_recognition_enabled or face_region.size == 0:
            return "People", "N/A", ""
        
        # Check if region is too small
        if face_region.shape[0] < 30 or face_region.shape[1] < 30:
            return "People", "N/A", ""
        
        gray_region = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        
        # Try multiple scale factors and parameters for better detection
        faces = []
        detection_params = [
            (1.1, 5, (20, 20)),  # More sensitive
            (1.05, 4, (25, 25)), # Very sensitive
            (1.2, 5, (30, 30)),  # Original
            (1.3, 4, (35, 35)),  # Less sensitive but more accurate
            (1.15, 3, (25, 25)), # Medium sensitivity
        ]
        
        for scale, neighbors, min_size in detection_params:
            detected = self.face_cascade.detectMultiScale(
                gray_region, scale, neighbors, minSize=min_size
            )
            if len(detected) > 0:
                faces = detected
                break

        # Default if no face / no match
        Name = "People"
        EmployeeID = "N/A"
        Email = ""

        for (x, y, w_face, h_face) in faces:
            crop_img = face_region[y:y+h_face, x:x+w_face, :]
            
            if crop_img.size == 0:
                continue
          
            # Use 75x100 => 7500 features to match FACES shape
            try:
                resized_img = cv2.resize(crop_img, (75, 100))
                gray = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
                resized_flat = gray.reshape(1, -1) / 255.0

                # Check if feature dimension matches
                if resized_flat.shape[1] != self.FACES.shape[1]:
                    continue

                distances, indices = self.knn.kneighbors(resized_flat)
                
                # Use configurable threshold
                if distances[0][0] <= self.face_recognition_threshold:
                    output = self.knn.predict(resized_flat)
                    emp_id = output[0]
                    
                    if emp_id in self.EmployeeIDs:
                        index = self.EmployeeIDs.index(emp_id)
                        Name = self.NAMES[index]
                        EmployeeID = self.EmployeeIDs[index] if index < len(self.EmployeeIDs) else "N/A"
                        Email = self.EMAILS[index] if index < len(self.EMAILS) else ""
                        break  # Use first detected face
            except Exception as e:
                # If any error occurs, continue to next face
                continue

        return Name, EmployeeID, Email

    def get_name_from_centroid(self, cx, cy, frame):
        """Extract name from face recognition around a centroid"""
        h, w = frame.shape[:2]
        
        # Try multiple search regions with different sizes and positions
        # This helps catch faces in different positions relative to the body centroid
        search_regions = [
            # Main region - centered above centroid (where head usually is)
            (max(0, cx - 180), max(0, cy - 280), min(w, cx + 180), min(h, cy + 50)),
            # Wider region
            (max(0, cx - 220), max(0, cy - 320), min(w, cx + 220), min(h, cy + 80)),
            # Higher up (for people far from camera)
            (max(0, cx - 150), max(0, cy - 350), min(w, cx + 150), min(h, cy)),
            # Lower region (for people close to camera)
            (max(0, cx - 200), max(0, cy - 200), min(w, cx + 200), min(h, cy + 150)),
        ]
        
        # Try each search region until we find a recognized face
        for idx, (rx1, ry1, rx2, ry2) in enumerate(search_regions):
            # Draw face search region in debug mode
            if self.debug_mode:
                color = [(255, 0, 255), (0, 255, 255), (255, 255, 0), (255, 128, 0)][idx]
                cv2.rectangle(frame, (rx1, ry1), (rx2, ry2), color, 2)
                cv2.putText(frame, f"Search {idx+1}", (rx1, ry1 - 5), 
                           cv2.FONT_HERSHEY_COMPLEX, 0.4, color, 1)
            
            face_region = frame[ry1:ry2, rx1:rx2]
            name, EmployeeID, email = self.get_name_from_face(face_region)
            
            # If we found a recognized person (not "People"), return immediately
            if name != "People":
                return name, EmployeeID, email
        
        # If no recognized face found in any region, return "People"
        return "People", "N/A", ""

    def run(self, use_simple_mode=False):
        """Main tracking loop
        
        Args:
            use_simple_mode: If True, uses simpler detection with face recognition.
                           If False, uses full detailed tracking.
        """
        cap = cv2.VideoCapture(0)
        
        if not cap.isOpened():
            self.log_event("Error: Cannot open camera")
            return
        
        # Set camera resolution for better quality
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
        cap.set(cv2.CAP_PROP_FPS, 30)
        
        # Get actual resolution
        actual_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        actual_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.log_event(f"Camera resolution: {actual_width}x{actual_height}")
        
        # Detect desk area from initial frames
        self.detect_desk_area(cap)
        
        # Reset camera to beginning
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        
        if use_simple_mode:
            self.log_event("Running in SIMPLE mode (face recognition + presence)")
            self._run_simple_mode(cap)
        else:
            self.log_event("Running in FULL mode (face recognition + detailed tracking)")
            self._run_full_mode(cap)
    
    def _run_simple_mode(self, cap):
        """Run with simple presence detection and face recognition"""
        frame_count = 0
        log_interval = 30  # Log every 30 frames
        
        # Create named window with resizable option
        cv2.namedWindow('Employee Desk Tracking - Simple Mode', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking - Simple Mode', self.display_width, self.display_height)
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            h, w = frame.shape[:2]
            
            # YOLO person detection
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                continue
            
            # Variables to store detection info
            boxes = []
            confidences = []
            
            # Process detections to find people
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    
                    # Check if the detected object is a person
                    if confidence > self.confidence_threshold and class_id == 0:
                        center_x = int(obj_detection[0] * w)
                        center_y = int(obj_detection[1] * h)
                        width_box = int(obj_detection[2] * w)
                        height_box = int(obj_detection[3] * h)
                        
                        # Calculate top-left corner coordinates
                        x = int(center_x - width_box / 2)
                        y = int(center_y - height_box / 2)
                        
                        boxes.append([x, y, width_box, height_box])
                        confidences.append(float(confidence))
            
            # Apply non-maximum suppression
            people_in_desk = []
            people_outside_desk = []
            employee_detected = False
            visitor_detected = False
            
            if len(boxes) > 0:
                indices = cv2.dnn.NMSBoxes(boxes, confidences, self.confidence_threshold, 0.4)
                
                if len(indices) > 0:
                    for i in indices.flatten():
                        x, y, width_box, height_box = boxes[i]
                        cx, cy = x + width_box // 2, y + height_box // 2
                        
                        # Get name from face recognition
                        name, EmployeeID, email = self.get_name_from_centroid(cx, cy, frame)
                        
                        # Calculate person box
                        person_box = (x, y, x + width_box, y + height_box)
                        
                        # Calculate intersection with monitor area
                        x_intersection = max(self.monitor_area[0], person_box[0])
                        y_intersection = max(self.monitor_area[1], person_box[1])
                        w_intersection = min(self.monitor_area[2], person_box[2]) - x_intersection
                        h_intersection = min(self.monitor_area[3], person_box[3]) - y_intersection
                        
                        is_in_desk_area = False
                        if w_intersection > 0 and h_intersection > 0:
                            intersection_area = w_intersection * h_intersection
                            person_area = width_box * height_box
                            overlap_ratio = intersection_area / person_area
                            
                            if overlap_ratio > 0.3:  # If more than 30% of person is in desk area
                                is_in_desk_area = True
                        
                        location = "Inside Desk" if is_in_desk_area else "Outside Desk"
                        
                        # Track status changes and log
                        person_key = name
                        if person_key not in self.person_status:
                            self.person_status[person_key] = location
                            self.log_person_event(name, "Detected", location, EmployeeID, email)
                        elif self.person_status[person_key] != location:
                            event = "Entered Desk" if location == "Inside Desk" else "Left Desk"
                            self.log_person_event(name, event, location, EmployeeID, email)
                            self.person_status[person_key] = location
                        
                        # Log position periodically
                        if frame_count % log_interval == 0:
                            self.log_person_position(name, location, confidences[i], cx, cy, EmployeeID)
                        
                        # Update employee/visitor tracking
                        if is_in_desk_area:
                            people_in_desk.append(name)
                            if name != "People":  # Known employee
                                employee_detected = True
                                self.last_employee_present = time.time()
                                if self.employee_absence_start is not None:
                                    self.employee_absence_start = None
                            color = (0, 255, 0)  # Green for people in desk area
                            label = f"{name} - Employee (Inside)"
                        else:
                            people_outside_desk.append(name)
                            # Anyone outside desk area is a visitor
                            visitor_detected = True
                            self.last_visitor_present = time.time()
                            if self.visitor_absence_start is not None:
                                self.visitor_absence_start = None
                            color = (0, 0, 255)  # Red for people outside
                            label = f"{name} - Visitor (Outside)"
                        
                        # Draw person boxes with different colors and thicker lines
                        cv2.rectangle(frame, (x, y), (x + width_box, y + height_box), color, 3)
                        
                        # Draw label background with larger size
                        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_COMPLEX, 0.8, 2)[0]
                        cv2.rectangle(frame, (x, y - 40), (x + label_size[0] + 15, y), color, -1)
                        cv2.putText(frame, label, (x + 8, y - 12), 
                                  cv2.FONT_HERSHEY_COMPLEX, 0.8, (255, 255, 255), 2)
                        
                        # Show confidence and EmployeeID with larger text
                        EmployeeID_display = EmployeeID if EmployeeID and EmployeeID != "N/A" else ("Employee" if is_in_desk_area else "Visitor")
                        EmployeeID_label = f"{EmployeeID_display} | Conf: {confidences[i]:.2f}"
                        cv2.putText(frame, EmployeeID_label, (x, y + height_box + 25), 
                                  cv2.FONT_HERSHEY_COMPLEX, 0.6, color, 2)
            
            # Update absence tracking
            if not employee_detected and self.employee_absence_start is None:
                self.employee_absence_start = time.time()
            
            if not visitor_detected and self.visitor_absence_start is None:
                self.visitor_absence_start = time.time()
            
            # Draw monitoring area with thicker line
            cv2.rectangle(frame, 
                         (self.monitor_area[0], self.monitor_area[1]), 
                         (self.monitor_area[2], self.monitor_area[3]), 
                         (255, 255, 0), 4)  # Thicker yellow rectangle for monitored area
            
            # Add corner markers for desk area
            marker_size = 30
            # Top-left
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[1]), 
                    (self.monitor_area[0] + marker_size, self.monitor_area[1]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[1]), 
                    (self.monitor_area[0], self.monitor_area[1] + marker_size), (0, 255, 255), 6)
            # Top-right
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[1]), 
                    (self.monitor_area[2] - marker_size, self.monitor_area[1]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[1]), 
                    (self.monitor_area[2], self.monitor_area[1] + marker_size), (0, 255, 255), 6)
            # Bottom-left
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[3]), 
                    (self.monitor_area[0] + marker_size, self.monitor_area[3]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[0], self.monitor_area[3]), 
                    (self.monitor_area[0], self.monitor_area[3] - marker_size), (0, 255, 255), 6)
            # Bottom-right
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[3]), 
                    (self.monitor_area[2] - marker_size, self.monitor_area[3]), (0, 255, 255), 6)
            cv2.line(frame, (self.monitor_area[2], self.monitor_area[3]), 
                    (self.monitor_area[2], self.monitor_area[3] - marker_size), (0, 255, 255), 6)
            
            # Display status on frame with larger text and enhanced status display
            cv2.rectangle(frame, (5, 5), (700, 260), (0, 0, 0), -1)
            
            # Employee Status
            employee_status_text = "Employee Status: PRESENT" if employee_detected else "Employee Status: ABSENT"
            employee_color = (0, 255, 0) if employee_detected else (0, 0, 255)
            cv2.putText(frame, employee_status_text, (15, 35), cv2.FONT_HERSHEY_COMPLEX, 
                       0.7, employee_color, 2)
            
            # Employee absence duration
            if not employee_detected and self.employee_absence_start is not None:
                absence_duration = time.time() - self.employee_absence_start
                duration_text = f"Employee Absence: {absence_duration:.1f}s"
                cv2.putText(frame, duration_text, (15, 65), cv2.FONT_HERSHEY_COMPLEX, 
                           0.6, (0, 0, 255), 2)
            
            # Visitor Status with count
            visitor_count = len(people_outside_desk)
            visitor_status_text = f"Visitor Status: {visitor_count} PRESENT" if visitor_detected else "Visitor Status: NONE"
            visitor_color = (0, 165, 255) if visitor_detected else (128, 128, 128)
            cv2.putText(frame, visitor_status_text, (15, 100), cv2.FONT_HERSHEY_COMPLEX, 
                       0.7, visitor_color, 2)
            
            # Count display
            count_text = f"In Desk: {len(people_in_desk)} | Outside: {len(people_outside_desk)}"
            cv2.putText(frame, count_text, (15, 135), cv2.FONT_HERSHEY_COMPLEX, 
                       0.7, (255, 255, 0), 2)
            
            # Show names inside with larger text
            if people_in_desk:
                names_in = ", ".join(people_in_desk[:3])  # Show max 3 names
                if len(people_in_desk) > 3:
                    names_in += f" +{len(people_in_desk)-3}"
                cv2.putText(frame, f"Inside: {names_in}", (15, 165), 
                           cv2.FONT_HERSHEY_COMPLEX, 0.6, (0, 255, 0), 2)
            
            # Show names outside with larger text
            if people_outside_desk:
                names_out = ", ".join(people_outside_desk[:3])  # Show max 3 names
                if len(people_outside_desk) > 3:
                    names_out += f" +{len(people_outside_desk)-3}"
                cv2.putText(frame, f"Outside: {names_out}", (15, 195), 
                           cv2.FONT_HERSHEY_COMPLEX, 0.6, (0, 165, 255), 2)
            
            # Add timestamp with larger text
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {timestamp}", (15, 225), 
                       cv2.FONT_HERSHEY_COMPLEX, 0.6, (255, 255, 255), 2)
            cv2.putText(frame, "Mode: SIMPLE | Press 'q' to quit", (15, 250), 
                       cv2.FONT_HERSHEY_COMPLEX, 0.6, (255, 200, 0), 2)
            
            cv2.imshow('Employee Desk Tracking - Simple Mode', frame)
            
            frame_count += 1
            
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved to: {self.log_file} and {self.detailed_log_file}")
    
    def _run_full_mode(self, cap):
        """Run with full face recognition and detailed tracking"""
        last_cleanup = time.time()
        frame_count = 0
        log_interval = 30  # Log every 30 frames
        
        # Create named window with resizable option
        cv2.namedWindow('Employee Desk Tracking - Full Mode', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking - Full Mode', self.display_width, self.display_height)
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            h, w = frame.shape[:2]

            # Create desk mask if needed
            if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                self.desk_mask = np.zeros((h, w), np.uint8)
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(self.desk_mask, self.DESK_ROI_PT1, self.DESK_ROI_PT2, 255, -1)

            # YOLO person detection
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            outputs = self.net.forward(self.output_layers)

            boxes, confidences = [], []
            for output in outputs:
                for detection in output:
                    scores = detection[5:]
                    class_id = np.argmax(scores)
                    conf = scores[class_id]
                    if conf > 0.5 and self.classes[class_id] == 'person':
                        cx = int(detection[0] * w)
                        cy = int(detection[1] * h)
                        bw = int(detection[2] * w)
                        bh = int(detection[3] * h)
                        x1 = int(cx - bw / 2)
                        y1 = int(cy - bh / 2)
                        boxes.append([x1, y1, bw, bh])
                        confidences.append(float(conf))

            indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, self.nms_threshold)

            current_in_desk = {}  # name -> (EmployeeID, email)
            current_outside = {}  # name -> (EmployeeID, email)
            employee_detected = False
            visitor_detected = False
            detections = indices.flatten() if len(indices) > 0 else []

            for i in detections:
                x1, y1, bw, bh = boxes[i]
                x2, y2 = x1 + bw, y1 + bh
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

                name, EmployeeID, email = self.get_name_from_centroid(cx, cy, frame)

                if name not in self.trackers:
                    self.trackers[name] = {
                        'centroids': [], 
                        'last_seen': 0,
                        'EmployeeID': EmployeeID,
                        'email': email
                    }
                
                self.trackers[name]['bbox'] = (x1, y1, x2, y2)
                self.trackers[name]['centroids'].append((cx, cy))
                self.trackers[name]['last_seen'] = time.time()
                self.trackers[name]['EmployeeID'] = EmployeeID
                self.trackers[name]['email'] = email
                
                if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                    self.trackers[name]['centroids'].pop(0)

                # Calculate overlap with desk area
                overlap_ratio = 0
                if x2 > x1 and y2 > y1:
                    y_slice = slice(max(0, y1), min(h, y2))
                    x_slice = slice(max(0, x1), min(w, x2))
                    mask_roi = self.desk_mask[y_slice, x_slice]
                    if mask_roi.size > 0:
                        overlap_ratio = np.sum(mask_roi > 0) / mask_roi.size
                
                location = 'Inside Desk' if overlap_ratio > 0.4 else 'Outside Desk'
                
                if 'Inside' in location:
                    current_in_desk[name] = (EmployeeID, email)
                    if name != "People":
                        employee_detected = True
                        self.last_employee_present = time.time()
                        if self.employee_absence_start is not None:
                            self.employee_absence_start = None
                else:
                    current_outside[name] = (EmployeeID, email)
                    # Anyone outside desk area is a visitor
                    visitor_detected = True
                    self.last_visitor_present = time.time()
                    if self.visitor_absence_start is not None:
                        self.visitor_absence_start = None
                
                # Track status changes
                if name not in self.person_status:
                    self.person_status[name] = location
                    self.log_person_event(name, "Detected", location, EmployeeID, email)
                elif self.person_status[name] != location:
                    event = "Entered Desk" if location == "Inside Desk" else "Left Desk"
                    self.log_person_event(name, event, location, EmployeeID, email)
                    self.person_status[name] = location
                
                # Log position periodically
                if frame_count % log_interval == 0:
                    self.log_person_position(name, location, confidences[i], cx, cy, EmployeeID)

                # Draw bounding box and labels with thicker lines
                color = (0, 255, 0) if 'Inside' in location else (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                
                # Label with name and location with larger text
                role = "Employee" if 'Inside' in location else "Visitor"
                label = f"{name} - {role} ({location.split()[0]})"
                label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_COMPLEX, 0.8, 2)[0]
                cv2.rectangle(frame, (x1, y1 - 45), (x1 + label_size[0] + 15, y1), color, -1)
                cv2.putText(frame, label, (x1 + 8, y1 - 15), 
                           cv2.FONT_HERSHEY_COMPLEX, 0.8, (255, 255, 255), 2)
                
                # Show EmployeeID and confidence with larger text
                EmployeeID_display = EmployeeID if EmployeeID and EmployeeID != "N/A" else ("Employee" if 'Inside' in location else "Visitor")
                info_label = f"{EmployeeID_display} | Conf: {confidences[i]:.2f}"
                cv2.putText(frame, info_label, (x1, y2 + 25), 
                           cv2.FONT_HERSHEY_COMPLEX, 0.6, color, 2)

            # Update absence tracking
            if not employee_detected and self.employee_absence_start is None:
                self.employee_absence_start = time.time()
            
            if not visitor_detected and self.visitor_absence_start is None:
                self.visitor_absence_start = time.time()

            # Cleanup disappeared trackers
            if time.time() - last_cleanup > 1.0:
                to_remove = [name for name, data in self.trackers.items() 
                            if time.time() - data['last_seen'] > self.DISAPPEARANCE_TIME]
                for name in to_remove:
                    if name in self.person_status:
                        EmployeeID = self.trackers[name].get('EmployeeID', '')
                        email = self.trackers[name].get('email', '')
                        self.log_person_event(name, "Disappeared", "Unknown", EmployeeID, email)
                        del self.person_status[name]
                    del self.trackers[name]
                last_cleanup = time.time()

            # Draw desk ROI with enhanced visualization
            if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                # Draw main rectangle
                cv2.rectangle(frame, self.DESK_ROI_PT1, self.DESK_ROI_PT2, (255, 255, 0), 4)
                
                # Add corner markers
                marker_size = 40
                # Top-left
                cv2.line(frame, self.DESK_ROI_PT1, 
                        (self.DESK_ROI_PT1[0] + marker_size, self.DESK_ROI_PT1[1]), (0, 255, 255), 6)
                cv2.line(frame, self.DESK_ROI_PT1, 
                        (self.DESK_ROI_PT1[0], self.DESK_ROI_PT1[1] + marker_size), (0, 255, 255), 6)
                # Top-right
                cv2.line(frame, (self.DESK_ROI_PT2[0], self.DESK_ROI_PT1[1]), 
                        (self.DESK_ROI_PT2[0] - marker_size, self.DESK_ROI_PT1[1]), (0, 255, 255), 6)
                cv2.line(frame, (self.DESK_ROI_PT2[0], self.DESK_ROI_PT1[1]), 
                        (self.DESK_ROI_PT2[0], self.DESK_ROI_PT1[1] + marker_size), (0, 255, 255), 6)
                # Bottom-left
                cv2.line(frame, (self.DESK_ROI_PT1[0], self.DESK_ROI_PT2[1]), 
                        (self.DESK_ROI_PT1[0] + marker_size, self.DESK_ROI_PT2[1]), (0, 255, 255), 6)
                cv2.line(frame, (self.DESK_ROI_PT1[0], self.DESK_ROI_PT2[1]), 
                        (self.DESK_ROI_PT1[0], self.DESK_ROI_PT2[1] - marker_size), (0, 255, 255), 6)
                # Bottom-right
                cv2.line(frame, self.DESK_ROI_PT2, 
                        (self.DESK_ROI_PT2[0] - marker_size, self.DESK_ROI_PT2[1]), (0, 255, 255), 6)
                cv2.line(frame, self.DESK_ROI_PT2, 
                        (self.DESK_ROI_PT2[0], self.DESK_ROI_PT2[1] - marker_size), (0, 255, 255), 6)
                

                
                # Employee Status
                employee_status_text = "Employee Status: PRESENT" if employee_detected else "Employee Status: ABSENT"
                employee_color = (0, 255, 0) if employee_detected else (0, 0, 255)
                cv2.putText(frame, employee_status_text, (15, 35), cv2.FONT_HERSHEY_COMPLEX, 
                           0.7, employee_color, 2)
                
                # Employee absence duration
                if not employee_detected and self.employee_absence_start is not None:
                    absence_duration = time.time() - self.employee_absence_start
                    duration_text = f"Employee Absence: {absence_duration:.1f}s"
                    cv2.putText(frame, duration_text, (15, 65), cv2.FONT_HERSHEY_COMPLEX, 
                               0.6, (0, 0, 255), 2)
                
                # Visitor Status with count
                visitor_count = len(current_outside)
                visitor_status_text = f"Visitor Status: {visitor_count} PRESENT" if visitor_detected else "Visitor Status: NONE"
                visitor_color = (0, 165, 255) if visitor_detected else (128, 128, 128)
                cv2.putText(frame, visitor_status_text, (15, 100), cv2.FONT_HERSHEY_COMPLEX, 
                           0.7, visitor_color, 2)
                
                # Count display
                cv2.putText(frame, f'Inside: {len(current_in_desk)} | Outside: {len(current_outside)} | Total: {len(self.trackers)}', 
                           (15, 135), cv2.FONT_HERSHEY_COMPLEX, 0.7, (255, 255, 0), 2)
                
                # Show people inside with larger text
                if current_in_desk:
                    names_in = ", ".join(list(current_in_desk.keys())[:3])
                    if len(current_in_desk) > 3:
                        names_in += f" +{len(current_in_desk)-3}"
                    cv2.putText(frame, f"Inside Desk: {names_in}", (15, 165), 
                               cv2.FONT_HERSHEY_COMPLEX, 0.6, (0, 255, 0), 2)
                
                # Show people outside with larger text
                if current_outside:
                    names_out = ", ".join(list(current_outside.keys())[:3])
                    if len(current_outside) > 3:
                        names_out += f" +{len(current_outside)-3}"
                    cv2.putText(frame, f"Outside Desk: {names_out}", (15, 195), 
                               cv2.FONT_HERSHEY_COMPLEX, 0.6, (0, 165, 255), 2)
                
                # Timestamp with larger text
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                cv2.putText(frame, f"Time: {timestamp}", (15, 225), 
                           cv2.FONT_HERSHEY_COMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(frame, "Mode: FULL | Press 'q' to quit", (15, 250), 
                           cv2.FONT_HERSHEY_COMPLEX, 0.6, (255, 200, 0), 2)

            cv2.imshow('Employee Desk Tracking - Full Mode (Face Recognition)', frame)
            
            frame_count += 1
            
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved to: {self.log_file} and {self.detailed_log_file}")


if __name__ == "__main__":
    import sys
    
    tracker = DeskTracker()
    
    # Check if user wants simple mode or full mode
    mode = "full"  # default
    debug = False
    
    for arg in sys.argv[1:]:
        if arg.lower() in ['simple', 's']:
            mode = "simple"
        elif arg.lower() in ['full', 'f']:
            mode = "full"
        elif arg.lower() in ['debug', 'd', '--debug']:
            debug = True
    
    if debug:
        tracker.debug_mode = True
        print("DEBUG MODE ENABLED - Face search regions will be visualized")
    
    use_simple = (mode == "simple")
    
    print("\n" + "="*60)
    print(f"Starting Desk Tracker in {mode.upper()} mode")
    if debug:
        print("DEBUG MODE: Face search regions visible")
    print("="*60)
    print("\nModes:")
    print("  SIMPLE: Face recognition + presence detection")
    print("  FULL:   Face recognition + detailed tracking + history")
    print("\nFeatures:")
    print("  ✓ Automatic desk area detection")
    print("  ✓ Enhanced face recognition with multiple search regions")
    print("  ✓ Inside/Outside desk tracking")
    print("  ✓ Employee/Visitor status tracking")
    print("  ✓ Absence duration monitoring")
    print("  ✓ Event logging (Entry/Exit)")
    print("  ✓ Detailed position logging")
    print("  ✓ EmployeeID and email tracking")
    print("\nUsage:")
    print("  python desk_tracking_fixed.py simple         # Simple mode")
    print("  python desk_tracking_fixed.py full           # Full mode")
    print("  python desk_tracking_fixed.py debug          # Enable debug visualization")
    print("  python desk_tracking_fixed.py simple debug   # Simple mode + debug")
    print("  python desk_tracking_fixed.py                # Full mode (default)")
    print("\nDebug Mode:")
    print("  Shows colored rectangles where the system searches for faces")
    print("  Useful for troubleshooting face recognition issues")
    print("\nPress 'q' to quit")
    print("="*60 + "\n")
    
    tracker.run(use_simple_mode=use_simple)

In [ ]:
import cv2
import numpy as np
import time
from datetime import datetime
import csv
import os
from collections import defaultdict
import pickle
from sklearn.neighbors import KNeighborsClassifier

class DeskTracker:
    def __init__(self):
        # Load YOLOv4-tiny model 
        cfg_path = 'Yolo_model/yolov4-tiny.cfg'
        names_path = 'Yolo_model/coco.names'
        weights_path = 'Yolo_model/yolov4-tiny.weights'

        # Load names
        with open(names_path, 'r') as f:
            self.classes = [line.strip() for line in f.readlines()]

        # Load model
        self.net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

        # Get output layer names
        layer_names = self.net.getLayerNames()
        self.output_layers = [layer_names[i - 1] for i in self.net.getUnconnectedOutLayers()]

        # Detection parameters
        self.confidence_threshold = 0.3
        self.nms_threshold = 0.4

        # Face recognition parameters
        self.face_recognition_threshold = 20
        self.face_min_size = (30, 30)

        # Load face recognition data
        try:
            with open('data/Organizations.pkl', 'rb') as f:
                self.ORGANIZATIONS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/EmployeeIDs.pkl', 'rb') as f:
                self.EmployeeIDs = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f)
                self.FACES = self.FACES / 255.0
                print(f"FACES shape: {self.FACES.shape}")

            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.EmployeeIDs)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []
            self.EmployeeIDs = []
            self.ORGANIZATIONS = []
            self.EMAILS = []
            self.face_recognition_enabled = False

        # Load face cascade
        self.face_cascade = cv2.CascadeClassifier('data/haarcascade_frontalface_default.xml')

        # Desk ROI
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask = None

        # Tracking
        self.trackers = {}
        self.person_status = {}
        self.MAX_CENTROIDS = 10
        self.DISAPPEARANCE_TIME = 2.0

        # Employee/visitor status
        self.employee_absence_start = None
        self.visitor_absence_start = None
        self.last_employee_present = None
        self.last_visitor_present = None

        # ── LOGGING SETUP ──────────────────────────────────────────────
        # Ensure Logs directory exists
        os.makedirs('Logs', exist_ok=True)

        date = datetime.now().strftime("%d-%m-%Y")

        # LOG FILE 1 – Detection events (Inside / Outside desk area)
        # Columns: Date & Time | EmployeeID | Name | Detection Status
        self.log_file = f'Logs/detection_log_{date}.csv'
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Date & Time', 'EmployeeID', 'Name', 'Detection Status'])

        # LOG FILE 2 – Absence log (only when absence > 3 seconds)
        # Columns: Date & Time | EmployeeID | Name | Absence Duration (s)
        self.absence_log_file = f'Logs/absence_log_{date}.csv'
        if not os.path.exists(self.absence_log_file):
            with open(self.absence_log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Date & Time', 'EmployeeID', 'Name', 'Absence Duration (s)'])

        # Tracks per-person absence start time  {name: start_time}
        self.person_absence_start = {}
        # Tracks last logged absence end time so we don't double-log  {name: end_time}
        self.person_absence_logged = {}
        # ───────────────────────────────────────────────────────────────

        self.monitor_area = None
        self.source_type = "camera"
        self.display_width = 1280
        self.display_height = 720
        self.debug_mode = False

    # ──────────────────────────────────────────────────────────────────
    # LOGGING HELPERS
    # ──────────────────────────────────────────────────────────────────
    def log_event(self, message):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def log_detection(self, name, employee_id, status):
        """Log FILE 1 – person detected inside or outside desk area."""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, employee_id, name, status])
        self.log_event(f"[DETECTION] {name} ({employee_id}) → {status}")

    def log_absence(self, name, employee_id, duration_seconds):
        """Log FILE 2 – absence only when duration > 3 seconds."""
        if duration_seconds <= 3:
            return
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.absence_log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, employee_id, name, f"{duration_seconds:.1f}"])
        self.log_event(f"[ABSENCE]   {name} ({employee_id}) absent for {duration_seconds:.1f}s")

    # ──────────────────────────────────────────────────────────────────
    # PER-PERSON ABSENCE TRACKING
    # ──────────────────────────────────────────────────────────────────
    def mark_person_present(self, name, employee_id):
        """Called every frame a person is detected. Closes any open absence window."""
        if name in self.person_absence_start:
            duration = time.time() - self.person_absence_start.pop(name)
            # Only log absence for recognised employees, not unidentified visitors
            if name != "People":
                self.log_absence(name, employee_id, duration)

    def mark_person_absent(self, name):
        """Called when a person disappears. Opens an absence window."""
        if name not in self.person_absence_start:
            self.person_absence_start[name] = time.time()

    # ──────────────────────────────────────────────────────────────────
    # DESK AREA DETECTION
    # ──────────────────────────────────────────────────────────────────
    def detect_desk_area(self, cap):
        self.log_event("Detecting desk area...")
        desk_candidates = []
        frame_count = 0
        frame = None

        while frame_count < 5:
            ret, frame = cap.read()
            if not ret:
                break
            height, width = frame.shape[:2]
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                break

            desks = []
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    desk_related_classes = [56, 60, 63, 64, 65]
                    if confidence > self.confidence_threshold and class_id in desk_related_classes:
                        center_x = int(obj_detection[0] * width)
                        center_y = int(obj_detection[1] * height)
                        w = int(obj_detection[2] * width)
                        h = int(obj_detection[3] * height)
                        x = int(center_x - w / 2)
                        y = int(center_y - h / 2)
                        desks.append([x, y, w, h, confidence, class_id])
            if desks:
                desk_candidates.extend(desks)
            frame_count += 1

        if len(desk_candidates) < 2:
            self.log_event("Not enough desk objects detected. Using default desk area.")
            if frame is not None:
                height, width = frame.shape[:2]
            else:
                height, width = 480, 640
            min_x = int(width * 0.2)
            min_y = int(height * 0.4)
            max_x = int(width * 0.8)
            max_y = int(height * 0.9)
        else:
            x_coords     = [d[0] for d in desk_candidates]
            y_coords     = [d[1] for d in desk_candidates]
            max_x_coords = [d[0] + d[2] for d in desk_candidates]
            max_y_coords = [d[1] + d[3] for d in desk_candidates]
            min_x = max(0,     int(min(x_coords)     - 0.1 * width))
            min_y = max(0,     int(min(y_coords)     - 0.1 * height))
            max_x = min(width, int(max(max_x_coords) + 0.1 * width))
            max_y = min(height,int(max(max_y_coords) + 0.1 * height))
            self.log_event(f"Desk area detected: ({min_x}, {min_y}) to ({max_x}, {max_y})")

        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        self.monitor_area = (min_x, min_y, max_x, max_y)
        return (min_x, min_y, max_x, max_y)

    # ──────────────────────────────────────────────────────────────────
    # FACE RECOGNITION
    # ──────────────────────────────────────────────────────────────────
    def get_name_from_face(self, face_region):
        if not self.face_recognition_enabled or face_region.size == 0:
            return "People", "N/A", ""
        if face_region.shape[0] < 30 or face_region.shape[1] < 30:
            return "People", "N/A", ""

        gray_region = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        faces = []
        detection_params = [
            (1.1, 5, (20, 20)),
            (1.05, 4, (25, 25)),
            (1.2, 5, (30, 30)),
            (1.3, 4, (35, 35)),
            (1.15, 3, (25, 25)),
        ]
        for scale, neighbors, min_size in detection_params:
            detected = self.face_cascade.detectMultiScale(
                gray_region, scale, neighbors, minSize=min_size)
            if len(detected) > 0:
                faces = detected
                break

        Name = "People"
        EmployeeID = "N/A"
        Email = ""

        for (x, y, w_face, h_face) in faces:
            crop_img = face_region[y:y+h_face, x:x+w_face, :]
            if crop_img.size == 0:
                continue
            try:
                resized_img  = cv2.resize(crop_img, (75, 100))
                gray         = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
                resized_flat = gray.reshape(1, -1) / 255.0
                if resized_flat.shape[1] != self.FACES.shape[1]:
                    continue
                distances, _ = self.knn.kneighbors(resized_flat)
                if distances[0][0] <= self.face_recognition_threshold:
                    output = self.knn.predict(resized_flat)
                    emp_id = output[0]
                    if emp_id in self.EmployeeIDs:
                        index      = self.EmployeeIDs.index(emp_id)
                        Name       = self.NAMES[index]
                        EmployeeID = self.EmployeeIDs[index] if index < len(self.EmployeeIDs) else "N/A"
                        Email      = self.EMAILS[index]      if index < len(self.EMAILS)      else ""
                        break
            except Exception:
                continue

        return Name, EmployeeID, Email

    def get_name_from_centroid(self, cx, cy, frame):
        h, w = frame.shape[:2]
        search_regions = [
            (max(0,cx-180), max(0,cy-280), min(w,cx+180), min(h,cy+50)),
            (max(0,cx-220), max(0,cy-320), min(w,cx+220), min(h,cy+80)),
            (max(0,cx-150), max(0,cy-350), min(w,cx+150), min(h,cy)),
            (max(0,cx-200), max(0,cy-200), min(w,cx+200), min(h,cy+150)),
        ]
        for idx, (rx1, ry1, rx2, ry2) in enumerate(search_regions):
            if self.debug_mode:
                color = [(255,0,255),(0,255,255),(255,255,0),(255,128,0)][idx]
                cv2.rectangle(frame, (rx1,ry1), (rx2,ry2), color, 2)
                cv2.putText(frame, f"Search {idx+1}", (rx1, ry1-5),
                            cv2.FONT_HERSHEY_COMPLEX, 0.4, color, 1)
            face_region = frame[ry1:ry2, rx1:rx2]
            name, EmployeeID, email = self.get_name_from_face(face_region)
            if name != "People":
                return name, EmployeeID, email
        return "People", "N/A", ""

    # ──────────────────────────────────────────────────────────────────
    # MAIN RUN
    # ──────────────────────────────────────────────────────────────────
    def run(self, use_simple_mode=False):
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            self.log_event("Error: Cannot open camera")
            return
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
        cap.set(cv2.CAP_PROP_FPS, 30)
        actual_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        actual_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.log_event(f"Camera resolution: {actual_width}x{actual_height}")
        self.detect_desk_area(cap)
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        if use_simple_mode:
            self.log_event("Running in SIMPLE mode")
            self._run_simple_mode(cap)
        else:
            self.log_event("Running in FULL mode")
            self._run_full_mode(cap)

    # ──────────────────────────────────────────────────────────────────
    # SIMPLE MODE
    # ──────────────────────────────────────────────────────────────────
    def _run_simple_mode(self, cap):
        frame_count  = 0
        log_interval = 30

        cv2.namedWindow('Employee Desk Tracking - Simple Mode', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking - Simple Mode', self.display_width, self.display_height)

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            h, w = frame.shape[:2]
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416,416), swapRB=True, crop=False)
            self.net.setInput(blob)
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Detection error: {e}")
                continue

            boxes, confidences = [], []
            for detection in detections:
                for obj in detection:
                    scores   = obj[5:]
                    class_id = np.argmax(scores)
                    conf     = scores[class_id]
                    if conf > self.confidence_threshold and class_id == 0:
                        center_x   = int(obj[0] * w)
                        center_y   = int(obj[1] * h)
                        width_box  = int(obj[2] * w)
                        height_box = int(obj[3] * h)
                        x = int(center_x - width_box  / 2)
                        y = int(center_y - height_box / 2)
                        boxes.append([x, y, width_box, height_box])
                        confidences.append(float(conf))

            people_in_desk    = []
            people_outside_desk = []
            employee_detected = False
            visitor_detected  = False
            seen_names        = set()   # names seen this frame

            if boxes:
                indices = cv2.dnn.NMSBoxes(boxes, confidences, self.confidence_threshold, 0.4)
                if len(indices) > 0:
                    for i in indices.flatten():
                        x, y, width_box, height_box = boxes[i]
                        cx = x + width_box  // 2
                        cy = y + height_box // 2

                        name, EmployeeID, email = self.get_name_from_centroid(cx, cy, frame)
                        seen_names.add(name)

                        # Desk overlap
                        px1, py1 = x, y
                        px2, py2 = x + width_box, y + height_box
                        xi = max(self.monitor_area[0], px1)
                        yi = max(self.monitor_area[1], py1)
                        wi = min(self.monitor_area[2], px2) - xi
                        hi = min(self.monitor_area[3], py2) - yi
                        is_in_desk_area = False
                        if wi > 0 and hi > 0:
                            overlap = (wi * hi) / (width_box * height_box)
                            if overlap > 0.3:
                                is_in_desk_area = True

                        location = "Inside Desk" if is_in_desk_area else "Outside Desk"

                        # ── FILE 1: log on status change ──────────────────
                        if name not in self.person_status:
                            self.person_status[name] = location
                            self.log_detection(name, EmployeeID, f"Detected - {location}")
                        elif self.person_status[name] != location:
                            self.person_status[name] = location
                            self.log_detection(name, EmployeeID, location)
                        # ─────────────────────────────────────────────────

                        # Update employee / visitor flags
                        if is_in_desk_area:
                            people_in_desk.append(name)
                            if name != "People":
                                employee_detected = True
                                self.last_employee_present = time.time()
                                if self.employee_absence_start is not None:
                                    self.employee_absence_start = None
                            color = (0, 255, 0)
                            label = f"{name} - Employee (Inside)"
                        else:
                            people_outside_desk.append(name)
                            visitor_detected = True
                            self.last_visitor_present = time.time()
                            if self.visitor_absence_start is not None:
                                self.visitor_absence_start = None
                            color = (0, 0, 255)
                            label = f"{name} - Visitor (Outside)"

                        # ── FILE 2: mark present → closes absence window ──
                        self.mark_person_present(name, EmployeeID)
                        # ─────────────────────────────────────────────────

                        # Draw bounding box
                        cv2.rectangle(frame, (x, y), (x+width_box, y+height_box), color, 3)
                        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_COMPLEX, 0.8, 2)[0]
                        cv2.rectangle(frame, (x, y-40), (x+label_size[0]+15, y), color, -1)
                        cv2.putText(frame, label, (x+8, y-12),
                                    cv2.FONT_HERSHEY_COMPLEX, 0.8, (255,255,255), 2)
                        id_label = f"{EmployeeID} | Conf: {confidences[i]:.2f}"
                        cv2.putText(frame, id_label, (x, y+height_box+25),
                                    cv2.FONT_HERSHEY_COMPLEX, 0.6, color, 2)

            # ── FILE 2: mark people NOT seen this frame as absent ─────
            for tracked_name in list(self.person_status.keys()):
                if tracked_name not in seen_names:
                    self.mark_person_absent(tracked_name)
            # ─────────────────────────────────────────────────────────

            # Absence tracking for overlay display
            if not employee_detected and self.employee_absence_start is None:
                self.employee_absence_start = time.time()
            if not visitor_detected and self.visitor_absence_start is None:
                self.visitor_absence_start = time.time()

            # ── Draw desk area ────────────────────────────────────────
            cv2.rectangle(frame,
                          (self.monitor_area[0], self.monitor_area[1]),
                          (self.monitor_area[2], self.monitor_area[3]),
                          (255,255,0), 4)
            ms = 30
            ma = self.monitor_area
            for pt, dx, dy in [
                ((ma[0],ma[1]),(ms,0),(0,ms)),
                ((ma[2],ma[1]),(-ms,0),(0,ms)),
                ((ma[0],ma[3]),(ms,0),(0,-ms)),
                ((ma[2],ma[3]),(-ms,0),(0,-ms)),
            ]:
                cv2.line(frame, pt, (pt[0]+dx[0], pt[1]+dx[1]), (0,255,255), 6)
                cv2.line(frame, pt, (pt[0]+dy[0], pt[1]+dy[1]), (0,255,255), 6)

            

            emp_text  = "Employee Status: PRESENT" if employee_detected else "Employee Status: ABSENT"
            emp_color = (0,255,0) if employee_detected else (0,0,255)
            cv2.putText(frame, emp_text, (15,35), cv2.FONT_HERSHEY_COMPLEX, 0.7, emp_color, 2)

            if not employee_detected and self.employee_absence_start is not None:
                dur = time.time() - self.employee_absence_start
                cv2.putText(frame, f"Employee Absence: {dur:.1f}s",
                            (15,65), cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,0,255), 2)

            vis_count = len(people_outside_desk)
            vis_text  = f"Visitor Status: {vis_count} PRESENT" if visitor_detected else "Visitor Status: NONE"
            vis_color = (0,165,255) if visitor_detected else (128,128,128)
            cv2.putText(frame, vis_text, (15,100), cv2.FONT_HERSHEY_COMPLEX, 0.7, vis_color, 2)

            cv2.putText(frame, f"In Desk: {len(people_in_desk)} | Outside: {len(people_outside_desk)}",
                        (15,135), cv2.FONT_HERSHEY_COMPLEX, 0.7, (255,255,0), 2)

            if people_in_desk:
                names_in = ", ".join(people_in_desk[:3])
                if len(people_in_desk) > 3:
                    names_in += f" +{len(people_in_desk)-3}"
                cv2.putText(frame, f"Inside: {names_in}", (15,165),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,255,0), 2)

            if people_outside_desk:
                names_out = ", ".join(people_outside_desk[:3])
                if len(people_outside_desk) > 3:
                    names_out += f" +{len(people_outside_desk)-3}"
                cv2.putText(frame, f"Outside: {names_out}", (15,195),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,165,255), 2)

            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {ts}", (15,225), cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,255,255), 2)
            cv2.putText(frame, "Mode: SIMPLE | Press 'q' to quit", (15,250),
                        cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,200,0), 2)

            cv2.imshow('Employee Desk Tracking - Simple Mode', frame)
            frame_count += 1
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved → {self.log_file}  |  {self.absence_log_file}")

    # ──────────────────────────────────────────────────────────────────
    # FULL MODE
    # ──────────────────────────────────────────────────────────────────
    def _run_full_mode(self, cap):
        last_cleanup = time.time()
        frame_count  = 0
        log_interval = 30

        cv2.namedWindow('Employee Desk Tracking - Full Mode', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking - Full Mode', self.display_width, self.display_height)

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            h, w = frame.shape[:2]

            if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                self.desk_mask = np.zeros((h, w), np.uint8)
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(self.desk_mask, self.DESK_ROI_PT1, self.DESK_ROI_PT2, 255, -1)

            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416,416), swapRB=True, crop=False)
            self.net.setInput(blob)
            outputs = self.net.forward(self.output_layers)

            boxes, confidences = [], []
            for output in outputs:
                for detection in output:
                    scores   = detection[5:]
                    class_id = np.argmax(scores)
                    conf     = scores[class_id]
                    if conf > 0.5 and self.classes[class_id] == 'person':
                        cx = int(detection[0] * w)
                        cy = int(detection[1] * h)
                        bw = int(detection[2] * w)
                        bh = int(detection[3] * h)
                        boxes.append([int(cx-bw/2), int(cy-bh/2), bw, bh])
                        confidences.append(float(conf))

            indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, self.nms_threshold)

            current_in_desk  = {}
            current_outside  = {}
            employee_detected = False
            visitor_detected  = False
            seen_names        = set()
            det_list = indices.flatten() if len(indices) > 0 else []

            for i in det_list:
                x1, y1, bw, bh = boxes[i]
                x2, y2 = x1+bw, y1+bh
                cx, cy = (x1+x2)//2, (y1+y2)//2

                name, EmployeeID, email = self.get_name_from_centroid(cx, cy, frame)
                seen_names.add(name)

                if name not in self.trackers:
                    self.trackers[name] = {'centroids':[], 'last_seen':0,
                                           'EmployeeID': EmployeeID, 'email': email}
                self.trackers[name].update({'bbox':(x1,y1,x2,y2), 'last_seen':time.time(),
                                            'EmployeeID': EmployeeID, 'email': email})
                self.trackers[name]['centroids'].append((cx, cy))
                if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                    self.trackers[name]['centroids'].pop(0)

                # Desk overlap
                overlap_ratio = 0
                if x2 > x1 and y2 > y1:
                    ys = slice(max(0,y1), min(h,y2))
                    xs = slice(max(0,x1), min(w,x2))
                    roi = self.desk_mask[ys, xs]
                    if roi.size > 0:
                        overlap_ratio = np.sum(roi > 0) / roi.size

                location = 'Inside Desk' if overlap_ratio > 0.4 else 'Outside Desk'

                if 'Inside' in location:
                    current_in_desk[name] = (EmployeeID, email)
                    if name != "People":
                        employee_detected = True
                        self.last_employee_present = time.time()
                        if self.employee_absence_start is not None:
                            self.employee_absence_start = None
                else:
                    current_outside[name] = (EmployeeID, email)
                    visitor_detected = True
                    self.last_visitor_present = time.time()
                    if self.visitor_absence_start is not None:
                        self.visitor_absence_start = None

                # ── FILE 1: log on status change ──────────────────────
                if name not in self.person_status:
                    self.person_status[name] = location
                    self.log_detection(name, EmployeeID, f"Detected - {location}")
                elif self.person_status[name] != location:
                    self.person_status[name] = location
                    self.log_detection(name, EmployeeID, location)
                # ─────────────────────────────────────────────────────

                # ── FILE 2: mark present ──────────────────────────────
                self.mark_person_present(name, EmployeeID)
                # ─────────────────────────────────────────────────────

                # Draw
                color = (0,255,0) if 'Inside' in location else (0,0,255)
                cv2.rectangle(frame, (x1,y1), (x2,y2), color, 3)
                role  = "Employee" if 'Inside' in location else "Visitor"
                label = f"{name} - {role} ({location.split()[0]})"
                lsz   = cv2.getTextSize(label, cv2.FONT_HERSHEY_COMPLEX, 0.8, 2)[0]
                cv2.rectangle(frame, (x1,y1-45), (x1+lsz[0]+15, y1), color, -1)
                cv2.putText(frame, label, (x1+8, y1-15),
                            cv2.FONT_HERSHEY_COMPLEX, 0.8, (255,255,255), 2)
                id_display = EmployeeID if EmployeeID and EmployeeID != "N/A" else \
                             ("Employee" if 'Inside' in location else "Visitor")
                cv2.putText(frame, f"{id_display} | Conf: {confidences[i]:.2f}",
                            (x1, y2+25), cv2.FONT_HERSHEY_COMPLEX, 0.6, color, 2)

            # ── FILE 2: mark absent for people not seen this frame ────
            for tracked_name in list(self.person_status.keys()):
                if tracked_name not in seen_names:
                    self.mark_person_absent(tracked_name)
            # ─────────────────────────────────────────────────────────

            if not employee_detected and self.employee_absence_start is None:
                self.employee_absence_start = time.time()
            if not visitor_detected and self.visitor_absence_start is None:
                self.visitor_absence_start = time.time()

            # Cleanup disappeared trackers
            if time.time() - last_cleanup > 1.0:
                to_remove = [n for n, d in self.trackers.items()
                             if time.time() - d['last_seen'] > self.DISAPPEARANCE_TIME]
                for name in to_remove:
                    if name in self.person_status:
                        eid   = self.trackers[name].get('EmployeeID', 'N/A')
                        self.log_detection(name, eid, "Disappeared")
                        del self.person_status[name]
                    del self.trackers[name]
                last_cleanup = time.time()

            # ── Draw desk ROI ─────────────────────────────────────────
            if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                cv2.rectangle(frame, self.DESK_ROI_PT1, self.DESK_ROI_PT2, (255,255,0), 4)
                ms = 40
                p1, p2 = self.DESK_ROI_PT1, self.DESK_ROI_PT2
                for pt, dx, dy in [
                    (p1,           (ms,0),  (0,ms)),
                    ((p2[0],p1[1]),(-ms,0), (0,ms)),
                    ((p1[0],p2[1]),(ms,0),  (0,-ms)),
                    (p2,           (-ms,0), (0,-ms)),
                ]:
                    cv2.line(frame, pt, (pt[0]+dx[0],pt[1]+dx[1]), (0,255,255), 6)
                    cv2.line(frame, pt, (pt[0]+dy[0],pt[1]+dy[1]), (0,255,255), 6)


            emp_text  = "Employee Status: PRESENT" if employee_detected else "Employee Status: ABSENT"
            emp_color = (0,255,0) if employee_detected else (0,0,255)
            cv2.putText(frame, emp_text, (15,35), cv2.FONT_HERSHEY_COMPLEX, 0.7, emp_color, 2)

            if not employee_detected and self.employee_absence_start is not None:
                dur = time.time() - self.employee_absence_start
                cv2.putText(frame, f"Employee Absence: {dur:.1f}s",
                            (15,65), cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,0,255), 2)

            vis_count = len(current_outside)
            vis_text  = f"Visitor Status: {vis_count} PRESENT" if visitor_detected else "Visitor Status: NONE"
            vis_color = (0,165,255) if visitor_detected else (128,128,128)
            cv2.putText(frame, vis_text, (15,100), cv2.FONT_HERSHEY_COMPLEX, 0.7, vis_color, 2)

            cv2.putText(frame,
                        f"Inside: {len(current_in_desk)} | Outside: {len(current_outside)} | Total: {len(self.trackers)}",
                        (15,135), cv2.FONT_HERSHEY_COMPLEX, 0.7, (255,255,0), 2)

            if current_in_desk:
                nin = ", ".join(list(current_in_desk.keys())[:3])
                if len(current_in_desk) > 3: nin += f" +{len(current_in_desk)-3}"
                cv2.putText(frame, f"Inside Desk: {nin}", (15,165),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,255,0), 2)
            if current_outside:
                nout = ", ".join(list(current_outside.keys())[:3])
                if len(current_outside) > 3: nout += f" +{len(current_outside)-3}"
                cv2.putText(frame, f"Outside Desk: {nout}", (15,195),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,165,255), 2)

            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {ts}", (15,225), cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,255,255), 2)
            cv2.putText(frame, "Mode: FULL | Press 'q' to quit", (15,250),
                        cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,200,0), 2)

            cv2.imshow('Employee Desk Tracking - Full Mode (Face Recognition)', frame)
            frame_count += 1
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved → {self.log_file}  |  {self.absence_log_file}")


# ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import sys

    tracker = DeskTracker()

    mode  = "full"
    debug = False
    for arg in sys.argv[1:]:
        if arg.lower() in ['simple', 's']:   mode  = "simple"
        elif arg.lower() in ['full', 'f']:   mode  = "full"
        elif arg.lower() in ['debug','d','--debug']: debug = True

    if debug:
        tracker.debug_mode = True
        print("DEBUG MODE ENABLED - Face search regions will be visualized")

    print("\n" + "="*60)
    print(f"Starting Desk Tracker in {mode.upper()} mode")
    print("="*60)
    print("\nLog files created in  ./Logs/")
    print(f"  Detection log : Logs/detection_log_<date>.csv")
    print(f"    Columns: Date & Time | EmployeeID | Name | Detection Status")
    print(f"  Absence log   : Logs/absence_log_<date>.csv")
    print(f"    Columns: Date & Time | EmployeeID | Name | Absence Duration (s)")
    print(f"    (only records absences longer than 3 seconds)")
    print("\nPress 'q' to quit\n" + "="*60 + "\n")

    tracker.run(use_simple_mode=(mode == "simple"))

In [ ]:
import cv2
import numpy as np
import time
from datetime import datetime
import csv
import os
from collections import defaultdict
import pickle
from sklearn.neighbors import KNeighborsClassifier

class DeskTracker:
    def __init__(self):
        # Load YOLOv4-tiny model 
        cfg_path = 'Yolo_model/yolov4-tiny.cfg'
        names_path = 'Yolo_model/coco.names'
        weights_path = 'Yolo_model/yolov4-tiny.weights'

        # Load names
        with open(names_path, 'r') as f:
            self.classes = [line.strip() for line in f.readlines()]

        # Load model
        self.net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

        # Get output layer names
        layer_names = self.net.getLayerNames()
        self.output_layers = [layer_names[i - 1] for i in self.net.getUnconnectedOutLayers()]

        # Detection parameters
        self.confidence_threshold = 0.3
        self.nms_threshold = 0.4

        # Face recognition parameters
        self.face_recognition_threshold = 20
        self.face_min_size = (30, 30)

        # Load face recognition data
        try:
            with open('data/Organizations.pkl', 'rb') as f:
                self.ORGANIZATIONS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/EmployeeIDs.pkl', 'rb') as f:
                self.EmployeeIDs = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f)
                self.FACES = self.FACES / 255.0
                print(f"FACES shape: {self.FACES.shape}")

            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.EmployeeIDs)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []
            self.EmployeeIDs = []
            self.ORGANIZATIONS = []
            self.EMAILS = []
            self.face_recognition_enabled = False

        # Load face cascade
        self.face_cascade = cv2.CascadeClassifier('data/haarcascade_frontalface_default.xml')

        # Desk ROI
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask = None

        # Tracking
        self.trackers = {}
        self.person_status = {}
        self.MAX_CENTROIDS = 10
        self.DISAPPEARANCE_TIME = 2.0

        # Employee/visitor status
        self.employee_absence_start = None
        self.visitor_absence_start = None
        self.last_employee_present = None
        self.last_visitor_present = None

        # ── LOGGING SETUP ──────────────────────────────────────────────
        # Ensure dated subfolder exists:  Logs/Log_DD-MM-YYYY/
        date = datetime.now().strftime("%d-%m-%Y")
        log_folder = os.path.join('Logs', f'Log_{date}')
        os.makedirs(log_folder, exist_ok=True)

        # LOG FILE 1 – Detection events (Inside / Outside desk area)
        # Columns: Date & Time | EmployeeID | Name | Detection Status
        self.log_file = os.path.join(log_folder, f'detection_log_{date}.csv')
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Date & Time', 'EmployeeID', 'Name', 'Detection Status'])

        # LOG FILE 2 – Absence log (only when absence > 3 seconds)
        # Columns: Date & Time | EmployeeID | Name | Absence Duration (s)
        self.absence_log_file = os.path.join(log_folder, f'absence_log_{date}.csv')
        if not os.path.exists(self.absence_log_file):
            with open(self.absence_log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Date & Time', 'EmployeeID', 'Name', 'Absence Duration (s)'])

        # Tracks per-person absence start time  {name: start_time}
        self.person_absence_start = {}
        # Tracks last logged absence end time so we don't double-log  {name: end_time}
        self.person_absence_logged = {}
        # ───────────────────────────────────────────────────────────────

        self.monitor_area = None
        self.source_type = "camera"
        self.display_width = 1280
        self.display_height = 720
        self.debug_mode = False

    # ──────────────────────────────────────────────────────────────────
    # LOGGING HELPERS
    # ──────────────────────────────────────────────────────────────────
    def log_event(self, message):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def log_detection(self, name, employee_id, status):
        """Log FILE 1 – person detected inside or outside desk area."""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, employee_id, name, status])
        self.log_event(f"[DETECTION] {name} ({employee_id}) → {status}")

    def log_absence(self, name, employee_id, duration_seconds):
        """Log FILE 2 – absence only when duration > 3 seconds."""
        if duration_seconds <= 3:
            return
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.absence_log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, employee_id, name, f"{duration_seconds:.1f}"])
        self.log_event(f"[ABSENCE]   {name} ({employee_id}) absent for {duration_seconds:.1f}s")

    # ──────────────────────────────────────────────────────────────────
    # PER-PERSON ABSENCE TRACKING
    # ──────────────────────────────────────────────────────────────────
    def mark_person_present(self, name, employee_id):
        """Called every frame a person is detected. Closes any open absence window."""
        if name in self.person_absence_start:
            duration = time.time() - self.person_absence_start.pop(name)
            # Only log absence for recognised employees, not unidentified visitors
            if name != "People":
                self.log_absence(name, employee_id, duration)

    def mark_person_absent(self, name):
        """Called when a person disappears. Opens an absence window."""
        if name not in self.person_absence_start:
            self.person_absence_start[name] = time.time()

    # ──────────────────────────────────────────────────────────────────
    # DESK AREA DETECTION
    # ──────────────────────────────────────────────────────────────────
    def detect_desk_area(self, cap):
        self.log_event("Detecting desk area...")
        desk_candidates = []
        frame_count = 0
        frame = None

        while frame_count < 5:
            ret, frame = cap.read()
            if not ret:
                break
            height, width = frame.shape[:2]
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                break

            desks = []
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    desk_related_classes = [56, 60, 63, 64, 65]
                    if confidence > self.confidence_threshold and class_id in desk_related_classes:
                        center_x = int(obj_detection[0] * width)
                        center_y = int(obj_detection[1] * height)
                        w = int(obj_detection[2] * width)
                        h = int(obj_detection[3] * height)
                        x = int(center_x - w / 2)
                        y = int(center_y - h / 2)
                        desks.append([x, y, w, h, confidence, class_id])
            if desks:
                desk_candidates.extend(desks)
            frame_count += 1

        if len(desk_candidates) < 2:
            self.log_event("Not enough desk objects detected. Using default desk area.")
            if frame is not None:
                height, width = frame.shape[:2]
            else:
                height, width = 480, 640
            min_x = int(width * 0.2)
            min_y = int(height * 0.4)
            max_x = int(width * 0.8)
            max_y = int(height * 0.9)
        else:
            x_coords     = [d[0] for d in desk_candidates]
            y_coords     = [d[1] for d in desk_candidates]
            max_x_coords = [d[0] + d[2] for d in desk_candidates]
            max_y_coords = [d[1] + d[3] for d in desk_candidates]
            min_x = max(0,     int(min(x_coords)     - 0.1 * width))
            min_y = max(0,     int(min(y_coords)     - 0.1 * height))
            max_x = min(width, int(max(max_x_coords) + 0.1 * width))
            max_y = min(height,int(max(max_y_coords) + 0.1 * height))
            self.log_event(f"Desk area detected: ({min_x}, {min_y}) to ({max_x}, {max_y})")

        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        self.monitor_area = (min_x, min_y, max_x, max_y)
        return (min_x, min_y, max_x, max_y)

    # ──────────────────────────────────────────────────────────────────
    # FACE RECOGNITION
    # ──────────────────────────────────────────────────────────────────
    def get_name_from_face(self, face_region):
        if not self.face_recognition_enabled or face_region.size == 0:
            return "People", "N/A", ""
        if face_region.shape[0] < 30 or face_region.shape[1] < 30:
            return "People", "N/A", ""

        gray_region = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        faces = []
        detection_params = [
            (1.1, 5, (20, 20)),
            (1.05, 4, (25, 25)),
            (1.2, 5, (30, 30)),
            (1.3, 4, (35, 35)),
            (1.15, 3, (25, 25)),
        ]
        for scale, neighbors, min_size in detection_params:
            detected = self.face_cascade.detectMultiScale(
                gray_region, scale, neighbors, minSize=min_size)
            if len(detected) > 0:
                faces = detected
                break

        Name = "People"
        EmployeeID = "N/A"
        Email = ""

        for (x, y, w_face, h_face) in faces:
            crop_img = face_region[y:y+h_face, x:x+w_face, :]
            if crop_img.size == 0:
                continue
            try:
                resized_img  = cv2.resize(crop_img, (75, 100))
                gray         = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
                resized_flat = gray.reshape(1, -1) / 255.0
                if resized_flat.shape[1] != self.FACES.shape[1]:
                    continue
                distances, _ = self.knn.kneighbors(resized_flat)
                if distances[0][0] <= self.face_recognition_threshold:
                    output = self.knn.predict(resized_flat)
                    emp_id = output[0]
                    if emp_id in self.EmployeeIDs:
                        index      = self.EmployeeIDs.index(emp_id)
                        Name       = self.NAMES[index]
                        EmployeeID = self.EmployeeIDs[index] if index < len(self.EmployeeIDs) else "N/A"
                        Email      = self.EMAILS[index]      if index < len(self.EMAILS)      else ""
                        break
            except Exception:
                continue

        return Name, EmployeeID, Email

    def get_name_from_centroid(self, cx, cy, frame):
        h, w = frame.shape[:2]
        search_regions = [
            (max(0,cx-180), max(0,cy-280), min(w,cx+180), min(h,cy+50)),
            (max(0,cx-220), max(0,cy-320), min(w,cx+220), min(h,cy+80)),
            (max(0,cx-150), max(0,cy-350), min(w,cx+150), min(h,cy)),
            (max(0,cx-200), max(0,cy-200), min(w,cx+200), min(h,cy+150)),
        ]
        for idx, (rx1, ry1, rx2, ry2) in enumerate(search_regions):
            if self.debug_mode:
                color = [(255,0,255),(0,255,255),(255,255,0),(255,128,0)][idx]
                cv2.rectangle(frame, (rx1,ry1), (rx2,ry2), color, 2)
                cv2.putText(frame, f"Search {idx+1}", (rx1, ry1-5),
                            cv2.FONT_HERSHEY_COMPLEX, 0.4, color, 1)
            face_region = frame[ry1:ry2, rx1:rx2]
            name, EmployeeID, email = self.get_name_from_face(face_region)
            if name != "People":
                return name, EmployeeID, email
        return "People", "N/A", ""

    # ──────────────────────────────────────────────────────────────────
    # MAIN RUN
    # ──────────────────────────────────────────────────────────────────
    def run(self, use_simple_mode=False):
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            self.log_event("Error: Cannot open camera")
            return
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
        cap.set(cv2.CAP_PROP_FPS, 30)
        actual_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        actual_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.log_event(f"Camera resolution: {actual_width}x{actual_height}")
        self.detect_desk_area(cap)
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        if use_simple_mode:
            self.log_event("Running in SIMPLE mode")
            self._run_simple_mode(cap)
        else:
            self.log_event("Running in FULL mode")
            self._run_full_mode(cap)

    # ──────────────────────────────────────────────────────────────────
    # SIMPLE MODE
    # ──────────────────────────────────────────────────────────────────
    def _run_simple_mode(self, cap):
        frame_count  = 0
        log_interval = 30

        cv2.namedWindow('Employee Desk Tracking - Simple Mode', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking - Simple Mode', self.display_width, self.display_height)

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            h, w = frame.shape[:2]
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416,416), swapRB=True, crop=False)
            self.net.setInput(blob)
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Detection error: {e}")
                continue

            boxes, confidences = [], []
            for detection in detections:
                for obj in detection:
                    scores   = obj[5:]
                    class_id = np.argmax(scores)
                    conf     = scores[class_id]
                    if conf > self.confidence_threshold and class_id == 0:
                        center_x   = int(obj[0] * w)
                        center_y   = int(obj[1] * h)
                        width_box  = int(obj[2] * w)
                        height_box = int(obj[3] * h)
                        x = int(center_x - width_box  / 2)
                        y = int(center_y - height_box / 2)
                        boxes.append([x, y, width_box, height_box])
                        confidences.append(float(conf))

            people_in_desk    = []
            people_outside_desk = []
            employee_detected = False
            visitor_detected  = False
            seen_names        = set()   # names seen this frame

            if boxes:
                indices = cv2.dnn.NMSBoxes(boxes, confidences, self.confidence_threshold, 0.4)
                if len(indices) > 0:
                    for i in indices.flatten():
                        x, y, width_box, height_box = boxes[i]
                        cx = x + width_box  // 2
                        cy = y + height_box // 2

                        name, EmployeeID, email = self.get_name_from_centroid(cx, cy, frame)
                        seen_names.add(name)

                        # Desk overlap
                        px1, py1 = x, y
                        px2, py2 = x + width_box, y + height_box
                        xi = max(self.monitor_area[0], px1)
                        yi = max(self.monitor_area[1], py1)
                        wi = min(self.monitor_area[2], px2) - xi
                        hi = min(self.monitor_area[3], py2) - yi
                        is_in_desk_area = False
                        if wi > 0 and hi > 0:
                            overlap = (wi * hi) / (width_box * height_box)
                            if overlap > 0.3:
                                is_in_desk_area = True

                        location = "Inside Desk" if is_in_desk_area else "Outside Desk"

                        # ── FILE 1: log on status change ──────────────────
                        if name not in self.person_status:
                            self.person_status[name] = location
                            self.log_detection(name, EmployeeID, f"Detected - {location}")
                        elif self.person_status[name] != location:
                            self.person_status[name] = location
                            self.log_detection(name, EmployeeID, location)
                        # ─────────────────────────────────────────────────

                        # Update employee / visitor flags
                        if is_in_desk_area:
                            people_in_desk.append(name)
                            if name != "People":
                                employee_detected = True
                                self.last_employee_present = time.time()
                                if self.employee_absence_start is not None:
                                    self.employee_absence_start = None
                            color = (0, 255, 0)
                            label = f"{name} - Employee (Inside)"
                        else:
                            people_outside_desk.append(name)
                            visitor_detected = True
                            self.last_visitor_present = time.time()
                            if self.visitor_absence_start is not None:
                                self.visitor_absence_start = None
                            color = (0, 0, 255)
                            label = f"{name} - Visitor (Outside)"

                        # ── FILE 2: mark present → closes absence window ──
                        self.mark_person_present(name, EmployeeID)
                        # ─────────────────────────────────────────────────

                        # Draw bounding box
                        cv2.rectangle(frame, (x, y), (x+width_box, y+height_box), color, 3)
                        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_COMPLEX, 0.8, 2)[0]
                        cv2.rectangle(frame, (x, y-40), (x+label_size[0]+15, y), color, -1)
                        cv2.putText(frame, label, (x+8, y-12),
                                    cv2.FONT_HERSHEY_COMPLEX, 0.8, (255,255,255), 2)
                        id_label = f"{EmployeeID} | Conf: {confidences[i]:.2f}"
                        cv2.putText(frame, id_label, (x, y+height_box+25),
                                    cv2.FONT_HERSHEY_COMPLEX, 0.6, color, 2)

            # ── FILE 2: mark people NOT seen this frame as absent ─────
            for tracked_name in list(self.person_status.keys()):
                if tracked_name not in seen_names:
                    self.mark_person_absent(tracked_name)
            # ─────────────────────────────────────────────────────────

            # Absence tracking for overlay display
            if not employee_detected and self.employee_absence_start is None:
                self.employee_absence_start = time.time()
            if not visitor_detected and self.visitor_absence_start is None:
                self.visitor_absence_start = time.time()

            # ── Draw desk area ────────────────────────────────────────
            cv2.rectangle(frame,
                          (self.monitor_area[0], self.monitor_area[1]),
                          (self.monitor_area[2], self.monitor_area[3]),
                          (255,255,0), 4)
            ms = 30
            ma = self.monitor_area
            for pt, dx, dy in [
                ((ma[0],ma[1]),(ms,0),(0,ms)),
                ((ma[2],ma[1]),(-ms,0),(0,ms)),
                ((ma[0],ma[3]),(ms,0),(0,-ms)),
                ((ma[2],ma[3]),(-ms,0),(0,-ms)),
            ]:
                cv2.line(frame, pt, (pt[0]+dx[0], pt[1]+dx[1]), (0,255,255), 6)
                cv2.line(frame, pt, (pt[0]+dy[0], pt[1]+dy[1]), (0,255,255), 6)

            # ── Status overlay ────────────────────────────────────────
            cv2.rectangle(frame, (5,5), (700,260), (0,0,0), -1)

            emp_text  = "Employee Status: PRESENT" if employee_detected else "Employee Status: ABSENT"
            emp_color = (0,255,0) if employee_detected else (0,0,255)
            cv2.putText(frame, emp_text, (15,35), cv2.FONT_HERSHEY_COMPLEX, 0.7, emp_color, 2)

            if not employee_detected and self.employee_absence_start is not None:
                dur = time.time() - self.employee_absence_start
                cv2.putText(frame, f"Employee Absence: {dur:.1f}s",
                            (15,65), cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,0,255), 2)

            vis_count = len(people_outside_desk)
            vis_text  = f"Visitor Status: {vis_count} PRESENT" if visitor_detected else "Visitor Status: NONE"
            vis_color = (0,165,255) if visitor_detected else (128,128,128)
            cv2.putText(frame, vis_text, (15,100), cv2.FONT_HERSHEY_COMPLEX, 0.7, vis_color, 2)

            cv2.putText(frame, f"In Desk: {len(people_in_desk)} | Outside: {len(people_outside_desk)}",
                        (15,135), cv2.FONT_HERSHEY_COMPLEX, 0.7, (255,255,0), 2)

            if people_in_desk:
                names_in = ", ".join(people_in_desk[:3])
                if len(people_in_desk) > 3:
                    names_in += f" +{len(people_in_desk)-3}"
                cv2.putText(frame, f"Inside: {names_in}", (15,165),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,255,0), 2)

            if people_outside_desk:
                names_out = ", ".join(people_outside_desk[:3])
                if len(people_outside_desk) > 3:
                    names_out += f" +{len(people_outside_desk)-3}"
                cv2.putText(frame, f"Outside: {names_out}", (15,195),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,165,255), 2)

            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {ts}", (15,225), cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,255,255), 2)
            cv2.putText(frame, "Mode: SIMPLE | Press 'q' to quit", (15,250),
                        cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,200,0), 2)

            cv2.imshow('Employee Desk Tracking - Simple Mode', frame)
            frame_count += 1
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved → {self.log_file}  |  {self.absence_log_file}")

    # ──────────────────────────────────────────────────────────────────
    # FULL MODE
    # ──────────────────────────────────────────────────────────────────
    def _run_full_mode(self, cap):
        last_cleanup = time.time()
        frame_count  = 0
        log_interval = 30

        cv2.namedWindow('Employee Desk Tracking - Full Mode', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking - Full Mode', self.display_width, self.display_height)

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            h, w = frame.shape[:2]

            if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                self.desk_mask = np.zeros((h, w), np.uint8)
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(self.desk_mask, self.DESK_ROI_PT1, self.DESK_ROI_PT2, 255, -1)

            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416,416), swapRB=True, crop=False)
            self.net.setInput(blob)
            outputs = self.net.forward(self.output_layers)

            boxes, confidences = [], []
            for output in outputs:
                for detection in output:
                    scores   = detection[5:]
                    class_id = np.argmax(scores)
                    conf     = scores[class_id]
                    if conf > 0.5 and self.classes[class_id] == 'person':
                        cx = int(detection[0] * w)
                        cy = int(detection[1] * h)
                        bw = int(detection[2] * w)
                        bh = int(detection[3] * h)
                        boxes.append([int(cx-bw/2), int(cy-bh/2), bw, bh])
                        confidences.append(float(conf))

            indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, self.nms_threshold)

            current_in_desk  = {}
            current_outside  = {}
            employee_detected = False
            visitor_detected  = False
            seen_names        = set()
            det_list = indices.flatten() if len(indices) > 0 else []

            for i in det_list:
                x1, y1, bw, bh = boxes[i]
                x2, y2 = x1+bw, y1+bh
                cx, cy = (x1+x2)//2, (y1+y2)//2

                name, EmployeeID, email = self.get_name_from_centroid(cx, cy, frame)
                seen_names.add(name)

                if name not in self.trackers:
                    self.trackers[name] = {'centroids':[], 'last_seen':0,
                                           'EmployeeID': EmployeeID, 'email': email}
                self.trackers[name].update({'bbox':(x1,y1,x2,y2), 'last_seen':time.time(),
                                            'EmployeeID': EmployeeID, 'email': email})
                self.trackers[name]['centroids'].append((cx, cy))
                if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                    self.trackers[name]['centroids'].pop(0)

                # Desk overlap
                overlap_ratio = 0
                if x2 > x1 and y2 > y1:
                    ys = slice(max(0,y1), min(h,y2))
                    xs = slice(max(0,x1), min(w,x2))
                    roi = self.desk_mask[ys, xs]
                    if roi.size > 0:
                        overlap_ratio = np.sum(roi > 0) / roi.size

                location = 'Inside Desk' if overlap_ratio > 0.4 else 'Outside Desk'

                if 'Inside' in location:
                    current_in_desk[name] = (EmployeeID, email)
                    if name != "People":
                        employee_detected = True
                        self.last_employee_present = time.time()
                        if self.employee_absence_start is not None:
                            self.employee_absence_start = None
                else:
                    current_outside[name] = (EmployeeID, email)
                    visitor_detected = True
                    self.last_visitor_present = time.time()
                    if self.visitor_absence_start is not None:
                        self.visitor_absence_start = None

                # ── FILE 1: log on status change ──────────────────────
                if name not in self.person_status:
                    self.person_status[name] = location
                    self.log_detection(name, EmployeeID, f"Detected - {location}")
                elif self.person_status[name] != location:
                    self.person_status[name] = location
                    self.log_detection(name, EmployeeID, location)
                # ─────────────────────────────────────────────────────

                # ── FILE 2: mark present ──────────────────────────────
                self.mark_person_present(name, EmployeeID)
                # ─────────────────────────────────────────────────────

                # Draw
                color = (0,255,0) if 'Inside' in location else (0,0,255)
                cv2.rectangle(frame, (x1,y1), (x2,y2), color, 3)
                role  = "Employee" if 'Inside' in location else "Visitor"
                label = f"{name} - {role} ({location.split()[0]})"
                lsz   = cv2.getTextSize(label, cv2.FONT_HERSHEY_COMPLEX, 0.8, 2)[0]
                cv2.rectangle(frame, (x1,y1-45), (x1+lsz[0]+15, y1), color, -1)
                cv2.putText(frame, label, (x1+8, y1-15),
                            cv2.FONT_HERSHEY_COMPLEX, 0.8, (255,255,255), 2)
                id_display = EmployeeID if EmployeeID and EmployeeID != "N/A" else \
                             ("Employee" if 'Inside' in location else "Visitor")
                cv2.putText(frame, f"{id_display} | Conf: {confidences[i]:.2f}",
                            (x1, y2+25), cv2.FONT_HERSHEY_COMPLEX, 0.6, color, 2)

            # ── FILE 2: mark absent for people not seen this frame ────
            for tracked_name in list(self.person_status.keys()):
                if tracked_name not in seen_names:
                    self.mark_person_absent(tracked_name)
            # ─────────────────────────────────────────────────────────

            if not employee_detected and self.employee_absence_start is None:
                self.employee_absence_start = time.time()
            if not visitor_detected and self.visitor_absence_start is None:
                self.visitor_absence_start = time.time()

            # Cleanup disappeared trackers
            if time.time() - last_cleanup > 1.0:
                to_remove = [n for n, d in self.trackers.items()
                             if time.time() - d['last_seen'] > self.DISAPPEARANCE_TIME]
                for name in to_remove:
                    if name in self.person_status:
                        eid   = self.trackers[name].get('EmployeeID', 'N/A')
                        self.log_detection(name, eid, "Disappeared")
                        del self.person_status[name]
                    del self.trackers[name]
                last_cleanup = time.time()

            # ── Draw desk ROI ─────────────────────────────────────────
            if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                cv2.rectangle(frame, self.DESK_ROI_PT1, self.DESK_ROI_PT2, (255,255,0), 4)
                ms = 40
                p1, p2 = self.DESK_ROI_PT1, self.DESK_ROI_PT2
                for pt, dx, dy in [
                    (p1,           (ms,0),  (0,ms)),
                    ((p2[0],p1[1]),(-ms,0), (0,ms)),
                    ((p1[0],p2[1]),(ms,0),  (0,-ms)),
                    (p2,           (-ms,0), (0,-ms)),
                ]:
                    cv2.line(frame, pt, (pt[0]+dx[0],pt[1]+dx[1]), (0,255,255), 6)
                    cv2.line(frame, pt, (pt[0]+dy[0],pt[1]+dy[1]), (0,255,255), 6)

            # ── Status overlay ────────────────────────────────────────
            cv2.rectangle(frame, (5,5), (750,280), (0,0,0), -1)

            emp_text  = "Employee Status: PRESENT" if employee_detected else "Employee Status: ABSENT"
            emp_color = (0,255,0) if employee_detected else (0,0,255)
            cv2.putText(frame, emp_text, (15,35), cv2.FONT_HERSHEY_COMPLEX, 0.7, emp_color, 2)

            if not employee_detected and self.employee_absence_start is not None:
                dur = time.time() - self.employee_absence_start
                cv2.putText(frame, f"Employee Absence: {dur:.1f}s",
                            (15,65), cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,0,255), 2)

            vis_count = len(current_outside)
            vis_text  = f"Visitor Status: {vis_count} PRESENT" if visitor_detected else "Visitor Status: NONE"
            vis_color = (0,165,255) if visitor_detected else (128,128,128)
            cv2.putText(frame, vis_text, (15,100), cv2.FONT_HERSHEY_COMPLEX, 0.7, vis_color, 2)

            cv2.putText(frame,
                        f"Inside: {len(current_in_desk)} | Outside: {len(current_outside)} | Total: {len(self.trackers)}",
                        (15,135), cv2.FONT_HERSHEY_COMPLEX, 0.7, (255,255,0), 2)

            if current_in_desk:
                nin = ", ".join(list(current_in_desk.keys())[:3])
                if len(current_in_desk) > 3: nin += f" +{len(current_in_desk)-3}"
                cv2.putText(frame, f"Inside Desk: {nin}", (15,165),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,255,0), 2)
            if current_outside:
                nout = ", ".join(list(current_outside.keys())[:3])
                if len(current_outside) > 3: nout += f" +{len(current_outside)-3}"
                cv2.putText(frame, f"Outside Desk: {nout}", (15,195),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,165,255), 2)

            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {ts}", (15,225), cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,255,255), 2)
            cv2.putText(frame, "Mode: FULL | Press 'q' to quit", (15,250),
                        cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,200,0), 2)

            cv2.imshow('Employee Desk Tracking - Full Mode (Face Recognition)', frame)
            frame_count += 1
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved → {self.log_file}  |  {self.absence_log_file}")


# ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import sys

    tracker = DeskTracker()

    mode  = "full"
    debug = False
    for arg in sys.argv[1:]:
        if arg.lower() in ['simple', 's']:   mode  = "simple"
        elif arg.lower() in ['full', 'f']:   mode  = "full"
        elif arg.lower() in ['debug','d','--debug']: debug = True

    if debug:
        tracker.debug_mode = True
        print("DEBUG MODE ENABLED - Face search regions will be visualized")

    print("\n" + "="*60)
    print(f"Starting Desk Tracker in {mode.upper()} mode")
    print("="*60)
    print("\nLog files created in  ./Logs/Log_<date>/")
    print(f"  Detection log : Logs/Log_<date>/detection_log_<date>.csv")
    print(f"    Columns: Date & Time | EmployeeID | Name | Detection Status")
    print(f"  Absence log   : Logs/Log_<date>/absence_log_<date>.csv")
    print(f"    Columns: Date & Time | EmployeeID | Name | Absence Duration (s)")
    print(f"    (only records absences longer than 3 seconds, employees only)")
    print("\nPress 'q' to quit\n" + "="*60 + "\n")

    tracker.run(use_simple_mode=(mode == "simple"))

In [ ]:
#Remove Simple Mode

In [ ]:
#Only Full Mode

In [ ]:
import cv2
import numpy as np
import time
from datetime import datetime
import csv
import os
from collections import defaultdict
import pickle
from sklearn.neighbors import KNeighborsClassifier

class DeskTracker:
    def __init__(self):
        # Load YOLOv4-tiny model 
        cfg_path = 'Yolo_model/yolov4-tiny.cfg'
        names_path = 'Yolo_model/coco.names'
        weights_path = 'Yolo_model/yolov4-tiny.weights'

        # Load names
        with open(names_path, 'r') as f:
            self.classes = [line.strip() for line in f.readlines()]

        # Load model
        self.net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

        # Get output layer names
        layer_names = self.net.getLayerNames()
        self.output_layers = [layer_names[i - 1] for i in self.net.getUnconnectedOutLayers()]

        # Detection parameters
        self.confidence_threshold = 0.3
        self.nms_threshold = 0.4

        # Face recognition parameters
        self.face_recognition_threshold = 20
        self.face_min_size = (30, 30)

        # Load face recognition data
        try:
            with open('data/Organizations.pkl', 'rb') as f:
                self.ORGANIZATIONS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/EmployeeIDs.pkl', 'rb') as f:
                self.EmployeeIDs = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f)
                self.FACES = self.FACES / 255.0
                print(f"FACES shape: {self.FACES.shape}")

            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.EmployeeIDs)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []
            self.EmployeeIDs = []
            self.ORGANIZATIONS = []
            self.EMAILS = []
            self.face_recognition_enabled = False

        # Load face cascade
        self.face_cascade = cv2.CascadeClassifier('data/haarcascade_frontalface_default.xml')

        # Desk ROI
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask = None

        # Tracking
        self.trackers = {}
        self.person_status = {}
        self.MAX_CENTROIDS = 10
        self.DISAPPEARANCE_TIME = 2.0

        # Employee/visitor status
        self.employee_absence_start = None
        self.visitor_absence_start = None
        self.last_employee_present = None
        self.last_visitor_present = None

        # ── LOGGING SETUP ──────────────────────────────────────────────
        # Folder structure:  Logs/Log_DD-MM-YYYY/
        date = datetime.now().strftime("%d-%m-%Y")
        os.makedirs('Logs', exist_ok=True)                     # create Logs/ first
        log_folder = os.path.join('Logs', f'Log_{date}')
        os.makedirs(log_folder, exist_ok=True)                 # create Logs/Log_DD-MM-YYYY/
        print(f"[LOG FOLDER] {os.path.abspath(log_folder)}")

        # LOG FILE 1 – Detection events (Inside / Outside desk area)
        # Columns: Date & Time | EmployeeID | Name | Detection Status
        self.log_file = os.path.join(log_folder, f'detection_log_{date}.csv')
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Date & Time', 'EmployeeID', 'Name', 'Detection Status'])

        # LOG FILE 2 – Absence log (only when absence > 3 seconds)
        # Columns: Date & Time | EmployeeID | Name | Absence Duration (s)
        self.absence_log_file = os.path.join(log_folder, f'absence_log_{date}.csv')
        if not os.path.exists(self.absence_log_file):
            with open(self.absence_log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Date & Time', 'EmployeeID', 'Name', 'Absence Duration (s)'])

        # Tracks per-person absence start time  {name: start_time}
        self.person_absence_start = {}
        # Tracks last logged absence end time so we don't double-log  {name: end_time}
        self.person_absence_logged = {}
        # ───────────────────────────────────────────────────────────────

        self.monitor_area = None
        self.source_type = "camera"
        self.display_width = 1280
        self.display_height = 720
        self.debug_mode = False

    # ──────────────────────────────────────────────────────────────────
    # LOGGING HELPERS
    # ──────────────────────────────────────────────────────────────────
    def log_event(self, message):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def log_detection(self, name, employee_id, status):
        """Log FILE 1 – person detected inside or outside desk area."""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, employee_id, name, status])
        self.log_event(f"[DETECTION] {name} ({employee_id}) → {status}")

    def log_absence(self, name, employee_id, duration_seconds):
        """Log FILE 2 – absence only when duration > 3 seconds."""
        if duration_seconds <= 3:
            return
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.absence_log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, employee_id, name, f"{duration_seconds:.1f}"])
        self.log_event(f"[ABSENCE]   {name} ({employee_id}) absent for {duration_seconds:.1f}s")

    # ──────────────────────────────────────────────────────────────────
    # PER-PERSON ABSENCE TRACKING
    # ──────────────────────────────────────────────────────────────────
    def mark_person_present(self, name, employee_id):
        """Called every frame a person is detected. Closes any open absence window."""
        if name in self.person_absence_start:
            duration = time.time() - self.person_absence_start.pop(name)
            # Only log absence for recognised employees, not unidentified visitors
            if name != "People":
                self.log_absence(name, employee_id, duration)

    def mark_person_absent(self, name):
        """Called when a person disappears. Opens an absence window."""
        if name not in self.person_absence_start:
            self.person_absence_start[name] = time.time()

    # ──────────────────────────────────────────────────────────────────
    # DESK AREA DETECTION
    # ──────────────────────────────────────────────────────────────────
    def detect_desk_area(self, cap):
        self.log_event("Detecting desk area...")
        desk_candidates = []
        frame_count = 0
        frame = None

        while frame_count < 5:
            ret, frame = cap.read()
            if not ret or frame is None or frame.size == 0:
                time.sleep(0.05)
                continue
            height, width = frame.shape[:2]
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                break

            desks = []
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    desk_related_classes = [56, 60, 63, 64, 65]
                    if confidence > self.confidence_threshold and class_id in desk_related_classes:
                        center_x = int(obj_detection[0] * width)
                        center_y = int(obj_detection[1] * height)
                        w = int(obj_detection[2] * width)
                        h = int(obj_detection[3] * height)
                        x = int(center_x - w / 2)
                        y = int(center_y - h / 2)
                        desks.append([x, y, w, h, confidence, class_id])
            if desks:
                desk_candidates.extend(desks)
            frame_count += 1

        if len(desk_candidates) < 2:
            self.log_event("Not enough desk objects detected. Using default desk area.")
            if frame is not None:
                height, width = frame.shape[:2]
            else:
                height, width = 480, 640
            min_x = int(width * 0.2)
            min_y = int(height * 0.4)
            max_x = int(width * 0.8)
            max_y = int(height * 0.9)
        else:
            x_coords     = [d[0] for d in desk_candidates]
            y_coords     = [d[1] for d in desk_candidates]
            max_x_coords = [d[0] + d[2] for d in desk_candidates]
            max_y_coords = [d[1] + d[3] for d in desk_candidates]
            min_x = max(0,     int(min(x_coords)     - 0.1 * width))
            min_y = max(0,     int(min(y_coords)     - 0.1 * height))
            max_x = min(width, int(max(max_x_coords) + 0.1 * width))
            max_y = min(height,int(max(max_y_coords) + 0.1 * height))
            self.log_event(f"Desk area detected: ({min_x}, {min_y}) to ({max_x}, {max_y})")

        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        self.monitor_area = (min_x, min_y, max_x, max_y)
        return (min_x, min_y, max_x, max_y)

    # ──────────────────────────────────────────────────────────────────
    # FACE RECOGNITION
    # ──────────────────────────────────────────────────────────────────
    def get_name_from_face(self, face_region):
        if not self.face_recognition_enabled or face_region.size == 0:
            return "People", "N/A", ""
        if face_region.shape[0] < 30 or face_region.shape[1] < 30:
            return "People", "N/A", ""

        gray_region = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        faces = []
        detection_params = [
            (1.1, 5, (20, 20)),
            (1.05, 4, (25, 25)),
            (1.2, 5, (30, 30)),
            (1.3, 4, (35, 35)),
            (1.15, 3, (25, 25)),
        ]
        for scale, neighbors, min_size in detection_params:
            detected = self.face_cascade.detectMultiScale(
                gray_region, scale, neighbors, minSize=min_size)
            if len(detected) > 0:
                faces = detected
                break

        Name = "People"
        EmployeeID = "N/A"
        Email = ""

        for (x, y, w_face, h_face) in faces:
            crop_img = face_region[y:y+h_face, x:x+w_face, :]
            if crop_img.size == 0:
                continue
            try:
                resized_img  = cv2.resize(crop_img, (75, 100))
                gray         = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
                resized_flat = gray.reshape(1, -1) / 255.0
                if resized_flat.shape[1] != self.FACES.shape[1]:
                    continue
                distances, _ = self.knn.kneighbors(resized_flat)
                if distances[0][0] <= self.face_recognition_threshold:
                    output = self.knn.predict(resized_flat)
                    emp_id = output[0]
                    if emp_id in self.EmployeeIDs:
                        index      = self.EmployeeIDs.index(emp_id)
                        Name       = self.NAMES[index]
                        EmployeeID = self.EmployeeIDs[index] if index < len(self.EmployeeIDs) else "N/A"
                        Email      = self.EMAILS[index]      if index < len(self.EMAILS)      else ""
                        break
            except Exception:
                continue

        return Name, EmployeeID, Email

    def get_name_from_centroid(self, cx, cy, frame):
        h, w = frame.shape[:2]
        search_regions = [
            (max(0,cx-180), max(0,cy-280), min(w,cx+180), min(h,cy+50)),
            (max(0,cx-220), max(0,cy-320), min(w,cx+220), min(h,cy+80)),
            (max(0,cx-150), max(0,cy-350), min(w,cx+150), min(h,cy)),
            (max(0,cx-200), max(0,cy-200), min(w,cx+200), min(h,cy+150)),
        ]
        for idx, (rx1, ry1, rx2, ry2) in enumerate(search_regions):
            if self.debug_mode:
                color = [(255,0,255),(0,255,255),(255,255,0),(255,128,0)][idx]
                cv2.rectangle(frame, (rx1,ry1), (rx2,ry2), color, 2)
                cv2.putText(frame, f"Search {idx+1}", (rx1, ry1-5),
                            cv2.FONT_HERSHEY_COMPLEX, 0.4, color, 1)
            face_region = frame[ry1:ry2, rx1:rx2]
            name, EmployeeID, email = self.get_name_from_face(face_region)
            if name != "People":
                return name, EmployeeID, email
        return "People", "N/A", ""

    # ──────────────────────────────────────────────────────────────────
    # MAIN RUN
    # ──────────────────────────────────────────────────────────────────
    def run(self):
        """Open camera, detect desk area, then start tracking."""
        # Open default camera (index 0)
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            self.log_event("Error: Cannot open default camera.")
            return
        self.log_event("Default camera opened.")

        # Set camera resolution
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
        cap.set(cv2.CAP_PROP_FPS, 30)

        actual_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        actual_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.log_event(f"Resolution: {actual_width}x{actual_height}")

        # Warm up camera — wait until we get valid frames
        self.log_event("Warming up camera...")
        warm_up_count = 0
        max_warmup_attempts = 30
        while warm_up_count < max_warmup_attempts:
            ret, frame = cap.read()
            if ret and frame is not None and frame.size > 0:
                warm_up_count += 1
            time.sleep(0.05)
        self.log_event(f"Camera ready after {max_warmup_attempts} frames.")

        # Detect desk area from next few frames
        self.detect_desk_area(cap)

        self.log_event("Starting tracker... Press 'q' to quit.")
        self._run_full_mode(cap)

    # ──────────────────────────────────────────────────────────────────
    # FULL MODE
    # ──────────────────────────────────────────────────────────────────
    def _run_full_mode(self, cap):
        last_cleanup = time.time()
        frame_count  = 0
        log_interval = 30

        cv2.namedWindow('Employee Desk Tracking', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking', self.display_width, self.display_height)

        while True:
            ret, frame = cap.read()
            if not ret or frame is None or frame.size == 0:
                time.sleep(0.05)
                continue

            h, w = frame.shape[:2]

            if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                self.desk_mask = np.zeros((h, w), np.uint8)
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(self.desk_mask, self.DESK_ROI_PT1, self.DESK_ROI_PT2, 255, -1)

            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416,416), swapRB=True, crop=False)
            self.net.setInput(blob)
            outputs = self.net.forward(self.output_layers)

            boxes, confidences = [], []
            for output in outputs:
                for detection in output:
                    scores   = detection[5:]
                    class_id = np.argmax(scores)
                    conf     = scores[class_id]
                    if conf > 0.5 and self.classes[class_id] == 'person':
                        cx = int(detection[0] * w)
                        cy = int(detection[1] * h)
                        bw = int(detection[2] * w)
                        bh = int(detection[3] * h)
                        boxes.append([int(cx-bw/2), int(cy-bh/2), bw, bh])
                        confidences.append(float(conf))

            indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, self.nms_threshold)

            current_in_desk  = {}
            current_outside  = {}
            employee_detected = False
            visitor_detected  = False
            seen_names        = set()
            det_list = indices.flatten() if len(indices) > 0 else []

            for i in det_list:
                x1, y1, bw, bh = boxes[i]
                x2, y2 = x1+bw, y1+bh
                cx, cy = (x1+x2)//2, (y1+y2)//2

                name, EmployeeID, email = self.get_name_from_centroid(cx, cy, frame)
                seen_names.add(name)

                if name not in self.trackers:
                    self.trackers[name] = {'centroids':[], 'last_seen':0,
                                           'EmployeeID': EmployeeID, 'email': email}
                self.trackers[name].update({'bbox':(x1,y1,x2,y2), 'last_seen':time.time(),
                                            'EmployeeID': EmployeeID, 'email': email})
                self.trackers[name]['centroids'].append((cx, cy))
                if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                    self.trackers[name]['centroids'].pop(0)

                # Desk overlap
                overlap_ratio = 0
                if x2 > x1 and y2 > y1:
                    ys = slice(max(0,y1), min(h,y2))
                    xs = slice(max(0,x1), min(w,x2))
                    roi = self.desk_mask[ys, xs]
                    if roi.size > 0:
                        overlap_ratio = np.sum(roi > 0) / roi.size

                location = 'Inside Desk' if overlap_ratio > 0.4 else 'Outside Desk'

                if 'Inside' in location:
                    current_in_desk[name] = (EmployeeID, email)
                    if name != "People":
                        employee_detected = True
                        self.last_employee_present = time.time()
                        if self.employee_absence_start is not None:
                            self.employee_absence_start = None
                else:
                    current_outside[name] = (EmployeeID, email)
                    visitor_detected = True
                    self.last_visitor_present = time.time()
                    if self.visitor_absence_start is not None:
                        self.visitor_absence_start = None

                # ── FILE 1: log on status change ──────────────────────
                if name not in self.person_status:
                    self.person_status[name] = location
                    self.log_detection(name, EmployeeID, f"Detected - {location}")
                elif self.person_status[name] != location:
                    self.person_status[name] = location
                    self.log_detection(name, EmployeeID, location)
                # ─────────────────────────────────────────────────────

                # ── FILE 2: mark present ──────────────────────────────
                self.mark_person_present(name, EmployeeID)
                # ─────────────────────────────────────────────────────

                # Draw
                color = (0,255,0) if 'Inside' in location else (0,0,255)
                cv2.rectangle(frame, (x1,y1), (x2,y2), color, 3)
                role  = "Employee" if 'Inside' in location else "Visitor"
                label = f"{name} - {role} ({location.split()[0]})"
                lsz   = cv2.getTextSize(label, cv2.FONT_HERSHEY_COMPLEX, 0.8, 2)[0]
                cv2.rectangle(frame, (x1,y1-45), (x1+lsz[0]+15, y1), color, -1)
                cv2.putText(frame, label, (x1+8, y1-15),
                            cv2.FONT_HERSHEY_COMPLEX, 0.8, (255,255,255), 2)
                id_display = EmployeeID if EmployeeID and EmployeeID != "N/A" else \
                             ("Employee" if 'Inside' in location else "Visitor")
                cv2.putText(frame, f"{id_display} | Conf: {confidences[i]:.2f}",
                            (x1, y2+25), cv2.FONT_HERSHEY_COMPLEX, 0.6, color, 2)

            # ── FILE 2: mark absent for people not seen this frame ────
            for tracked_name in list(self.person_status.keys()):
                if tracked_name not in seen_names:
                    self.mark_person_absent(tracked_name)
            # ─────────────────────────────────────────────────────────

            if not employee_detected and self.employee_absence_start is None:
                self.employee_absence_start = time.time()
            if not visitor_detected and self.visitor_absence_start is None:
                self.visitor_absence_start = time.time()

            # Cleanup disappeared trackers
            if time.time() - last_cleanup > 1.0:
                to_remove = [n for n, d in self.trackers.items()
                             if time.time() - d['last_seen'] > self.DISAPPEARANCE_TIME]
                for name in to_remove:
                    if name in self.person_status:
                        eid   = self.trackers[name].get('EmployeeID', 'N/A')
                        self.log_detection(name, eid, "Disappeared")
                        del self.person_status[name]
                    del self.trackers[name]
                last_cleanup = time.time()

            # ── Draw desk ROI ─────────────────────────────────────────
            if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                cv2.rectangle(frame, self.DESK_ROI_PT1, self.DESK_ROI_PT2, (255,255,0), 4)
                ms = 40
                p1, p2 = self.DESK_ROI_PT1, self.DESK_ROI_PT2
                for pt, dx, dy in [
                    (p1,           (ms,0),  (0,ms)),
                    ((p2[0],p1[1]),(-ms,0), (0,ms)),
                    ((p1[0],p2[1]),(ms,0),  (0,-ms)),
                    (p2,           (-ms,0), (0,-ms)),
                ]:
                    cv2.line(frame, pt, (pt[0]+dx[0],pt[1]+dx[1]), (0,255,255), 6)
                    cv2.line(frame, pt, (pt[0]+dy[0],pt[1]+dy[1]), (0,255,255), 6)

            # ── Status overlay ────────────────────────────────────────
            

            emp_text  = "Employee Status: PRESENT" if employee_detected else "Employee Status: ABSENT"
            emp_color = (0,255,0) if employee_detected else (0,0,255)
            cv2.putText(frame, emp_text, (15,35), cv2.FONT_HERSHEY_COMPLEX, 0.7, emp_color, 2)

            if not employee_detected and self.employee_absence_start is not None:
                dur = time.time() - self.employee_absence_start
                cv2.putText(frame, f"Employee Absence: {dur:.1f}s",
                            (15,65), cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,0,255), 2)

            vis_count = len(current_outside)
            vis_text  = f"Visitor Status: {vis_count} PRESENT" if visitor_detected else "Visitor Status: NONE"
            vis_color = (0,165,255) if visitor_detected else (128,128,128)
            cv2.putText(frame, vis_text, (15,100), cv2.FONT_HERSHEY_COMPLEX, 0.7, vis_color, 2)

            cv2.putText(frame,
                        f"Inside: {len(current_in_desk)} | Outside: {len(current_outside)} | Total: {len(self.trackers)}",
                        (15,135), cv2.FONT_HERSHEY_COMPLEX, 0.7, (255,255,0), 2)

            if current_in_desk:
                nin = ", ".join(list(current_in_desk.keys())[:3])
                if len(current_in_desk) > 3: nin += f" +{len(current_in_desk)-3}"
                cv2.putText(frame, f"Inside Desk: {nin}", (15,165),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,255,0), 2)
            if current_outside:
                nout = ", ".join(list(current_outside.keys())[:3])
                if len(current_outside) > 3: nout += f" +{len(current_outside)-3}"
                cv2.putText(frame, f"Outside Desk: {nout}", (15,195),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,165,255), 2)

            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {ts}", (15,225), cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,255,255), 2)
            cv2.putText(frame, "Press q to quit", (15,250),
                        cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,200,0), 2)

            cv2.imshow('Employee Desk Tracking', frame)
            frame_count += 1
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved → {self.log_file}  |  {self.absence_log_file}")


# ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import sys

    tracker = DeskTracker()

    debug = False
    for arg in sys.argv[1:]:
        if arg.lower() in ['debug', 'd', '--debug']:
            debug = True

    if debug:
        tracker.debug_mode = True
        print("DEBUG MODE ENABLED - Face search regions will be visualized")

    print("\n" + "="*60)
    print("Starting Employee Desk Tracker")
    print("="*60)
    print("\nLog files saved in  ./Logs/Log_<date>/")
    print("  detection_log_<date>.csv  →  Date & Time | EmployeeID | Name | Detection Status")
    print("  absence_log_<date>.csv    →  Date & Time | EmployeeID | Name | Absence Duration (s)")
    print("  (absence log: employees only, absences > 3 seconds)")
    print("\nUsage:")
    print("  python desk_tracking.py         # Normal run")
    print("  python desk_tracking.py debug   # Show face search regions")
    print("\nPress 'q' to quit\n" + "="*60 + "\n")

    tracker.run()

In [ ]:
#Simple+Full

In [ ]:
import cv2
import numpy as np
import time
from datetime import datetime
import csv
import os
from collections import defaultdict
import pickle
from sklearn.neighbors import KNeighborsClassifier

class DeskTracker:
    def __init__(self):
        # Load YOLOv4-tiny model 
        cfg_path = 'Yolo_model/yolov4-tiny.cfg'
        names_path = 'Yolo_model/coco.names'
        weights_path = 'Yolo_model/yolov4-tiny.weights'

        # Load names
        with open(names_path, 'r') as f:
            self.classes = [line.strip() for line in f.readlines()]

        # Load model
        self.net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

        # Get output layer names
        layer_names = self.net.getLayerNames()
        self.output_layers = [layer_names[i - 1] for i in self.net.getUnconnectedOutLayers()]

        # Detection parameters
        self.confidence_threshold = 0.3
        self.nms_threshold = 0.4

        # Face recognition parameters
        self.face_recognition_threshold = 20
        self.face_min_size = (30, 30)

        # Load face recognition data
        try:
            with open('data/Organizations.pkl', 'rb') as f:
                self.ORGANIZATIONS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/EmployeeIDs.pkl', 'rb') as f:
                self.EmployeeIDs = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f)
                self.FACES = self.FACES / 255.0
                print(f"FACES shape: {self.FACES.shape}")

            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.EmployeeIDs)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []
            self.EmployeeIDs = []
            self.ORGANIZATIONS = []
            self.EMAILS = []
            self.face_recognition_enabled = False

        # Load face cascade
        self.face_cascade = cv2.CascadeClassifier('data/haarcascade_frontalface_default.xml')

        # Desk ROI
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask = None

        # Tracking
        self.trackers = {}
        self.person_status = {}
        self.MAX_CENTROIDS = 10
        self.DISAPPEARANCE_TIME = 2.0

        # Employee/visitor status
        self.employee_absence_start = None
        self.visitor_absence_start = None
        self.last_employee_present = None
        self.last_visitor_present = None

        # ── LOGGING SETUP ──────────────────────────────────────────────
        # Ensure dated subfolder exists:  Logs/Log_DD-MM-YYYY/
        date = datetime.now().strftime("%d-%m-%Y")
        log_folder = os.path.join('Logs', f'Log_{date}')
        os.makedirs(log_folder, exist_ok=True)

        # LOG FILE 1 – Detection events (Inside / Outside desk area)
        # Columns: Date & Time | EmployeeID | Name | Detection Status
        self.log_file = os.path.join(log_folder, f'detection_log_{date}.csv')
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Date & Time', 'EmployeeID', 'Name', 'Detection Status'])

        # LOG FILE 2 – Absence log (only when absence > 3 seconds)
        # Columns: Date & Time | EmployeeID | Name | Absence Duration (s)
        self.absence_log_file = os.path.join(log_folder, f'absence_log_{date}.csv')
        if not os.path.exists(self.absence_log_file):
            with open(self.absence_log_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Date & Time', 'EmployeeID', 'Name', 'Absence Duration (s)'])

        # Tracks per-person absence start time  {name: start_time}
        self.person_absence_start = {}
        # Tracks last logged absence end time so we don't double-log  {name: end_time}
        self.person_absence_logged = {}
        # ───────────────────────────────────────────────────────────────

        self.monitor_area = None
        self.source_type = "camera"
        self.display_width = 1280
        self.display_height = 720
        self.debug_mode = False

    # ──────────────────────────────────────────────────────────────────
    # LOGGING HELPERS
    # ──────────────────────────────────────────────────────────────────
    def log_event(self, message):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def log_detection(self, name, employee_id, status):
        """Log FILE 1 – person detected inside or outside desk area."""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, employee_id, name, status])
        self.log_event(f"[DETECTION] {name} ({employee_id}) → {status}")

    def log_absence(self, name, employee_id, duration_seconds):
        """Log FILE 2 – absence only when duration > 3 seconds."""
        if duration_seconds <= 3:
            return
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.absence_log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, employee_id, name, f"{duration_seconds:.1f}"])
        self.log_event(f"[ABSENCE]   {name} ({employee_id}) absent for {duration_seconds:.1f}s")

    # ──────────────────────────────────────────────────────────────────
    # PER-PERSON ABSENCE TRACKING
    # ──────────────────────────────────────────────────────────────────
    def mark_person_present(self, name, employee_id):
        """Called every frame a person is detected. Closes any open absence window."""
        if name in self.person_absence_start:
            duration = time.time() - self.person_absence_start.pop(name)
            # Only log absence for recognised employees, not unidentified visitors
            if name != "People":
                self.log_absence(name, employee_id, duration)

    def mark_person_absent(self, name):
        """Called when a person disappears. Opens an absence window."""
        if name not in self.person_absence_start:
            self.person_absence_start[name] = time.time()

    # ──────────────────────────────────────────────────────────────────
    # DESK AREA DETECTION
    # ──────────────────────────────────────────────────────────────────
    def detect_desk_area(self, cap):
        self.log_event("Detecting desk area...")
        desk_candidates = []
        frame_count = 0
        frame = None

        while frame_count < 5:
            ret, frame = cap.read()
            if not ret:
                break
            height, width = frame.shape[:2]
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
            self.net.setInput(blob)
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Error during detection: {e}")
                break

            desks = []
            for detection in detections:
                for obj_detection in detection:
                    scores = obj_detection[5:]
                    class_id = np.argmax(scores)
                    confidence = scores[class_id]
                    desk_related_classes = [56, 60, 63, 64, 65]
                    if confidence > self.confidence_threshold and class_id in desk_related_classes:
                        center_x = int(obj_detection[0] * width)
                        center_y = int(obj_detection[1] * height)
                        w = int(obj_detection[2] * width)
                        h = int(obj_detection[3] * height)
                        x = int(center_x - w / 2)
                        y = int(center_y - h / 2)
                        desks.append([x, y, w, h, confidence, class_id])
            if desks:
                desk_candidates.extend(desks)
            frame_count += 1

        if len(desk_candidates) < 2:
            self.log_event("Not enough desk objects detected. Using default desk area.")
            if frame is not None:
                height, width = frame.shape[:2]
            else:
                height, width = 480, 640
            min_x = int(width * 0.2)
            min_y = int(height * 0.4)
            max_x = int(width * 0.8)
            max_y = int(height * 0.9)
        else:
            x_coords     = [d[0] for d in desk_candidates]
            y_coords     = [d[1] for d in desk_candidates]
            max_x_coords = [d[0] + d[2] for d in desk_candidates]
            max_y_coords = [d[1] + d[3] for d in desk_candidates]
            min_x = max(0,     int(min(x_coords)     - 0.1 * width))
            min_y = max(0,     int(min(y_coords)     - 0.1 * height))
            max_x = min(width, int(max(max_x_coords) + 0.1 * width))
            max_y = min(height,int(max(max_y_coords) + 0.1 * height))
            self.log_event(f"Desk area detected: ({min_x}, {min_y}) to ({max_x}, {max_y})")

        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        self.monitor_area = (min_x, min_y, max_x, max_y)
        return (min_x, min_y, max_x, max_y)

    # ──────────────────────────────────────────────────────────────────
    # FACE RECOGNITION
    # ──────────────────────────────────────────────────────────────────
    def get_name_from_face(self, face_region):
        if not self.face_recognition_enabled or face_region.size == 0:
            return "People", "N/A", ""
        if face_region.shape[0] < 30 or face_region.shape[1] < 30:
            return "People", "N/A", ""

        gray_region = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        faces = []
        detection_params = [
            (1.1, 5, (20, 20)),
            (1.05, 4, (25, 25)),
            (1.2, 5, (30, 30)),
            (1.3, 4, (35, 35)),
            (1.15, 3, (25, 25)),
        ]
        for scale, neighbors, min_size in detection_params:
            detected = self.face_cascade.detectMultiScale(
                gray_region, scale, neighbors, minSize=min_size)
            if len(detected) > 0:
                faces = detected
                break

        Name = "People"
        EmployeeID = "N/A"
        Email = ""

        for (x, y, w_face, h_face) in faces:
            crop_img = face_region[y:y+h_face, x:x+w_face, :]
            if crop_img.size == 0:
                continue
            try:
                resized_img  = cv2.resize(crop_img, (75, 100))
                gray         = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
                resized_flat = gray.reshape(1, -1) / 255.0
                if resized_flat.shape[1] != self.FACES.shape[1]:
                    continue
                distances, _ = self.knn.kneighbors(resized_flat)
                if distances[0][0] <= self.face_recognition_threshold:
                    output = self.knn.predict(resized_flat)
                    emp_id = output[0]
                    if emp_id in self.EmployeeIDs:
                        index      = self.EmployeeIDs.index(emp_id)
                        Name       = self.NAMES[index]
                        EmployeeID = self.EmployeeIDs[index] if index < len(self.EmployeeIDs) else "N/A"
                        Email      = self.EMAILS[index]      if index < len(self.EMAILS)      else ""
                        break
            except Exception:
                continue

        return Name, EmployeeID, Email

    def get_name_from_centroid(self, cx, cy, frame):
        h, w = frame.shape[:2]
        search_regions = [
            (max(0,cx-180), max(0,cy-280), min(w,cx+180), min(h,cy+50)),
            (max(0,cx-220), max(0,cy-320), min(w,cx+220), min(h,cy+80)),
            (max(0,cx-150), max(0,cy-350), min(w,cx+150), min(h,cy)),
            (max(0,cx-200), max(0,cy-200), min(w,cx+200), min(h,cy+150)),
        ]
        for idx, (rx1, ry1, rx2, ry2) in enumerate(search_regions):
            if self.debug_mode:
                color = [(255,0,255),(0,255,255),(255,255,0),(255,128,0)][idx]
                cv2.rectangle(frame, (rx1,ry1), (rx2,ry2), color, 2)
                cv2.putText(frame, f"Search {idx+1}", (rx1, ry1-5),
                            cv2.FONT_HERSHEY_COMPLEX, 0.4, color, 1)
            face_region = frame[ry1:ry2, rx1:rx2]
            name, EmployeeID, email = self.get_name_from_face(face_region)
            if name != "People":
                return name, EmployeeID, email
        return "People", "N/A", ""

    # ──────────────────────────────────────────────────────────────────
    # MAIN RUN
    # ──────────────────────────────────────────────────────────────────
    def run(self, use_simple_mode=False):
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            self.log_event("Error: Cannot open camera")
            return
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
        cap.set(cv2.CAP_PROP_FPS, 30)
        actual_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        actual_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.log_event(f"Camera resolution: {actual_width}x{actual_height}")
        self.detect_desk_area(cap)
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        if use_simple_mode:
            self.log_event("Running in SIMPLE mode")
            self._run_simple_mode(cap)
        else:
            self.log_event("Running in FULL mode")
            self._run_full_mode(cap)

    # ──────────────────────────────────────────────────────────────────
    # SIMPLE MODE
    # ──────────────────────────────────────────────────────────────────
    def _run_simple_mode(self, cap):
        frame_count  = 0
        log_interval = 30

        cv2.namedWindow('Employee Desk Tracking - Simple Mode', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking - Simple Mode', self.display_width, self.display_height)

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            h, w = frame.shape[:2]
            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416,416), swapRB=True, crop=False)
            self.net.setInput(blob)
            try:
                detections = self.net.forward(self.output_layers)
            except Exception as e:
                self.log_event(f"Detection error: {e}")
                continue

            boxes, confidences = [], []
            for detection in detections:
                for obj in detection:
                    scores   = obj[5:]
                    class_id = np.argmax(scores)
                    conf     = scores[class_id]
                    if conf > self.confidence_threshold and class_id == 0:
                        center_x   = int(obj[0] * w)
                        center_y   = int(obj[1] * h)
                        width_box  = int(obj[2] * w)
                        height_box = int(obj[3] * h)
                        x = int(center_x - width_box  / 2)
                        y = int(center_y - height_box / 2)
                        boxes.append([x, y, width_box, height_box])
                        confidences.append(float(conf))

            people_in_desk    = []
            people_outside_desk = []
            employee_detected = False
            visitor_detected  = False
            seen_names        = set()   # names seen this frame

            if boxes:
                indices = cv2.dnn.NMSBoxes(boxes, confidences, self.confidence_threshold, 0.4)
                if len(indices) > 0:
                    for i in indices.flatten():
                        x, y, width_box, height_box = boxes[i]
                        cx = x + width_box  // 2
                        cy = y + height_box // 2

                        name, EmployeeID, email = self.get_name_from_centroid(cx, cy, frame)
                        seen_names.add(name)

                        # Desk overlap
                        px1, py1 = x, y
                        px2, py2 = x + width_box, y + height_box
                        xi = max(self.monitor_area[0], px1)
                        yi = max(self.monitor_area[1], py1)
                        wi = min(self.monitor_area[2], px2) - xi
                        hi = min(self.monitor_area[3], py2) - yi
                        is_in_desk_area = False
                        if wi > 0 and hi > 0:
                            overlap = (wi * hi) / (width_box * height_box)
                            if overlap > 0.3:
                                is_in_desk_area = True

                        location = "Inside Desk" if is_in_desk_area else "Outside Desk"

                        # ── FILE 1: log on status change ──────────────────
                        if name not in self.person_status:
                            self.person_status[name] = location
                            self.log_detection(name, EmployeeID, f"Detected - {location}")
                        elif self.person_status[name] != location:
                            self.person_status[name] = location
                            self.log_detection(name, EmployeeID, location)
                        # ─────────────────────────────────────────────────

                        # Update employee / visitor flags
                        if is_in_desk_area:
                            people_in_desk.append(name)
                            if name != "People":
                                employee_detected = True
                                self.last_employee_present = time.time()
                                if self.employee_absence_start is not None:
                                    self.employee_absence_start = None
                            color = (0, 255, 0)
                            label = f"{name} - Employee (Inside)"
                        else:
                            people_outside_desk.append(name)
                            visitor_detected = True
                            self.last_visitor_present = time.time()
                            if self.visitor_absence_start is not None:
                                self.visitor_absence_start = None
                            color = (0, 0, 255)
                            label = f"{name} - Visitor (Outside)"

                        # ── FILE 2: mark present → closes absence window ──
                        self.mark_person_present(name, EmployeeID)
                        # ─────────────────────────────────────────────────

                        # Draw bounding box
                        cv2.rectangle(frame, (x, y), (x+width_box, y+height_box), color, 3)
                        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_COMPLEX, 0.8, 2)[0]
                        cv2.rectangle(frame, (x, y-40), (x+label_size[0]+15, y), color, -1)
                        cv2.putText(frame, label, (x+8, y-12),
                                    cv2.FONT_HERSHEY_COMPLEX, 0.8, (255,255,255), 2)
                        id_label = f"{EmployeeID} | Conf: {confidences[i]:.2f}"
                        cv2.putText(frame, id_label, (x, y+height_box+25),
                                    cv2.FONT_HERSHEY_COMPLEX, 0.6, color, 2)

            # ── FILE 2: mark people NOT seen this frame as absent ─────
            for tracked_name in list(self.person_status.keys()):
                if tracked_name not in seen_names:
                    self.mark_person_absent(tracked_name)
            # ─────────────────────────────────────────────────────────

            # Absence tracking for overlay display
            if not employee_detected and self.employee_absence_start is None:
                self.employee_absence_start = time.time()
            if not visitor_detected and self.visitor_absence_start is None:
                self.visitor_absence_start = time.time()

            # ── Draw desk area ────────────────────────────────────────
            cv2.rectangle(frame,
                          (self.monitor_area[0], self.monitor_area[1]),
                          (self.monitor_area[2], self.monitor_area[3]),
                          (255,255,0), 4)
            ms = 30
            ma = self.monitor_area
            for pt, dx, dy in [
                ((ma[0],ma[1]),(ms,0),(0,ms)),
                ((ma[2],ma[1]),(-ms,0),(0,ms)),
                ((ma[0],ma[3]),(ms,0),(0,-ms)),
                ((ma[2],ma[3]),(-ms,0),(0,-ms)),
            ]:
                cv2.line(frame, pt, (pt[0]+dx[0], pt[1]+dx[1]), (0,255,255), 6)
                cv2.line(frame, pt, (pt[0]+dy[0], pt[1]+dy[1]), (0,255,255), 6)

            # ── Status overlay ────────────────────────────────────────
                         #removed by me

            emp_text  = "Employee Status: PRESENT" if employee_detected else "Employee Status: ABSENT"
            emp_color = (0,255,0) if employee_detected else (0,0,255)
            cv2.putText(frame, emp_text, (15,35), cv2.FONT_HERSHEY_COMPLEX, 0.7, emp_color, 2)

            if not employee_detected and self.employee_absence_start is not None:
                dur = time.time() - self.employee_absence_start
                cv2.putText(frame, f"Employee Absence: {dur:.1f}s",
                            (15,65), cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,0,255), 2)

            vis_count = len(people_outside_desk)
            vis_text  = f"Visitor Status: {vis_count} PRESENT" if visitor_detected else "Visitor Status: NONE"
            vis_color = (0,165,255) if visitor_detected else (128,128,128)
            cv2.putText(frame, vis_text, (15,100), cv2.FONT_HERSHEY_COMPLEX, 0.7, vis_color, 2)

            cv2.putText(frame, f"In Desk: {len(people_in_desk)} | Outside: {len(people_outside_desk)}",
                        (15,135), cv2.FONT_HERSHEY_COMPLEX, 0.7, (255,255,0), 2)

            if people_in_desk:
                names_in = ", ".join(people_in_desk[:3])
                if len(people_in_desk) > 3:
                    names_in += f" +{len(people_in_desk)-3}"
                cv2.putText(frame, f"Inside: {names_in}", (15,165),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,255,0), 2)

            if people_outside_desk:
                names_out = ", ".join(people_outside_desk[:3])
                if len(people_outside_desk) > 3:
                    names_out += f" +{len(people_outside_desk)-3}"
                cv2.putText(frame, f"Outside: {names_out}", (15,195),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,165,255), 2)

            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {ts}", (15,225), cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,255,255), 2)
            cv2.putText(frame, "Mode: SIMPLE | Press 'q' to quit", (15,250),
                        cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,200,0), 2)

            cv2.imshow('Employee Desk Tracking - Simple Mode', frame)
            frame_count += 1
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved → {self.log_file}  |  {self.absence_log_file}")

    # ──────────────────────────────────────────────────────────────────
    # FULL MODE
    # ──────────────────────────────────────────────────────────────────
    def _run_full_mode(self, cap):
        last_cleanup = time.time()
        frame_count  = 0
        log_interval = 30

        cv2.namedWindow('Employee Desk Tracking - Full Mode', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking - Full Mode', self.display_width, self.display_height)

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            h, w = frame.shape[:2]

            if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                self.desk_mask = np.zeros((h, w), np.uint8)
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(self.desk_mask, self.DESK_ROI_PT1, self.DESK_ROI_PT2, 255, -1)

            blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416,416), swapRB=True, crop=False)
            self.net.setInput(blob)
            outputs = self.net.forward(self.output_layers)

            boxes, confidences = [], []
            for output in outputs:
                for detection in output:
                    scores   = detection[5:]
                    class_id = np.argmax(scores)
                    conf     = scores[class_id]
                    if conf > 0.5 and self.classes[class_id] == 'person':
                        cx = int(detection[0] * w)
                        cy = int(detection[1] * h)
                        bw = int(detection[2] * w)
                        bh = int(detection[3] * h)
                        boxes.append([int(cx-bw/2), int(cy-bh/2), bw, bh])
                        confidences.append(float(conf))

            indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, self.nms_threshold)

            current_in_desk  = {}
            current_outside  = {}
            employee_detected = False
            visitor_detected  = False
            seen_names        = set()
            det_list = indices.flatten() if len(indices) > 0 else []

            for i in det_list:
                x1, y1, bw, bh = boxes[i]
                x2, y2 = x1+bw, y1+bh
                cx, cy = (x1+x2)//2, (y1+y2)//2

                name, EmployeeID, email = self.get_name_from_centroid(cx, cy, frame)
                seen_names.add(name)

                if name not in self.trackers:
                    self.trackers[name] = {'centroids':[], 'last_seen':0,
                                           'EmployeeID': EmployeeID, 'email': email}
                self.trackers[name].update({'bbox':(x1,y1,x2,y2), 'last_seen':time.time(),
                                            'EmployeeID': EmployeeID, 'email': email})
                self.trackers[name]['centroids'].append((cx, cy))
                if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                    self.trackers[name]['centroids'].pop(0)

                # Desk overlap
                overlap_ratio = 0
                if x2 > x1 and y2 > y1:
                    ys = slice(max(0,y1), min(h,y2))
                    xs = slice(max(0,x1), min(w,x2))
                    roi = self.desk_mask[ys, xs]
                    if roi.size > 0:
                        overlap_ratio = np.sum(roi > 0) / roi.size

                location = 'Inside Desk' if overlap_ratio > 0.4 else 'Outside Desk'

                if 'Inside' in location:
                    current_in_desk[name] = (EmployeeID, email)
                    if name != "People":
                        employee_detected = True
                        self.last_employee_present = time.time()
                        if self.employee_absence_start is not None:
                            self.employee_absence_start = None
                else:
                    current_outside[name] = (EmployeeID, email)
                    visitor_detected = True
                    self.last_visitor_present = time.time()
                    if self.visitor_absence_start is not None:
                        self.visitor_absence_start = None

                # ── FILE 1: log on status change ──────────────────────
                if name not in self.person_status:
                    self.person_status[name] = location
                    self.log_detection(name, EmployeeID, f"Detected - {location}")
                elif self.person_status[name] != location:
                    self.person_status[name] = location
                    self.log_detection(name, EmployeeID, location)
                # ─────────────────────────────────────────────────────

                # ── FILE 2: mark present ──────────────────────────────
                self.mark_person_present(name, EmployeeID)
                # ─────────────────────────────────────────────────────

                # Draw
                color = (0,255,0) if 'Inside' in location else (0,0,255)
                cv2.rectangle(frame, (x1,y1), (x2,y2), color, 3)
                role  = "Employee" if 'Inside' in location else "Visitor"
                label = f"{name} - {role} ({location.split()[0]})"
                lsz   = cv2.getTextSize(label, cv2.FONT_HERSHEY_COMPLEX, 0.8, 2)[0]
                cv2.rectangle(frame, (x1,y1-45), (x1+lsz[0]+15, y1), color, -1)
                cv2.putText(frame, label, (x1+8, y1-15),
                            cv2.FONT_HERSHEY_COMPLEX, 0.8, (255,255,255), 2)
                id_display = EmployeeID if EmployeeID and EmployeeID != "N/A" else \
                             ("Employee" if 'Inside' in location else "Visitor")
                cv2.putText(frame, f"{id_display} | Conf: {confidences[i]:.2f}",
                            (x1, y2+25), cv2.FONT_HERSHEY_COMPLEX, 0.6, color, 2)

            # ── FILE 2: mark absent for people not seen this frame ────
            for tracked_name in list(self.person_status.keys()):
                if tracked_name not in seen_names:
                    self.mark_person_absent(tracked_name)
            # ─────────────────────────────────────────────────────────

            if not employee_detected and self.employee_absence_start is None:
                self.employee_absence_start = time.time()
            if not visitor_detected and self.visitor_absence_start is None:
                self.visitor_absence_start = time.time()

            # Cleanup disappeared trackers
            if time.time() - last_cleanup > 1.0:
                to_remove = [n for n, d in self.trackers.items()
                             if time.time() - d['last_seen'] > self.DISAPPEARANCE_TIME]
                for name in to_remove:
                    if name in self.person_status:
                        eid   = self.trackers[name].get('EmployeeID', 'N/A')
                        self.log_detection(name, eid, "Disappeared")
                        del self.person_status[name]
                    del self.trackers[name]
                last_cleanup = time.time()

            # ── Draw desk ROI ─────────────────────────────────────────
            if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                cv2.rectangle(frame, self.DESK_ROI_PT1, self.DESK_ROI_PT2, (255,255,0), 4)
                ms = 40
                p1, p2 = self.DESK_ROI_PT1, self.DESK_ROI_PT2
                for pt, dx, dy in [
                    (p1,           (ms,0),  (0,ms)),
                    ((p2[0],p1[1]),(-ms,0), (0,ms)),
                    ((p1[0],p2[1]),(ms,0),  (0,-ms)),
                    (p2,           (-ms,0), (0,-ms)),
                ]:
                    cv2.line(frame, pt, (pt[0]+dx[0],pt[1]+dx[1]), (0,255,255), 6)
                    cv2.line(frame, pt, (pt[0]+dy[0],pt[1]+dy[1]), (0,255,255), 6)

            # ── Status overlay ────────────────────────────────────────
                         #removed by me

            emp_text  = "Employee Status: PRESENT" if employee_detected else "Employee Status: ABSENT"
            emp_color = (0,255,0) if employee_detected else (0,0,255)
            cv2.putText(frame, emp_text, (15,35), cv2.FONT_HERSHEY_COMPLEX, 0.7, emp_color, 2)

            if not employee_detected and self.employee_absence_start is not None:
                dur = time.time() - self.employee_absence_start
                cv2.putText(frame, f"Employee Absence: {dur:.1f}s",
                            (15,65), cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,0,255), 2)

            vis_count = len(current_outside)
            vis_text  = f"Visitor Status: {vis_count} PRESENT" if visitor_detected else "Visitor Status: NONE"
            vis_color = (0,165,255) if visitor_detected else (128,128,128)
            cv2.putText(frame, vis_text, (15,100), cv2.FONT_HERSHEY_COMPLEX, 0.7, vis_color, 2)

            cv2.putText(frame,
                        f"Inside: {len(current_in_desk)} | Outside: {len(current_outside)} | Total: {len(self.trackers)}",
                        (15,135), cv2.FONT_HERSHEY_COMPLEX, 0.7, (255,255,0), 2)

            if current_in_desk:
                nin = ", ".join(list(current_in_desk.keys())[:3])
                if len(current_in_desk) > 3: nin += f" +{len(current_in_desk)-3}"
                cv2.putText(frame, f"Inside Desk: {nin}", (15,165),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,255,0), 2)
            if current_outside:
                nout = ", ".join(list(current_outside.keys())[:3])
                if len(current_outside) > 3: nout += f" +{len(current_outside)-3}"
                cv2.putText(frame, f"Outside Desk: {nout}", (15,195),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,165,255), 2)

            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {ts}", (15,225), cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,255,255), 2)
            cv2.putText(frame, "Mode: FULL | Press 'q' to quit", (15,250),
                        cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,200,0), 2)

            cv2.imshow('Employee Desk Tracking - Full Mode (Face Recognition)', frame)
            frame_count += 1
            if cv2.waitKey(2) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(f"Logs saved → {self.log_file}  |  {self.absence_log_file}")


# ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import sys

    tracker = DeskTracker()

    mode  = "full"
    debug = False
    for arg in sys.argv[1:]:
        if arg.lower() in ['simple', 's']:   mode  = "simple"
        elif arg.lower() in ['full', 'f']:   mode  = "full"
        elif arg.lower() in ['debug','d','--debug']: debug = True

    if debug:
        tracker.debug_mode = True
        print("DEBUG MODE ENABLED - Face search regions will be visualized")

    print("\n" + "="*60)
    print(f"Starting Desk Tracker in {mode.upper()} mode")
    print("="*60)
    print("\nLog files created in  ./Logs/Log_<date>/")
    print(f"  Detection log : Logs/Log_<date>/detection_log_<date>.csv")
    print(f"    Columns: Date & Time | EmployeeID | Name | Detection Status")
    print(f"  Absence log   : Logs/Log_<date>/absence_log_<date>.csv")
    print(f"    Columns: Date & Time | EmployeeID | Name | Absence Duration (s)")
    print(f"    (only records absences longer than 3 seconds, employees only)")
    print("\nPress 'q' to quit\n" + "="*60 + "\n")

    tracker.run(use_simple_mode=(mode == "simple"))

[13:14:48] Camera resolution: 1280x720
[13:14:48] Detecting desk area...


[13:14:48] Not enough desk objects detected. Using default desk area.
[13:14:48] Running in FULL mode


[13:14:49] [DETECTION] Piyush (23bcs061) → Detected - Inside Desk


[13:14:53] [DETECTION] People (N/A) → Detected - Outside Desk


[13:14:55] [DETECTION] Piyush (23bcs061) → Disappeared
[13:14:55] [DETECTION] People (N/A) → Disappeared


...

[13:15:16] [DETECTION] People (N/A) → Disappeared


[13:15:18] [DETECTION] People (N/A) → Detected - Outside Desk


[13:15:21] [DETECTION] People (N/A) → Disappeared


[13:15:30] [DETECTION] People (N/A) → Detected - Inside Desk


[13:15:30] Logs saved → Logs\Log_17-02-2026\detection_log_17-02-2026.csv  |  Logs\Log_17-02-2026\absence_log_17-02-2026.csv


In [ ]:
#YoloV8

In [ ]:
pip install ultralytics

In [3]:
import cv2
import numpy as np
import time
from datetime import datetime
import csv
import os
import pickle
from sklearn.neighbors import KNeighborsClassifier

# ── YOLOv8 via ultralytics ────────────────────────────────────────────
from ultralytics import YOLO   # pip install ultralytics


class DeskTracker:
    def __init__(self):
        # ── YOLOv8 model ─────────────────────────────────────────────
        # Replace 'yolov8n.pt' with any YOLOv8 variant:
        #   yolov8n.pt  (nano  – fastest / smallest)
        #   yolov8s.pt  (small)
        #   yolov8m.pt  (medium)
        #   yolov8l.pt  (large)
        #   yolov8x.pt  (extra-large – most accurate)
        # The first run downloads the weights automatically.
        self.model = YOLO('Yolo_model/yolov8n.pt')

        # COCO class indices used in desk-area detection
        # 56=chair, 60=dining table, 63=laptop, 64=mouse, 65=remote
        self.DESK_RELATED_CLASSES = {56, 60, 63, 64, 65}
        self.PERSON_CLASS_ID = 0           # COCO class 0 = person

        # Detection thresholds
        self.confidence_threshold = 0.3    # used during desk-area scan
        self.person_conf = 0.50            # used during tracking

        # ── Face recognition ─────────────────────────────────────────
        self.face_recognition_threshold = 20
        self.face_min_size = (30, 30)

        try:
            with open('data/Organizations.pkl', 'rb') as f:
                self.ORGANIZATIONS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/EmployeeIDs.pkl', 'rb') as f:
                self.EmployeeIDs = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f)
                self.FACES = self.FACES / 255.0
                print(f"FACES shape: {self.FACES.shape}")

            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.EmployeeIDs)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []
            self.EmployeeIDs = []
            self.ORGANIZATIONS = []
            self.EMAILS = []
            self.face_recognition_enabled = False

        self.face_cascade = cv2.CascadeClassifier(
            'data/haarcascade_frontalface_default.xml')

        # ── Desk ROI ─────────────────────────────────────────────────
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask = None

        # ── Tracking state ───────────────────────────────────────────
        self.trackers = {}
        self.person_status = {}
        self.MAX_CENTROIDS = 10
        self.DISAPPEARANCE_TIME = 2.0

        self.employee_absence_start = None
        self.visitor_absence_start = None
        self.last_employee_present = None
        self.last_visitor_present = None

        # ── Logging setup ────────────────────────────────────────────
        date = datetime.now().strftime("%d-%m-%Y")
        os.makedirs('Logs', exist_ok=True)
        log_folder = os.path.join('Logs', f'Log_{date}')
        os.makedirs(log_folder, exist_ok=True)

        self.log_file = os.path.join(log_folder, f'detection_log_{date}.csv')
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                csv.writer(f).writerow(
                    ['Date & Time', 'EmployeeID', 'Name', 'Detection Status'])

        self.absence_log_file = os.path.join(
            log_folder, f'absence_log_{date}.csv')
        if not os.path.exists(self.absence_log_file):
            with open(self.absence_log_file, 'w', newline='') as f:
                csv.writer(f).writerow(
                    ['Date & Time', 'EmployeeID', 'Name', 'Absence Duration (s)'])

        self.person_absence_start = {}
        self.person_absence_logged = {}

        self.monitor_area = None
        self.display_width = 1280
        self.display_height = 720
        self.debug_mode = False

    # ─────────────────────────────────────────────────────────────────
    # LOGGING HELPERS
    # ─────────────────────────────────────────────────────────────────
    def log_event(self, message):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def log_detection(self, name, employee_id, status):
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.log_file, 'a', newline='') as f:
            csv.writer(f).writerow([timestamp, employee_id, name, status])
        self.log_event(f"[DETECTION] {name} ({employee_id}) → {status}")

    def log_absence(self, name, employee_id, duration_seconds):
        if duration_seconds <= 3:
            return
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.absence_log_file, 'a', newline='') as f:
            csv.writer(f).writerow(
                [timestamp, employee_id, name, f"{duration_seconds:.1f}"])
        self.log_event(
            f"[ABSENCE]   {name} ({employee_id}) absent for {duration_seconds:.1f}s")

    # ─────────────────────────────────────────────────────────────────
    # PER-PERSON ABSENCE TRACKING
    # ─────────────────────────────────────────────────────────────────
    def mark_person_present(self, name, employee_id):
        if name in self.person_absence_start:
            duration = time.time() - self.person_absence_start.pop(name)
            if name != "People":
                self.log_absence(name, employee_id, duration)

    def mark_person_absent(self, name):
        if name not in self.person_absence_start:
            self.person_absence_start[name] = time.time()

    # ─────────────────────────────────────────────────────────────────
    # DESK AREA DETECTION  (YOLOv8 version)
    # ─────────────────────────────────────────────────────────────────
    def detect_desk_area(self, cap):
        self.log_event("Detecting desk area with YOLOv8 …")
        desk_candidates = []
        frame_count = 0
        frame = None

        while frame_count < 5:
            ret, frame = cap.read()
            if not ret or frame is None or frame.size == 0:
                time.sleep(0.05)
                continue

            height, width = frame.shape[:2]

            # ── YOLOv8 inference ─────────────────────────────────────
            results = self.model(
                frame,
                conf=self.confidence_threshold,
                verbose=False
            )

            for box in results[0].boxes:
                cls_id = int(box.cls[0])
                conf   = float(box.conf[0])
                if cls_id in self.DESK_RELATED_CLASSES:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    desk_candidates.append(
                        [x1, y1, x2 - x1, y2 - y1, conf, cls_id])

            frame_count += 1

        # Compute ROI from candidates
        if len(desk_candidates) < 2:
            self.log_event(
                "Not enough desk objects detected – using default area.")
            if frame is not None:
                height, width = frame.shape[:2]
            else:
                height, width = 480, 640
            min_x = int(width  * 0.2)
            min_y = int(height * 0.4)
            max_x = int(width  * 0.8)
            max_y = int(height * 0.9)
        else:
            x_coords     = [d[0]          for d in desk_candidates]
            y_coords     = [d[1]          for d in desk_candidates]
            max_x_coords = [d[0] + d[2]   for d in desk_candidates]
            max_y_coords = [d[1] + d[3]   for d in desk_candidates]
            min_x = max(0,      int(min(x_coords)     - 0.1 * width))
            min_y = max(0,      int(min(y_coords)     - 0.1 * height))
            max_x = min(width,  int(max(max_x_coords) + 0.1 * width))
            max_y = min(height, int(max(max_y_coords) + 0.1 * height))
            self.log_event(
                f"Desk area detected: ({min_x}, {min_y}) → ({max_x}, {max_y})")

        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        self.monitor_area = (min_x, min_y, max_x, max_y)
        return (min_x, min_y, max_x, max_y)

    # ─────────────────────────────────────────────────────────────────
    # FACE RECOGNITION
    # ─────────────────────────────────────────────────────────────────
    def get_name_from_face(self, face_region):
        if not self.face_recognition_enabled or face_region.size == 0:
            return "People", "N/A", ""
        if face_region.shape[0] < 30 or face_region.shape[1] < 30:
            return "People", "N/A", ""

        gray_region = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        faces = []
        for scale, neighbors, min_size in [
            (1.1,  5, (20, 20)),
            (1.05, 4, (25, 25)),
            (1.2,  5, (30, 30)),
            (1.3,  4, (35, 35)),
            (1.15, 3, (25, 25)),
        ]:
            detected = self.face_cascade.detectMultiScale(
                gray_region, scale, neighbors, minSize=min_size)
            if len(detected) > 0:
                faces = detected
                break

        Name, EmployeeID, Email = "People", "N/A", ""

        for (x, y, w_face, h_face) in faces:
            crop_img = face_region[y:y+h_face, x:x+w_face, :]
            if crop_img.size == 0:
                continue
            try:
                resized_img  = cv2.resize(crop_img, (75, 100))
                gray         = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
                resized_flat = gray.reshape(1, -1) / 255.0
                if resized_flat.shape[1] != self.FACES.shape[1]:
                    continue
                distances, _ = self.knn.kneighbors(resized_flat)
                if distances[0][0] <= self.face_recognition_threshold:
                    output = self.knn.predict(resized_flat)
                    emp_id = output[0]
                    if emp_id in self.EmployeeIDs:
                        index      = self.EmployeeIDs.index(emp_id)
                        Name       = self.NAMES[index]
                        EmployeeID = self.EmployeeIDs[index] if index < len(self.EmployeeIDs) else "N/A"
                        Email      = self.EMAILS[index]      if index < len(self.EMAILS)      else ""
                        break
            except Exception:
                continue

        return Name, EmployeeID, Email

    def get_name_from_centroid(self, cx, cy, frame):
        h, w = frame.shape[:2]
        search_regions = [
            (max(0,cx-180), max(0,cy-280), min(w,cx+180), min(h,cy+50)),
            (max(0,cx-220), max(0,cy-320), min(w,cx+220), min(h,cy+80)),
            (max(0,cx-150), max(0,cy-350), min(w,cx+150), min(h,cy)),
            (max(0,cx-200), max(0,cy-200), min(w,cx+200), min(h,cy+150)),
        ]
        for idx, (rx1, ry1, rx2, ry2) in enumerate(search_regions):
            if self.debug_mode:
                color = [(255,0,255),(0,255,255),(255,255,0),(255,128,0)][idx]
                cv2.rectangle(frame, (rx1,ry1), (rx2,ry2), color, 2)
                cv2.putText(frame, f"Search {idx+1}", (rx1, ry1-5),
                            cv2.FONT_HERSHEY_COMPLEX, 0.4, color, 1)
            face_region = frame[ry1:ry2, rx1:rx2]
            name, EmployeeID, email = self.get_name_from_face(face_region)
            if name != "People":
                return name, EmployeeID, email
        return "People", "N/A", ""

    # ─────────────────────────────────────────────────────────────────
    # MAIN RUN
    # ─────────────────────────────────────────────────────────────────
    def run(self):
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            self.log_event("Error: Cannot open default camera.")
            return
        self.log_event("Default camera opened.")

        cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1920)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
        cap.set(cv2.CAP_PROP_FPS, 30)

        actual_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        actual_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.log_event(f"Resolution: {actual_width}x{actual_height}")

        self.detect_desk_area(cap)
        self.log_event("Starting tracker… Press 'q' to quit.")
        self._run_full_mode(cap)

    # ─────────────────────────────────────────────────────────────────
    # FULL MODE  (YOLOv8 version)
    # ─────────────────────────────────────────────────────────────────
    def _run_full_mode(self, cap):
        last_cleanup = time.time()

        cv2.namedWindow('Employee Desk Tracking', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking',
                         self.display_width, self.display_height)

        while True:
            ret, frame = cap.read()
            if not ret or frame is None or frame.size == 0:
                time.sleep(0.05)
                continue

            h, w = frame.shape[:2]

            # Rebuild desk mask if needed
            if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                self.desk_mask = np.zeros((h, w), np.uint8)
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(self.desk_mask,
                                  self.DESK_ROI_PT1, self.DESK_ROI_PT2,
                                  255, -1)

            # ── YOLOv8 inference ─────────────────────────────────────
            results = self.model(
                frame,
                conf=self.person_conf,
                classes=[self.PERSON_CLASS_ID],   # person only
                verbose=False
            )

            current_in_desk   = {}
            current_outside   = {}
            employee_detected = False
            visitor_detected  = False
            seen_names        = set()

            for box in results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf            = float(box.conf[0])
                cx, cy          = (x1 + x2) // 2, (y1 + y2) // 2

                name, EmployeeID, email = self.get_name_from_centroid(
                    cx, cy, frame)
                seen_names.add(name)

                # Update tracker
                if name not in self.trackers:
                    self.trackers[name] = {
                        'centroids': [], 'last_seen': 0,
                        'EmployeeID': EmployeeID, 'email': email}
                self.trackers[name].update({
                    'bbox': (x1, y1, x2, y2),
                    'last_seen': time.time(),
                    'EmployeeID': EmployeeID,
                    'email': email})
                self.trackers[name]['centroids'].append((cx, cy))
                if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                    self.trackers[name]['centroids'].pop(0)

                # Desk overlap
                overlap_ratio = 0
                if x2 > x1 and y2 > y1:
                    ys  = slice(max(0,y1), min(h,y2))
                    xs  = slice(max(0,x1), min(w,x2))
                    roi = self.desk_mask[ys, xs]
                    if roi.size > 0:
                        overlap_ratio = np.sum(roi > 0) / roi.size

                location = 'Inside Desk' if overlap_ratio > 0.4 else 'Outside Desk'

                if 'Inside' in location:
                    current_in_desk[name] = (EmployeeID, email)
                    if name != "People":
                        employee_detected = True
                        self.last_employee_present = time.time()
                        self.employee_absence_start = None
                else:
                    current_outside[name] = (EmployeeID, email)
                    visitor_detected = True
                    self.last_visitor_present = time.time()
                    self.visitor_absence_start = None

                # Log status changes (File 1)
                if name not in self.person_status:
                    self.person_status[name] = location
                    self.log_detection(name, EmployeeID,
                                       f"Detected - {location}")
                elif self.person_status[name] != location:
                    self.person_status[name] = location
                    self.log_detection(name, EmployeeID, location)

                # Mark present (File 2)
                self.mark_person_present(name, EmployeeID)

                # ── Draw bounding box ─────────────────────────────────
                color = (0, 255, 0) if 'Inside' in location else (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                role  = "Employee" if 'Inside' in location else "Visitor"
                label = f"{name} - {role} ({location.split()[0]})"
                lsz   = cv2.getTextSize(
                    label, cv2.FONT_HERSHEY_COMPLEX, 0.8, 2)[0]
                cv2.rectangle(frame,
                              (x1, y1-45), (x1+lsz[0]+15, y1), color, -1)
                cv2.putText(frame, label, (x1+8, y1-15),
                            cv2.FONT_HERSHEY_COMPLEX, 0.8, (255,255,255), 2)
                id_display = (EmployeeID
                              if EmployeeID and EmployeeID != "N/A"
                              else ("Employee" if 'Inside' in location
                                    else "Visitor"))
                cv2.putText(frame,
                            f"{id_display} | Conf: {conf:.2f}",
                            (x1, y2+25),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, color, 2)

            # Mark absent for people not seen this frame
            for tracked_name in list(self.person_status.keys()):
                if tracked_name not in seen_names:
                    self.mark_person_absent(tracked_name)

            if not employee_detected and self.employee_absence_start is None:
                self.employee_absence_start = time.time()
            if not visitor_detected and self.visitor_absence_start is None:
                self.visitor_absence_start = time.time()

            # Cleanup disappeared trackers
            if time.time() - last_cleanup > 1.0:
                to_remove = [
                    n for n, d in self.trackers.items()
                    if time.time() - d['last_seen'] > self.DISAPPEARANCE_TIME]
                for name in to_remove:
                    if name in self.person_status:
                        eid = self.trackers[name].get('EmployeeID', 'N/A')
                        self.log_detection(name, eid, "Disappeared")
                        del self.person_status[name]
                    del self.trackers[name]
                last_cleanup = time.time()

            # ── Draw desk ROI corners ─────────────────────────────────
            if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                cv2.rectangle(frame,
                              self.DESK_ROI_PT1, self.DESK_ROI_PT2,
                              (255, 255, 0), 4)
                ms = 40
                p1, p2 = self.DESK_ROI_PT1, self.DESK_ROI_PT2
                for pt, dx, dy in [
                    (p1,            (ms, 0),   (0,  ms)),
                    ((p2[0], p1[1]),(-ms, 0),  (0,  ms)),
                    ((p1[0], p2[1]),(ms, 0),   (0, -ms)),
                    (p2,            (-ms, 0),  (0, -ms)),
                ]:
                    cv2.line(frame, pt,
                             (pt[0]+dx[0], pt[1]+dx[1]), (0,255,255), 6)
                    cv2.line(frame, pt,
                             (pt[0]+dy[0], pt[1]+dy[1]), (0,255,255), 6)

            # ── Status overlay ────────────────────────────────────────
            emp_text  = ("Employee Status: PRESENT"
                         if employee_detected else "Employee Status: ABSENT")
            emp_color = (0,255,0) if employee_detected else (0,0,255)
            cv2.putText(frame, emp_text, (15,35),
                        cv2.FONT_HERSHEY_COMPLEX, 0.7, emp_color, 2)

            if not employee_detected and self.employee_absence_start:
                dur = time.time() - self.employee_absence_start
                cv2.putText(frame, f"Employee Absence: {dur:.1f}s",
                            (15,65), cv2.FONT_HERSHEY_COMPLEX,
                            0.6, (0,0,255), 2)

            vis_count = len(current_outside)
            vis_text  = (f"Visitor Status: {vis_count} PRESENT"
                         if visitor_detected else "Visitor Status: NONE")
            vis_color = (0,165,255) if visitor_detected else (128,128,128)
            cv2.putText(frame, vis_text, (15,100),
                        cv2.FONT_HERSHEY_COMPLEX, 0.7, vis_color, 2)

            cv2.putText(
                frame,
                f"Inside: {len(current_in_desk)} | "
                f"Outside: {len(current_outside)} | "
                f"Total: {len(self.trackers)}",
                (15,135), cv2.FONT_HERSHEY_COMPLEX, 0.7, (255,255,0), 2)

            if current_in_desk:
                nin = ", ".join(list(current_in_desk.keys())[:3])
                if len(current_in_desk) > 3:
                    nin += f" +{len(current_in_desk)-3}"
                cv2.putText(frame, f"Inside Desk: {nin}", (15,165),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,255,0), 2)
            if current_outside:
                nout = ", ".join(list(current_outside.keys())[:3])
                if len(current_outside) > 3:
                    nout += f" +{len(current_outside)-3}"
                cv2.putText(frame, f"Outside Desk: {nout}", (15,195),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,165,255), 2)

            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {ts}", (15,225),
                        cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,255,255), 2)
            cv2.putText(frame, "Press q to Quit", (15,250),
                        cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,200,0), 2)

            cv2.imshow('Employee Desk Tracking', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(
            f"Logs saved → {self.log_file}  |  {self.absence_log_file}")


# ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import sys

    tracker = DeskTracker()

    for arg in sys.argv[1:]:
        if arg.lower() in ['debug', 'd', '--debug']:
            tracker.debug_mode = True
            print("DEBUG MODE ENABLED – face search regions visualised")

    print("\n" + "="*60)
    print("Starting Employee Desk Tracker  [YOLOv8]")
    print("="*60)
    print("\nPress 'q' to quit\n" + "="*60 + "\n")

    tracker.run()

FACES shape: (50, 7500)
Face recognition loaded successfully.

Starting Employee Desk Tracker  [YOLOv8]

Press 'q' to quit

[10:02:25] Default camera opened.
[10:02:32] Resolution: 1280x720
[10:02:32] Detecting desk area with YOLOv8 …
[10:02:32] Not enough desk objects detected – using default area.
[10:02:32] Starting tracker… Press 'q' to quit.
[10:02:33] [DETECTION] Piyush (23bcs061) → Detected - Inside Desk
[10:02:46] [DETECTION] People (N/A) → Detected - Inside Desk
[10:02:47] Logs saved → Logs\Log_18-02-2026\detection_log_18-02-2026.csv  |  Logs\Log_18-02-2026\absence_log_18-02-2026.csv
